# GNN Powerflow V2.7 — Training

Loads network datasets, runs hyperparameter sweeps, saves results.

**Input**: dataset JSONs from `GNN_Powerflow_V2.6_DataGen.ipynb`

**Output**: sweep result JSONs in `TRAINING_RESULTS_DIR`

In [ ]:

import copy
import logging
import os
import pickle
import random
import time
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pypsa
import pypowsybl as pp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch, Data, Dataset
from torch_geometric.nn import GATv2Conv, GCNConv, GraphConv, TransformerConv, global_mean_pool, global_max_pool  # [STSI120526] Added TransformerConv import
from tqdm.auto import tqdm
#from tqdm.notebook import tqdm
import urllib

import statsmodels.api as sm
from statsmodels.formula.api import ols

# Suppress verbose output from PyPSA and dependencies during data generation
for _log in ("pypsa", "pypsa.pf", "pypsa.components", "numexpr", "linopy"):
    logging.getLogger(_log).setLevel(logging.WARNING)
logging.getLogger("numexpr").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import re
from scipy import ndimage

In [ ]:
# ── Configurable paths ───────────────────────────────────────────────────────
# Set DATA_ROOT to wherever your data folders live.
# Use "." to keep everything relative to the repo (default).
# Example: DATA_ROOT = r"C:\Users\you\OneDrive\Data_PF_GNN"
import os as _os
DATA_ROOT             = r"C:\Users\STSI\OneDrive - USN\Data_PF_GNN"                                             
#DATA_ROOT             = r"C:\Users\ssimo\OneDrive - USN\Data_PF_GNN"                                             
GRID_MODEL_FILES      = _os.path.join(DATA_ROOT, "grid_model_files")    # static grid CSV/RAW files
TRAINING_NETWORKS_DIR = _os.path.join(DATA_ROOT, "training_networks_saved")  # dataset JSONs
TRAINING_RESULTS_DIR  = _os.path.join(DATA_ROOT, "training_results_saved")   # sweep result JSONs/CSVs


In [ ]:
#test and execution toggles

run_regression_test=False # to run a regression test on the GNN training loop (with very limited epochs and data)


In [ ]:
import itertools

# ── Style pools ──────────────────────────────────────────────────────────────
_LINE_STYLES  = ["-", "--", "-.", ":"]
_MARKERS      = ["", "o", "s", "^", "D", "v", "P", "X"]
_COLOR_CYCLE  = plt.rcParams["axes.prop_cycle"].by_key()["color"]

## Data Generation Functions
_Paste from V2.5.3 cells 20, 22, 23, 25, 26_

In [ ]:
#==================save and load genrated datasets========================

# not needed
def save_dataset_list(
    dataset_list: list,
    index_path: str,
    tag: str = "dataset",
    subfolder: str | None = None,
) -> str:
    """
    Save a list of PyPSA networks using NetCDF (avoids weakref pickle errors).

    Each network is saved as an individual .nc file inside a dedicated subfolder;
    an index JSON records the ordered list of paths so they can be reloaded in
    the same order.

    Parameters
    ----------
    dataset_list : list[pypsa.Network]
    index_path   : str        – path for the JSON index file
    tag          : str        – prefix for individual network filenames
    subfolder    : str | None – name of the subdirectory inside the index
                                directory that holds the .nc files.
                                Defaults to ``tag`` when not supplied.
                                Example: ``subfolder="dataset_118"``

    Returns
    -------
    str – path to the JSON index file
    """
    import json

    save_dir = os.path.dirname(index_path) or "."
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # .nc files go in their own subfolder to avoid cluttering save_dir
    nc_subdir = subfolder if subfolder is not None else tag
    nc_dir = os.path.join(save_dir, nc_subdir)
    os.makedirs(nc_dir, exist_ok=True)

    nc_paths: list[str] = []
    for i, net in enumerate(dataset_list):
        nc_name = f"{tag}_{timestamp}_{i:04d}.nc"
        nc_path = os.path.join(nc_dir, nc_name)
        net.export_to_netcdf(nc_path)
        nc_paths.append(os.path.join(nc_subdir, nc_name))  # relative to save_dir

    with open(index_path, "w") as f:
        json.dump({"tag": tag, "timestamp": timestamp, "files": nc_paths}, f, indent=2)

    print(f"Saved {len(nc_paths)} networks → {nc_dir}/ (index: {index_path})")
    return index_path

# needed
def load_dataset_list(index_path: str) -> list:
    """
    Reload a list of PyPSA networks saved by save_dataset_list.
    """
    import json

    with open(index_path, "r") as f:
        meta = json.load(f)

    save_dir = os.path.dirname(index_path) or "."
    networks: list[pypsa.Network] = []
    for nc_name in meta["files"]:
        nc_path = os.path.join(save_dir, nc_name)
        networks.append(pypsa.Network(nc_path))

    print(f"Loaded {len(networks)} networks from {index_path}")
    return networks



## Power System Helpers

In [ ]:
# =============================================================================
# SECTION 2: PTDF COMPUTATION
# =============================================================================

def compute_ptdf_matrix(network, ptdf_branch_mode="lines"):  # [STSI 260526]:added ptdf_branch_mode param 
    """
    Compute the Power Transfer Distribution Factor (PTDF) matrix for a PyPSA network.
    Returns a numpy array indexed by branches, columns by buses.

    Args:
        ptdf_branch_mode: "lines" returns [n_lines, n_buses],
                          "all" returns [n_lines + n_trafos, n_buses].
    """
    buses = list(network.buses.index)
    lines = list(network.lines.index)
    n_buses = len(buses)
    n_lines = len(lines)

    bus_to_idx = {b: i for i, b in enumerate(buses)}

    # Build susceptance matrix B and branch-bus incidence A
    B = np.zeros((n_buses, n_buses))
    n_trafos = len(network.transformers)  # [STSI 260526]:compute branch count
    n_branches = n_lines + n_trafos if ptdf_branch_mode == "all" else n_lines
    A = np.zeros((n_branches, n_buses))

    for i, line in enumerate(lines):
        b_val = 1.0 / network.lines.loc[line, "x"] if network.lines.loc[line, "x"] != 0 else 0.0
        from_idx = bus_to_idx[network.lines.loc[line, "bus0"]]
        to_idx = bus_to_idx[network.lines.loc[line, "bus1"]]
        B[from_idx, from_idx] += b_val
        B[to_idx, to_idx] += b_val
        B[from_idx, to_idx] -= b_val
        B[to_idx, from_idx] -= b_val
        A[i, from_idx] = b_val
        A[i, to_idx] = -b_val


    # Add transformer susceptance to B (transformers are series impedances in flat-pu)  # [STSI 230526]:fix missing trafo in B matrix
    for trafo_i, trafo in enumerate(network.transformers.index):
        row = network.transformers.loc[trafo]
        if "x_pu_eff" in row.index and pd.notna(row.get("x_pu_eff")) and float(row["x_pu_eff"]) != 0:
            x_val = float(row["x_pu_eff"])
        else:
            x_val = float(row["x"])
        if x_val == 0:
            continue
        b_val = 1.0 / x_val
        from_idx = bus_to_idx[row["bus0"]]
        to_idx = bus_to_idx[row["bus1"]]
        B[from_idx, from_idx] += b_val
        B[to_idx, to_idx] += b_val
        B[from_idx, to_idx] -= b_val
        B[to_idx, from_idx] -= b_val
        if ptdf_branch_mode == "all":  # [STSI 260526]:include trafo rows in incidence matrix A 
            A[n_lines + trafo_i, from_idx] = b_val
            A[n_lines + trafo_i, to_idx] = -b_val
    # Identify slack bus (first generator with control='Slack')
    slack_bus = None
    for gen in network.generators.index:
        if network.generators.loc[gen, "control"] == "Slack":
            slack_bus = network.generators.loc[gen, "bus"]
            break
    if slack_bus is None:
        slack_bus = buses[0]
    slack_idx = bus_to_idx[slack_bus]

    # Reduce B matrix (remove slack row/col)
    non_slack = [i for i in range(n_buses) if i != slack_idx]
    B_red = B[np.ix_(non_slack, non_slack)]
    A_red = A[:, non_slack]

    try:
        B_red_inv = np.linalg.inv(B_red)
    except np.linalg.LinAlgError:
        B_red_inv = np.linalg.pinv(B_red)

    PTDF_red = A_red @ B_red_inv  # (n_branches, n_buses-1)

    # Re-insert slack column as zeros
    PTDF = np.zeros((n_branches, n_buses))  # [STSI 260526]:use n_branches for mode=all
    non_slack_col = 0
    for j in range(n_buses):
        if j != slack_idx:
            PTDF[:, j] = PTDF_red[:, non_slack_col]
            non_slack_col += 1

    return PTDF



# =============================================================================
# SECTION 3: ADMITTANCE MATRIX
# =============================================================================

def compute_admittance_matrix(network, device="cpu", return_format="torch"):
    """
    Compute bus admittance matrix Y for a PyPSA network.

    Improvements over original:
      - Handles arbitrary bus naming/ordering
      - Includes line shunt b/2 at both ends
      - Includes transformer series impedance and tap_ratio
      - GPU-capable PyTorch tensors
      - Optional complex or split format

    Args:
        network: PyPSA network object
        device:  Device for PyTorch tensors ('cpu' or 'cuda')
        return_format: 'torch' -> (Y_real, Y_imag) as torch tensors
                       'numpy' -> complex numpy array
                       'complex_torch' -> complex torch tensor
    """
    num_buses = len(network.buses)
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    Y = np.zeros((num_buses, num_buses), dtype=complex)

    # --------------------
    # Lines: π-model
    # --------------------
    for line in network.lines.index:
        row = network.lines.loc[line]
        from_bus = row["bus0"]
        to_bus   = row["bus1"]
        r = row["r"]
        x = row["x"]
        b = row["b"] if "b" in network.lines.columns else 0.0

        z = complex(r, x)
        if abs(z) < 1e-12:
            # avoid division by zero; skip or handle specially
            continue
        y_series = 1.0 / z
        y_shunt  = complex(0.0, b / 2.0)  # b is total line charging susceptance

        fi = bus_to_idx[from_bus]
        ti = bus_to_idx[to_bus]

        # series
        Y[fi, fi] += y_series
        Y[ti, ti] += y_series
        Y[fi, ti] -= y_series
        Y[ti, fi] -= y_series

        # shunts at both ends
        Y[fi, fi] += y_shunt
        Y[ti, ti] += y_shunt

    # --------------------
    # Transformers: series + tap ratio (no shunt for now)
    # --------------------
    # Transformers: use x_pu_eff/r_pu_eff (system base) instead of raw x/r
    for trafo in network.transformers.index:
        row = network.transformers.loc[trafo]
        from_bus = row["bus0"]
        to_bus   = row["bus1"]

        # STSI260320: raw x/r are on transformer MVA base; x_pu_eff/r_pu_eff
        # are already converted to system base — use these to match PyPSA's Ybus
        if "x_pu_eff" in row.index and pd.notna(row["x_pu_eff"]) and float(row["x_pu_eff"]) != 0.0:
            x = float(row["x_pu_eff"])
            r = float(row["r_pu_eff"]) if "r_pu_eff" in row.index and pd.notna(row["r_pu_eff"]) else 0.0
        else:
            x = float(row["x"])
            r = float(row.get("r", 0.0))

        tap = float(row["tap_ratio"]) if "tap_ratio" in row.index else 1.0

        z = complex(r, x)
        if abs(z) < 1e-12:
            continue
        y_series = 1.0 / z

        fi = bus_to_idx[from_bus]
        ti = bus_to_idx[to_bus]
        t  = complex(tap, 0.0)

        Y[fi, fi] += y_series / (t * t.conjugate())
        Y[ti, ti] += y_series
        Y[fi, ti] -= y_series / t.conjugate()
        Y[ti, fi] -= y_series / t


    # ---------------
    # Return formats
    # ---------------
    if return_format == "numpy":
        return Y
    elif return_format == "complex_torch":
        return torch.tensor(Y, dtype=torch.complex64, device=device)
    else:  # 'torch' -> split real/imag
        Y_real = torch.tensor(Y.real, dtype=torch.float32, device=device)
        Y_imag = torch.tensor(Y.imag, dtype=torch.float32, device=device)
        return Y_real, Y_imag

def get_pypsa_Y_numpy(net: pypsa.Network) -> np.ndarray | None:
    """
    Get PyPSA's Ybus via subnetwork API.

    STSI260320: Works on a deepcopy to avoid mutating training networks.
    determine_network_topology() modifies net.buses['sub_network'] in-place,
    which can corrupt dataset construction if called on live training networks.
    """
    if hasattr(net, "admittance_matrix"):
        Y = net.admittance_matrix()
        return np.asarray(Y.todense())

    if not hasattr(net, "determine_network_topology"):
        return None

    try:
        # STSI260320: deepcopy to prevent mutation of training network objects
        net_copy = copy.deepcopy(net)
        net_copy.determine_network_topology()
    except Exception as e:
        logger.debug(f"[YCHECK] determine_network_topology failed: {e}")
        return None

    if not hasattr(net_copy, "sub_networks") or len(net_copy.sub_networks) == 0:
        return None

    sn_label = list(net_copy.sub_networks.index)[0]
    sn = net_copy.sub_networks.at[sn_label, "obj"]

    if not hasattr(sn, "calculate_Y"):
        return None

    try:
        sn.calculate_Y()
        Y_sparse = sn.Y
        Y_dense  = np.asarray(Y_sparse.todense())

        # [STSI 210526]: fixed type-mismatch guard; was always returning Y in buses_o order (str vs int comparison always False) -- reorder buses_o->buses_i instead
        sn_buses_i = list(sn.buses_i())
        sn_buses_o = list(sn.buses_o)
        if sn_buses_i == sn_buses_o:
            return Y_dense
        pos_in_o = {b: j for j, b in enumerate(sn_buses_o)}
        perm = [pos_in_o[b] for b in sn_buses_i]
        return Y_dense[np.ix_(perm, perm)]

    except Exception as e:
        logger.debug(f"[YCHECK] calculate_Y() failed: {e}")
        return None

def precompute_Y_matrices(
    networks: list,
    device: str | torch.device = "cpu",
    y_tolerance: float = 1e-6,
    y_matrix_source: str = "auto",   # STSI260320: "manual", "pypsa", "auto"
):
    """
    Pre-compute Y matrices for a list of networks.

    y_matrix_source:
      "manual" — always use compute_admittance_matrix (your implementation).
      "pypsa"  — always use PyPSA's subnetwork Y; raise if not available.
      "auto"   — use PyPSA Y if available and consistent within y_tolerance,
                 otherwise fall back to manual Y and log a warning.

    STSI260320: Added y_matrix_source argument so this can be controlled as
    a hyperparameter in run_hparam_sweep, allowing ablation between manual
    and PyPSA-derived Y matrices in the physics loss.
    """
    assert y_matrix_source in ("manual", "pypsa", "auto"), \
        f"y_matrix_source must be 'manual', 'pypsa', or 'auto', got {y_matrix_source!r}"

    if isinstance(device, str):
        device = torch.device(device)

    Y_list  = []
    n_manual = 0
    n_pypsa  = 0
    n_diff   = 0

    for idx, net in enumerate(networks):
        Y_manual = compute_admittance_matrix(net, device=device, return_format="numpy")
        Y_pypsa  = get_pypsa_Y_numpy(net)  # None if unavailable

        if y_matrix_source == "manual":
            Y_use = Y_manual
            n_manual += 1

        elif y_matrix_source == "pypsa":
            if Y_pypsa is None:
                raise RuntimeError(
                    f"y_matrix_source='pypsa' but PyPSA Y not available for network {idx}."
                )
            Y_use = Y_pypsa
            n_pypsa += 1

        else:  # "auto"
            if Y_pypsa is None:
                logger.debug(f"[YCHECK] Network {idx}: PyPSA Y unavailable — using manual.")
                Y_use = Y_manual
                n_manual += 1
            elif Y_manual.shape != Y_pypsa.shape:
                logger.warning(
                    f"[YCHECK] Network {idx}: shape mismatch "
                    f"{Y_manual.shape} vs {Y_pypsa.shape} — using PyPSA Y."
                )
                Y_use = Y_pypsa
                n_pypsa += 1
                n_diff  += 1
            else:
                max_diff = np.max(np.abs(Y_manual - Y_pypsa))
                if max_diff > y_tolerance:
                    logger.warning(
                        f"[YCHECK] Network {idx}: max|Y_manual-Y_pypsa|={max_diff:.3e} "
                        f"> tol={y_tolerance:.1e} — using PyPSA Y."
                    )
                    Y_use = Y_pypsa
                    n_pypsa += 1
                    n_diff  += 1
                else:
                    logger.debug(
                        f"[YCHECK] Network {idx}: manual Y matches PyPSA "
                        f"(max diff {max_diff:.3e})."
                    )
                    Y_use = Y_manual
                    n_manual += 1

        Y_real = torch.from_numpy(Y_use.real).to(device=device, dtype=torch.float32)
        Y_imag = torch.from_numpy(Y_use.imag).to(device=device, dtype=torch.float32)
        Y_list.append((Y_real, Y_imag))

    # STSI260320: single summary log instead of per-network messages
    if n_diff > 0:
        logger.warning(
            f"[YCHECK] Summary ({y_matrix_source}): {n_pypsa} PyPSA Y, "
            f"{n_manual} manual Y, {n_diff} had mismatches > {y_tolerance:.1e}."
        )
    else:
        logger.info(
            f"[YCHECK] Summary ({y_matrix_source}): {n_pypsa} PyPSA Y, "
            f"{n_manual} manual Y, 0 mismatches."
        )

    return Y_list


In [ ]:
# =============================================================================
# SECTION 2B: EDGE Δθ UTILITIES (BFS precomputation + θ reconstruction)
# =============================================================================
# [STSI 250526]:added edge-delta-theta prediction utilities 

from collections import deque

def precompute_bfs_order(edge_index_fwd, slack_idx, num_nodes):
    """
    Compute BFS traversal from slack for θ reconstruction from Δθ.
    
    Returns:
        bfs_edge_idx:   [N-1] indices into forward edges — edges in BFS tree order
        bfs_signs:      [N-1] float tensor — +1 or -1
        bfs_node_order: [N-1] node indices in BFS visit order (excludes slack)
        bfs_parent:     [N-1] parent node index for each BFS child
    
    Convention: θ_child = θ_parent - sign * Δθ_edge
        sign = +1 when edge is parent→child (forward direction = from→to)
        sign = -1 when edge is child→parent (we traverse against forward direction)
    """
    # Build undirected adjacency from forward edges
    adj = [[] for _ in range(num_nodes)]
    n_edges = edge_index_fwd.size(1)
    for e in range(n_edges):
        u = edge_index_fwd[0, e].item()
        v = edge_index_fwd[1, e].item()
        adj[u].append((v, e, +1.0))   # forward direction: θ_v = θ_u - (+1)*Δθ_e
        adj[v].append((u, e, -1.0))   # reverse direction:  θ_u = θ_v - (-1)*Δθ_e
    
    visited = [False] * num_nodes
    visited[slack_idx] = True
    queue = deque([slack_idx])
    
    bfs_edge_idx = []
    bfs_signs = []
    bfs_node_order = []
    bfs_parent = []
    
    while queue:
        node = queue.popleft()
        for neighbor, edge_e, sign in adj[node]:
            if not visited[neighbor]:
                visited[neighbor] = True
                bfs_edge_idx.append(edge_e)
                bfs_signs.append(sign)
                bfs_node_order.append(neighbor)
                bfs_parent.append(node)
                queue.append(neighbor)
    
    assert len(bfs_node_order) == num_nodes - 1, (
        f"BFS did not reach all nodes: visited {len(bfs_node_order)+1}/{num_nodes}"
    )
    
    return (
        torch.tensor(bfs_edge_idx, dtype=torch.long),
        torch.tensor(bfs_signs, dtype=torch.float32),
        torch.tensor(bfs_node_order, dtype=torch.long),
        torch.tensor(bfs_parent, dtype=torch.long),
    )


def reconstruct_theta_from_delta(delta_theta_fwd, bfs_edge_idx, bfs_signs,
                                  bfs_node_order, bfs_parent, num_nodes):
    """
    Reconstruct absolute θ from edge Δθ predictions using pre-computed BFS order.
    Differentiable w.r.t. delta_theta_fwd (chain of additions).
    
    Args:
        delta_theta_fwd: [E_fwd] predicted Δθ per forward edge
        bfs_edge_idx:    [N-1] edge indices in BFS traversal order
        bfs_signs:       [N-1] signs (+1/-1)
        bfs_node_order:  [N-1] child node indices in BFS visit order
        bfs_parent:      [N-1] parent node index for each child
        num_nodes:       int, total nodes
    
    Returns:
        theta: [N] reconstructed absolute angles (θ_slack = 0)
    """
    theta = torch.zeros(num_nodes, device=delta_theta_fwd.device, dtype=delta_theta_fwd.dtype)
    
    for i in range(len(bfs_node_order)):
        child = bfs_node_order[i].item()
        parent_node = bfs_parent[i].item()
        e = bfs_edge_idx[i]
        s = bfs_signs[i]
        theta[child] = theta[parent_node] - s * delta_theta_fwd[e]
    
    return theta


## GNN Architecture

In [ ]:
# =============================================================================
# SECTION 4: DATASET
# =============================================================================

import torch
from torch_geometric.data import Data, Dataset

class PowerFlowDataset(Dataset):
    """
    One data object per (network, snapshot) pair.

    The graph topology reflects the physical network topology:
    - buses → GNN nodes
    - lines + transformers → GNN edges (bidirectional)

    Each graph carries:
      x         : [n_buses, 7 or 8] node features (bus type flags, known P/Q/V/Vang, optional p_nom_share)
      y         : [n_buses, 4]       targets (vmag, vang, p, q)
      edge_index: [2, n_edges]       bidirectional
      edge_attr : [n_edges, 6]       edge features (r, x, b/2, tap, g_series, b_series)
      masks     : slack_mask, pv_mask, pq_mask
      y_line_p  : [n_fwd_lines]      forward-direction real power per supervised edge (PTDF supervision)
      ptdf_edge_row_idx : [n_edges]  PTDF row index for each edge (-1 for non-supervised/reverse)  # STSI 26.04.07
      network_idx: int
    """

    def __init__(
        self,
        networks: list,
        use_edge_features: bool = True,
        use_pnet_balance: bool = True,   # [STSI100526] kept for backfill compat (ignored internally)
        use_pnom_share: bool = False,     # [STSI100526] col7: p_nom_share (was col7=p_bal + col8=pnom_share before 100526)
        transform=None,
        pre_transform=None,
        ptdf_branch_mode: str = "lines",  # "lines" or "all" — controls PTDF supervision scope
        angle_mode: str = "node",          # [STSI 250526] "node" (default) or "edge_delta"
    ):
        super().__init__(root=None, transform=transform, pre_transform=pre_transform)
        self.networks          = networks
        self.use_edge_features = use_edge_features
        self.use_pnom_share    = use_pnom_share
        self.ptdf_branch_mode  = ptdf_branch_mode #STSI260402: pass ptdf_branch_mode to _create_graph_data for edge feature control
        self.angle_mode        = angle_mode  # [STSI 250526] "node" or "edge_delta"

        # Build flat index: list of (network_idx, timestep_idx)
        self._index = []
        for net_idx, net in enumerate(networks):
            for t_idx in range(len(net.snapshots)):
                self._index.append((net_idx, t_idx))

    def len(self):
        return len(self._index)

    def get(self, idx):
        net_idx, t_idx = self._index[idx]
        network = self.networks[net_idx]
        data = self._create_graph_data(network, t_idx, net_idx)
        return data

    def _create_graph_data(self, network: pypsa.Network, t_idx: int, net_idx: int) -> Data:
        buses     = list(network.buses.index)
        n_buses   = len(buses)
        bus_to_i  = {b: i for i, b in enumerate(buses)}
        snapshot  = network.snapshots[t_idx]

        # ── Bus type flags ────────────────────────────────────────────────────
        is_slack = torch.zeros(n_buses, dtype=torch.float)
        is_pv    = torch.zeros(n_buses, dtype=torch.float)
        is_pq    = torch.zeros(n_buses, dtype=torch.float)

        slack_mask = torch.zeros(n_buses, dtype=torch.bool)
        pv_mask    = torch.zeros(n_buses, dtype=torch.bool)
        pq_mask    = torch.zeros(n_buses, dtype=torch.bool)

        for gen_name, gen in network.generators.iterrows():
            bus_name = gen["bus"]
            if bus_name not in bus_to_i:
                continue
            i = bus_to_i[bus_name]
            ctrl = gen["control"]
            if ctrl == "Slack":
                is_slack[i] = 1.0
                slack_mask[i] = True
            elif ctrl == "PV":
                is_pv[i] = 1.0
                pv_mask[i] = True

        # Buses with no generator → PQ
        for i in range(n_buses):
            if not slack_mask[i] and not pv_mask[i]:
                is_pq[i]   = 1.0
                pq_mask[i] = True

        # ── Bus injections (P, Q) from PyPSA post-solve ───────────────────────
        # STSI260404: BUGFIX — use .loc[snapshot, buses] to force column order
        # to match network.buses.index.  Without this, buses_t columns may be
        # in lexicographic order (e.g. "Bus 10" before "Bus 2"), mis-aligning
        # P/Q/Vang with bus-type masks and the Y-matrix built from buses.index.
        p_bus = torch.tensor(
            network.buses_t.p.loc[snapshot, buses].values, dtype=torch.float
        )
        q_bus = torch.tensor(
            network.buses_t.q.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Voltages ─────────────────────────────────────────────────────────────
        v_mag = torch.tensor(
            network.buses_t.v_mag_pu.loc[snapshot, buses].values, dtype=torch.float
        )
        v_ang = torch.tensor(
            network.buses_t.v_ang.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Input features: GNN receives known values, zeros where unknown ────
        # Known before solve: bus type flags, P_inj for PQ+PV, Q_inj for PQ,
        #                     Vmag for PV+Slack, Vang for Slack (=0)
        #
        # x layout: [is_slack(0), is_PV(1), is_PQ(2), P(3), Q(4), Vmag(5), Vang(6)]
        #
        # NOTE: Slack P input x[:,3]=0 is a known issue — the model sees no
        # varying signal for slack P.  # [STSI100526] removed p_net_balance (was here)
        x_p    = p_bus.clone()
        x_q    = q_bus.clone()
        x_vmag = v_mag.clone()
        x_vang = v_ang.clone()

        # Mask unknowns to zero in input (prevents trivial leakage)
        x_p[slack_mask]    = 0.0   # Slack P unknown (to be predicted)
        x_q[slack_mask]    = 0.0   # Slack Q unknown
        x_q[pv_mask]       = 0.0   # PV Q unknown
        x_vmag[pq_mask]    = 0.0   # PQ Vmag unknown
        x_vang[pq_mask]    = 0.0   # PQ Vang unknown
        x_vang[pv_mask]    = 0.0   # PV Vang unknown

        x = torch.stack([is_slack, is_pv, is_pq, x_p, x_q, x_vmag, x_vang], dim=1)

        if self.use_pnom_share:  # [STSI100526] removed p_bal_col (redundant); GNN computes load sum internally
            # col7: static p_nom share per gen bus (p_nom_i / Σp_nom)
            # Tells the model each generator's relative capacity for slack distribution.
            p_nom_share_col = torch.zeros(n_buses, 1)
            p_noms = network.generators["p_nom"].values.astype(float)
            total_p_nom = float(p_noms.sum()) if p_noms.sum() > 0 else 1.0
            for gen_name, gen in network.generators.iterrows():
                b = gen["bus"]
                if b in bus_to_i:
                    p_nom_share_col[bus_to_i[b], 0] += gen["p_nom"] / total_p_nom
            x = torch.cat([x, p_nom_share_col], dim=1)

        # ── Target: full post-solve state for all buses ──────────────────────────
        y = torch.stack([v_mag, v_ang, p_bus, q_bus], dim=1)

        # ── Edge construction (lines + transformers) ──────────────────────────────
        edge_index_list = []
        edge_attr_list  = []
        forward_edge_mask = []   # True for first direction (not reverse duplicate)
        dc_flow_mask = []        # [STSI 020626]:True only for forward LINE edges (not transformers) — for DC flow loss
        fwd_line_idx_list  = []   # one entry per forward supervised edge
        ptdf_edge_row_idx_list = []  # STSI 26.04.07: PTDF row index per edge for learned_edge mode

        branches = []
        #STSI260402: control which branches are included in PTDF supervision via self.ptdf_branch_mode
        # lines always included
        for line_name, line in network.lines.iterrows():
            if line["bus0"] in bus_to_i and line["bus1"] in bus_to_i:
                branches.append(
                    (
                        line.name,  # name (not used in PTDF but useful for debugging)
                        line["bus0"], line["bus1"],
                        line["r"], line["x"],
                        line.get("b", 0.0), 1.0, "line"
                    )
                )

        # STSI260404: BUGFIX — transformers must ALWAYS be added as graph edges
        # for GNN message passing, regardless of ptdf_branch_mode.
        # ptdf_branch_mode only controls which branches are supervised by PTDF
        # loss (via forward_edge_mask / fwd_line_idx_list), NOT graph topology.
        # Previously gated behind `if self.ptdf_branch_mode == "all":` which
        # silently severed the graph when mode was "lines" (the default).
        for tr_name, tr in network.transformers.iterrows():
            if tr["bus0"] in bus_to_i and tr["bus1"] in bus_to_i:
                tap = tr.get("tap_ratio", 1.0)
                tap = tap if not pd.isna(tap) else 1.0
                branches.append(
                    (
                        tr.name,  # name (not used in PTDF but useful for debugging)
                        tr["bus0"], tr["bus1"],
                        tr["r"], tr["x"],
                        tr.get("b", 0.0), float(tap), "trafo"
                    )
                )


        for branch_idx, (name, b0, b1, r, x_val, b_val, tap, btype) in enumerate(branches):
            i0, i1 = bus_to_i[b0], bus_to_i[b1]
            z      = complex(r, x_val)
            y_ser  = (1.0 / z) if abs(z) > 1e-12 else 0.0
            g_ser, b_ser = y_ser.real, y_ser.imag

            if self.use_edge_features:
                attr = [r, x_val, b_val / 2.0, tap, g_ser, b_ser]
            else:
                attr = [1.0]

            # Forward edge
            edge_index_list.append([i0, i1])
            edge_attr_list.append(attr)

            # PTDF supervision: only lines (or all branches) get True in forward_edge_mask
            is_supervised = (btype == "line") if self.ptdf_branch_mode == "lines" else True
            forward_edge_mask.append(is_supervised)
            dc_flow_mask.append(btype == "line")  # [STSI 020626]:DC mask excludes transformers
            if is_supervised:
                fwd_line_idx_list.append(branch_idx)
                ptdf_edge_row_idx_list.append(branch_idx)  # STSI 26.04.07: supervised forward edge
            else:
                ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: unsupervised forward edge

            # Reverse edge
            edge_index_list.append([i1, i0])
            edge_attr_list.append(attr)
            forward_edge_mask.append(False)
            dc_flow_mask.append(False)  # [STSI 020626]:reverse edge always False
            ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: reverse edge

        if edge_index_list:
            edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float)
            fwd_mask   = torch.tensor(forward_edge_mask, dtype=torch.bool)
            dc_fwd_mask = torch.tensor(dc_flow_mask, dtype=torch.bool)  # [STSI 020626]:DC-only line mask
            line_idx   = torch.tensor(fwd_line_idx_list, dtype=torch.long)
            ptdf_edge_row_idx = torch.tensor(ptdf_edge_row_idx_list, dtype=torch.long)  # STSI 26.04.07
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr  = torch.zeros((0, len(attr) if branches else 1), dtype=torch.float)
            fwd_mask   = torch.zeros(0, dtype=torch.bool)
            dc_fwd_mask = torch.zeros(0, dtype=torch.bool)  # [STSI 020626]:DC-only line mask (empty)
            line_idx   = torch.zeros(0, dtype=torch.long)
            ptdf_edge_row_idx = torch.zeros(0, dtype=torch.long)  # STSI 26.04.07


        try:
            ptdf_matrix = compute_ptdf_matrix(network, ptdf_branch_mode=self.ptdf_branch_mode)  # [STSI 260526]:pass branch mode 
            y_ptdf = torch.tensor(ptdf_matrix, dtype=torch.float)
        except Exception:
            n_lines_fb = len(network.lines)
            n_branches_fb = n_lines_fb + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines_fb
            y_ptdf = torch.zeros((n_branches_fb, n_buses), dtype=torch.float)  # [STSI 260526]:use branch count in fallback 

        # STSI260402 Assertions — verify PTDF supervision scope and consistency
        n_lines = len(network.lines)
        if self.ptdf_branch_mode == "lines":
            assert int(fwd_mask.sum()) == n_lines, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines {n_lines}"
        else:  # ptdf_branch_mode == "all": lines + transformers
            n_trafos = len(network.transformers)
            assert int(fwd_mask.sum()) == n_lines + n_trafos, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines+n_trafos {n_lines + n_trafos}"
        expected_ptdf_rows = n_lines + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines  # [STSI 260526]: expected PTDF rows based on supervised branches
        assert y_ptdf.shape[0] == expected_ptdf_rows, \
            f"y_ptdf rows {y_ptdf.shape[0]} != expected {expected_ptdf_rows} (mode={self.ptdf_branch_mode})"

        # ── Line active power flows ───────────────────────────────────────────
        if not network.lines_t.p0.empty:
            y_line_p = torch.tensor(
                network.lines_t.p0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_p = torch.zeros(len(network.lines), dtype=torch.float)

        # [STSI 260526]: Append transformer flows when supervising all branches 
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.p0.empty:
                y_trafo_p = torch.tensor(
                    network.transformers_t.p0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_p = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_p = torch.cat([y_line_p, y_trafo_p])

        # ── Q line flows (for AC per-edge flow loss) ──── [STSI 020626]
        if not network.lines_t.q0.empty:
            y_line_q = torch.tensor(
                network.lines_t.q0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_q = torch.zeros(len(network.lines), dtype=torch.float)

        # Append transformer Q flows when supervising all branches
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.q0.empty:
                y_trafo_q = torch.tensor(
                    network.transformers_t.q0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_q = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_q = torch.cat([y_line_q, y_trafo_q])


        # [STSI 260526]: Precompute diagonal admittance for edge-local physics 
        # g_diag[i] = Σ_j g_series_ij ; b_diag[i] = Σ_j (b_series_ij + b_half_ij)
        if edge_index.size(1) > 0 and edge_attr.size(0) > 0:
            from torch_geometric.utils import scatter
            _src_nodes = edge_index[0]  # [2E] source node indices
            _g_ser_all = edge_attr[:, 4]  # [2E] series conductance
            _b_ser_all = edge_attr[:, 5]  # [2E] series susceptance
            _b_half_all = edge_attr[:, 2]  # [2E] line charging b/2 per end
            g_diag = scatter(_g_ser_all, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
            b_diag = scatter(_b_ser_all + _b_half_all, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
        else:
            g_diag = torch.zeros(n_buses, dtype=torch.float)
            b_diag = torch.zeros(n_buses, dtype=torch.float)

        # ── Assemble Data object ──────────────────────────────────────────────
        data = Data(
            x          = x,
            y          = y,
            edge_index = edge_index,
            edge_attr  = edge_attr,
        )

        # Bus-type masks — node-level bool tensors (critical for collate + loss)
        data.slack_mask = slack_mask   # [n_nodes] bool
        data.pv_mask    = pv_mask      # [n_nodes] bool
        data.pq_mask    = pq_mask      # [n_nodes] bool

        # Variable-size targets (handled specially in collate_with_ptdf)
        data.y_ptdf      = y_ptdf
        data.y_line_p    = y_line_p
        data.y_line_q    = y_line_q  # [STSI 020626]: reactive flow target for AC per-edge loss
        data.forward_edge_mask = fwd_mask
        data.dc_flow_mask = dc_fwd_mask  # [STSI 020626]:lines-only mask for DC flow loss (excludes transformers)
        data.g_diag = g_diag  # [STSI 260526]: diagonal conductance for edge-local physics
        data.b_diag = b_diag  # [STSI 260526]: diagonal susceptance for edge-local physics
        data.ptdf_line_index = line_idx
        data.ptdf_edge_row_idx = ptdf_edge_row_idx  # STSI 26.04.07: per-edge PTDF row index for learned_edge mode

        # Graph-level metadata
        data.network_idx    = torch.tensor([net_idx], dtype=torch.long)

        # ── Δθ target and BFS metadata (for angle_mode="edge_delta" or "both") ── [STSI 250526]
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:store Δθ targets for both modes
            # All forward edges (lines + transformers) for full BFS connectivity
            all_fwd_mask = torch.zeros(edge_index.size(1), dtype=torch.bool)
            all_fwd_mask[::2] = True  # even indices = forward edges
            edge_index_fwd = edge_index[:, all_fwd_mask]
            
            # Compute Δθ target: θ_from - θ_to for each forward edge
            theta_true = y[:, 1]  # absolute angles from power flow solution
            y_delta_theta = theta_true[edge_index_fwd[0]] - theta_true[edge_index_fwd[1]]
            data.y_delta_theta = y_delta_theta  # [E_fwd]
            
            # Precompute BFS order from slack
            slack_idx_val = int(slack_mask.nonzero(as_tuple=True)[0][0])
            bfs_edge_idx, bfs_signs, bfs_node_order, bfs_parent = precompute_bfs_order(
                edge_index_fwd, slack_idx_val, n_buses
            )
            data.bfs_edge_idx = bfs_edge_idx
            data.bfs_signs = bfs_signs
            data.bfs_node_order = bfs_node_order
            data.bfs_parent = bfs_parent

        return data

def collate_with_ptdf(batch):
    """
    Custom collate function for variable-size PTDF matrices and per-graph attributes.
    
    FIX 2026-03-25: Explicitly propagate slack_mask, pv_mask, pq_mask so that
    _masked_mse_loss and physics_informed_loss_batch receive valid mask tensors.
    Without this, all masked losses silently return 0.

    """
    # ── Variable-size lists: remove before PyG batching ──────────────────────
    y_ptdf_list   = [data.y_ptdf   for data in batch]
    y_line_p_list = [data.y_line_p for data in batch]
    y_line_q_list = [data.y_line_q for data in batch]  # [STSI 020626]
    ptdf_line_index_list = [data.ptdf_line_index for data in batch]


    # [STSI 250526]:also remove BFS metadata before PyG batching (variable-size per graph)
    bfs_keys = ['bfs_edge_idx', 'bfs_signs', 'bfs_node_order', 'bfs_parent', 'y_delta_theta']
    bfs_data = {k: [] for k in bfs_keys}
    for data in batch:
        for k in bfs_keys:
            if hasattr(data, k):
                bfs_data[k].append(getattr(data, k))
                delattr(data, k)

    for data in batch:
        del data.y_ptdf
        del data.y_line_p #STSI260402: added deletion of y_line_p to prevent PyG from trying to batch it and causing errors due to variable sizes
        del data.y_line_q  # [STSI 020626]
        del data.ptdf_line_index
    # ── PyG auto-batches all node-level tensors (x, y, edge_index, etc.) ─────
    # slack_mask / pv_mask / pq_mask are node-level bool tensors stored on each
    # Data object by PowerFlowDataset — Batch.from_data_list will concat them
    # along the node dimension automatically IF they are present.
    # ptdf_edge_row_idx is edge-level [n_edges] long — also auto-concatenated (STSI 26.04.07)
    batched = Batch.from_data_list(batch)

    # ── Restore variable-size list attributes ────────────────────────────────
    batched.y_ptdf_list   = y_ptdf_list
    batched.y_line_p_list = y_line_p_list
    batched.y_line_q_list = y_line_q_list  # [STSI 020626]
    batched.ptdf_line_index_list = ptdf_line_index_list

    # [STSI 250526]:restore BFS metadata as lists (for angle_mode="edge_delta")
    for k, v_list in bfs_data.items():
        if v_list:
            setattr(batched, k, v_list)

    # Restore on originals for re-use in dataset
    for i, (data, y_ptdf, y_line_p, y_line_q, ptdf_idx) in enumerate(zip(batch, y_ptdf_list, y_line_p_list, y_line_q_list, ptdf_line_index_list)):  # [STSI 020626]:added y_line_q
        data.y_ptdf   = y_ptdf
        data.y_line_p = y_line_p
        data.y_line_q = y_line_q  # [STSI 020626]
        data.ptdf_line_index = ptdf_idx
        # [STSI 250526]:restore BFS attrs on originals
        for k in bfs_keys:
            if bfs_data[k]:
                setattr(data, k, bfs_data[k][i])

    # ── Defensive check: abort loudly if masks are missing ───────────────────
    for mask_attr in ("slack_mask", "pv_mask", "pq_mask"):
        if not hasattr(batched, mask_attr):
            raise RuntimeError(
                f"collate_with_ptdf: batched object missing '{mask_attr}'. "
                f"Ensure PowerFlowDataset stores data.{mask_attr} as a node-level "
                f"bool tensor (shape [n_nodes]) on every Data object."
            )

    return batched

# training data validation
def validate_training_data(networks: list, name: str = "dataset", n_sample: int = 2) -> bool:
    """
    Validate that a list of PyPSA networks is compatible with PowerFlowDataset.
    Checks all fields accessed by _create_graph_data and compute_ptdf_matrix.

    Parameters
    ----------
    networks : list of pypsa.Network (post-PF)
    name     : label for print output
    n_sample : how many networks to inspect in detail

    Returns
    -------
    bool : True if all checks pass, False if any critical issue found
    """
    import numpy as np
    import torch

    issues   = []   # critical — will break the loader
    warnings = []   # non-critical — may degrade training

    print(f"\n{'='*60}")
    print(f"Validating '{name}': {len(networks)} networks")
    print(f"{'='*60}")

    if len(networks) == 0:
        print("CRITICAL: Empty network list")
        return False

    for net_idx, net in enumerate(networks[:n_sample]):
        tag = f"[net {net_idx}]"
        print(f"\n--- Network {net_idx} ---")

        # ── 1. Topology ──────────────────────────────────────────
        n_buses  = len(net.buses)
        n_lines  = len(net.lines)
        n_trafos = len(net.transformers)
        n_gens   = len(net.generators)
        n_loads  = len(net.loads)
        n_snaps  = len(net.snapshots)
        print(f"  Buses={n_buses}, Lines={n_lines}, Trafos={n_trafos}, "
              f"Gens={n_gens}, Loads={n_loads}, Snapshots={n_snaps}")

        if n_buses == 0:
            issues.append(f"{tag} No buses")
        if n_lines + n_trafos == 0:
            issues.append(f"{tag} No lines or transformers — edge_index will be empty")
        if n_snaps == 0:
            issues.append(f"{tag} No snapshots — dataset will be empty")
        if n_gens == 0:
            warnings.append(f"{tag} No generators — bus type will default to PQ everywhere")

        # ── 2. Required buses_t tables (read in _create_graph_data) ──
        print(f"\n  buses_t tables:")
        for tbl, attr in [("buses_t.p",       "p"),
                           ("buses_t.q",       "q"),
                           ("buses_t.v_mag_pu","v_mag_pu"),
                           ("buses_t.v_ang",   "v_ang")]:
            df = getattr(net.buses_t, attr)
            ok = not df.empty and df.shape == (n_snaps, n_buses)
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "❌"
            if not ok:
                issues.append(f"{tag} {tbl} missing or wrong shape: {df.shape} "
                              f"(expect {(n_snaps, n_buses)})")
            elif not finite:
                issues.append(f"{tag} {tbl} contains NaN/Inf")
            print(f"    {status} {tbl}: shape={df.shape}, finite={finite}")

        # ── 3. Voltage range sanity ───────────────────────────────
        if not net.buses_t.v_mag_pu.empty:
            v = net.buses_t.v_mag_pu.values.flatten()
            v_ok = np.all((v > 0.5) & (v < 1.5))
            status = "✅" if v_ok else "⚠️"
            print(f"    {status} v_mag_pu range: [{v.min():.4f}, {v.max():.4f}]")
            if not v_ok:
                warnings.append(f"{tag} v_mag_pu out of [0.5, 1.5]: "
                                f"min={v.min():.4f} max={v.max():.4f}")

        # ── 4. lines_t.p0 (used for y_line_p) ────────────────────
        print(f"\n  lines_t tables:")
        p0 = net.lines_t.p0
        ok_p0 = not p0.empty and p0.shape == (n_snaps, n_lines)
        finite_p0 = np.isfinite(p0.values).all() if ok_p0 else False
        status = "✅" if (ok_p0 and finite_p0) else "❌"
        if not ok_p0:
            issues.append(f"{tag} lines_t.p0 missing or wrong shape: {p0.shape} "
                         f"(expect {(n_snaps, n_lines)})")
        elif not finite_p0:
            issues.append(f"{tag} lines_t.p0 contains NaN/Inf")
        print(f"    {status} lines_t.p0: shape={p0.shape}, finite={finite_p0}")

        # ── 5. generators_t.p and q (used for node features) ─────
        print(f"\n  generators_t tables:")
        for attr in ["p", "q"]:
            df = getattr(net.generators_t, attr)
            ok = not df.empty
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "⚠️"
            if not ok:
                warnings.append(f"{tag} generators_t.{attr} empty")
            print(f"    {status} generators_t.{attr}: shape={df.shape}, finite={finite}")

        # ── 6. Generator control column ───────────────────────────
        print(f"\n  Generator control types:")
        if "control" not in net.generators.columns:
            issues.append(f"{tag} net.generators missing 'control' column")
        else:
            ctrl_counts = net.generators["control"].value_counts().to_dict()
            has_slack = "Slack" in ctrl_counts
            status = "✅" if has_slack else "❌"
            print(f"    {status} control types: {ctrl_counts}")
            if not has_slack:
                issues.append(f"{tag} No Slack generator — bus type encoding will be wrong")

        # ── 7. Line parameters (edge_attr: r, x, b, s_nom) ───────
        print(f"\n  Line parameters:")
        for col in ["r", "x", "b", "s_nom"]:
            present = col in net.lines.columns
            if present:
                vals = net.lines[col].values.astype(float)
                finite = np.isfinite(vals).all()
                status = "✅" if finite else "❌"
                print(f"    {status} lines.{col}: min={vals.min():.5f} "
                      f"max={vals.max():.5f}")
                if not finite:
                    issues.append(f"{tag} lines.{col} contains NaN/Inf")
                if col in ["r", "x"] and (vals <= 0).any():
                    warnings.append(f"{tag} lines.{col} has zero/negative values")
            else:
                issues.append(f"{tag} lines.{col} column missing")
                print(f"    ❌ lines.{col}: MISSING")

        # ── 8. Bus–line connectivity (bus0/bus1 in buses.index) ───
        missing_buses = set()
        for col in ["bus0", "bus1"]:
            missing = set(net.lines[col]) - set(net.buses.index)
            missing_buses |= missing
        if missing_buses:
            issues.append(f"{tag} Lines reference buses not in buses.index: "
                         f"{missing_buses}")
        else:
            print(f"\n    ✅ All line endpoints exist in buses.index")

        # ── 9. Simulate one graph build (catches index errors) ────
        print(f"\n  Graph construction smoke test (snapshot 0):")
        try:
            snap = net.snapshots[0]
            bus_to_idx = {b: i for i, b in enumerate(net.buses.index)}
            edges = []
            for _, line in net.lines.iterrows():
                i = bus_to_idx[line.bus0]
                j = bus_to_idx[line.bus1]
                edges += [[i, j], [j, i]]
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            # STSI260404: BUGFIX — align buses_t columns to buses.index order
            x_check = np.column_stack([
                net.buses_t.v_mag_pu.loc[snap, list(net.buses.index)].values,
                net.buses_t.v_ang.loc[snap, list(net.buses.index)].values,
                net.buses_t.p.loc[snap, list(net.buses.index)].values,
                net.buses_t.q.loc[snap, list(net.buses.index)].values,
            ])
            assert np.isfinite(x_check).all(), "Non-finite node features"
            print(f"    ✅ edge_index: {edge_index.shape}, "
                  f"node features: {x_check.shape}, all finite")
        except Exception as e:
            issues.append(f"{tag} Graph construction failed: {e}")
            print(f"    ❌ Graph construction failed: {e}")

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"SUMMARY for '{name}'")
    if issues:
        print(f"\n❌ {len(issues)} CRITICAL issue(s):")
        for iss in issues:
            print(f"   • {iss}")
    if warnings:
        print(f"\n⚠️  {len(warnings)} warning(s):")
        for w in warnings:
            print(f"   • {w}")
    if not issues and not warnings:
        print("✅ All checks passed — dataset is compatible with PowerFlowDataset")
    elif not issues:
        print("✅ No critical issues — dataset should load correctly")

    print(f"{'='*60}\n")
    return len(issues) == 0




In [ ]:
# =============================================================================
# SECTION 5: GNN MODEL
# =============================================================================

class PowerFlowGNN(nn.Module):
    """
    Graph Attention Network (GATv2) for power flow prediction.
    Predicts per-bus: [vmag, vang, P, Q]
    Predicts per-edge bilinear PTDF: hedges @ W @ hnodes.T
    """

    def __init__(
        self,
        node_features: int = 7,
        edge_features: int = 4,
        hidden_dim: int = 64, # Hiddem dimensions means the size of the feature vectors that are passed between layers in the GNN. A larger hidden_dim allows the model to capture more complex relationships, but also increases computational cost and risk of overfitting. 64 is a common choice for a balance between expressiveness and efficiency.
        num_layers: int = 3,# Number of GNN layers (graph convolutional layers). More layers allow the model to capture more complex interactions between nodes, but can also lead to over-smoothing where node features become too similar. 3 layers is a common choice for many graph tasks.
        heads: int = 4,# Number of attention heads in GAT layers. Multiple heads allow the model to attend to different aspects of the graph structure simultaneously. 4 heads is a common choice that provides a good balance between expressiveness and computational cost.
        dropout: float = 0.0,
        conv_type: str = "gatv2", # Placeholder for potential future extension to other convolution types
        head_mode:str = "standard", # or with new encoder
        # ── Conv block flags (independent toggles, replace conv_mode) ── [STSI 120526]:refactored conv_mode→independent flags
        use_residual: bool = False,
        norm_type: str = None,         # None | "layer" | "graph"
        activation: str = "leaky_relu",  # "leaky_relu" | "gelu" | "relu" | "elu" | "silu"
        use_dropout: bool = False,
        drop_rate: float = 0.1,
        # ── Deprecated shorthand ──
        conv_mode: str = None,         # if set, overrides individual flags
        # ── Edge Δθ prediction ── [STSI 250526]: added angle_mode to control whether we predict node angles (standard) or edge angle differences (edge_delta)
        angle_mode: str = "node",      # "node" (default), "edge_delta", or "both"  # [STSI 020626]:added "both" for dual-head
        # ── V_mag residual ── [STSI 280526]: predict ΔV from flat baseline instead of absolute V_mag
        vmag_mode: str = "absolute",   # "absolute" (default) or "residual"
        # ── Global graph context ── [STSI 290526]: whole-graph awareness via mean+max pooling
        use_global_pool: bool = False,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.conv_type = conv_type
        self.head_mode = head_mode
        self.angle_mode = angle_mode  # [STSI 250526]:new angle_mode option
        self.vmag_mode = vmag_mode    # [STSI 280526]:residual V_mag prediction
        self.use_global_pool = use_global_pool  # [STSI 290526]
        # ── Deprecated conv_mode shorthand ── [STSI 120526]:backward compat mapping
        if conv_mode is not None:
            _map = {
                "old":            (False, None,    "leaky_relu", False),
                "residual":       (True,  None,    "gelu",       False),
                "res_norm":       (True,  "layer", "gelu",       False),
                "res_norm_drop":  (True,  "layer", "gelu",       True),
            }
            use_residual, norm_type, activation, use_dropout = _map[conv_mode]

        self.use_residual = use_residual
        self.norm_type = norm_type
        self.use_dropout = use_dropout
        self.drop_rate = drop_rate
        self.activation_name = activation
        # 1. Node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # 2. GAT convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(self._make_conv(hidden_dim, hidden_dim, edge_features))

        # 2b. Optional normalization + dropout for conv block [STSI 120526]:independent flags
        if self.norm_type is not None:
            if self.norm_type == "layer":
                self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
            elif self.norm_type == "graph":
                from torch_geometric.nn.norm import GraphNorm
                self.norms = nn.ModuleList([GraphNorm(hidden_dim) for _ in range(num_layers)])
            else:
                raise ValueError(f"Unknown norm_type: {self.norm_type!r}")
        if self.use_dropout:
            self.drop = nn.Dropout(self.drop_rate)


        # 2c. Optional global pooling projection [STSI 290526]: graph-level context → per-node
        if self.use_global_pool:
            self.global_proj = nn.Linear(2 * hidden_dim, hidden_dim)

        # Activation function lookup [STSI 120526]:configurable activation
        _act_map = {
            "leaky_relu": F.leaky_relu,
            "gelu": F.gelu,
            "relu": F.relu,
            "elu": F.elu,
            "silu": F.silu,
        }
        if activation not in _act_map:
            raise ValueError(f"Unknown activation: {activation!r}")
        self.activation_fn = _act_map[activation]

        # 3. Output layers for different predictions
        if head_mode=="standard":
            if angle_mode == "edge_delta":  # [STSI 250526]:no angle head for node prediction
                self.vmag_pred = nn.Linear(hidden_dim, 1)
                self.p_pred = nn.Linear(hidden_dim, 1)
                self.q_pred = nn.Linear(hidden_dim, 1)
            else:
                # one prediction head for each of the properties we want to predict
                self.vmag_pred = nn.Linear(hidden_dim, 1)   # Voltage magnitude
                self.vang_pred = nn.Linear(hidden_dim, 1)   # Voltage angle
                self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
                self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        if head_mode=="with_encoder":
            if angle_mode == "edge_delta":  
                self.pq_head = nn.Linear(hidden_dim, 1)    # PQ: predict Vmag only
                self.pv_head = nn.Linear(hidden_dim, 1)    # PV: predict Q only
                self.slack_head = nn.Linear(hidden_dim, 2) # Slack: predict P, Q
            else:
                self.pq_head = nn.Linear(hidden_dim, 2)      # for PQ nodes
                self.pv_head = nn.Linear(hidden_dim, 2)      # For PV nodes
                self.slack_head = nn.Linear(hidden_dim, 2)   # For Slack nodes

        # STSI 26.02.11: Edge embedding MLP: (h_src || h_dst || edge_attr) -> hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_features, hidden_dim),
            nn.LeakyReLU(),
        )

        # [STSI 250526]:edge angle head for edge_delta/both mode
        if angle_mode in ("edge_delta", "both"):  # [STSI 020626]:extended to support dual-head "both" mode
            self.edge_angle_pred = nn.Linear(hidden_dim, 1)

        # Bilinear PTDF parameter: h_edge^T W h_bus
        self.ptdf_W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        
        self._initialize_weights()

    def _make_conv(self, in_dim: int, out_dim: int, edge_features: int):
        if self.conv_type == "gatv2":
            return GATv2Conv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0, #self.dropout, # Dropout is handled separately in conv_mode, so set to 0 here to avoid double dropout when conv_mode="res_norm_drop"
            )
        elif self.conv_type == "transformer":  # [STSI120526] New conv_type option: dot-product attention, edge features in K+V, gated residual (beta=True)
            return TransformerConv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0,
                beta=not self.use_residual,  # [STSI 120526]:avoid double residual: external skip OR internal gated, not both
            )
        elif self.conv_type == "gcn":   # [STSI120526] DEPRECATED — no edge_attr support
            return GCNConv(in_dim, out_dim, add_self_loops=False)
        elif self.conv_type == "graphconv":   # [STSI120526] DEPRECATED — no edge_attr support
            return GraphConv(in_dim, out_dim)
        else:
            raise ValueError(f"Unknown conv_type: {self.conv_type}")

    def _apply_conv(self, conv, h, edge_index, edge_attr):
        """Apply a single conv layer, dispatching on conv type."""
        if isinstance(conv, (GATv2Conv, TransformerConv)):  # [STSI120526] Added TransformerConv to edge_attr dispatch
            return conv(h, edge_index, edge_attr)
        else:
            return conv(h, edge_index)

    def forward(self, data, return_embeddings=False):
        """
        Forward pass: embed -> GAT layers -> node heads + bilinear PTDF.

        Args:
            data: PyG Data/Batch object
            return_embeddings: if True, return (node_pred, h_nodes, h_edges)
                               for use in PTDF loss outside forward
        Returns:
            (node_pred [num_nodes, 4], ptdf_pred [num_edges, num_nodes])
            or (node_pred, h_nodes, h_edges) if return_embeddings=True
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Initial embedding
        h = self.node_embedding(x)

        # Conv layers — unified path with independent flags [STSI 120526]:replaced 4-way if/elif
        for i, conv in enumerate(self.convs):
            h_in = h
            h = self._apply_conv(conv, h, edge_index, edge_attr)
            if self.norm_type is not None:
                h = self.norms[i](h)
            h = self.activation_fn(h)
            if self.use_dropout:
                h = self.drop(h)
            if self.use_residual:
                h = h + h_in


        h_nodes = h  # Final node features after all GAT layers

        # Global pooling context injection [STSI 290526]: broadcast graph-level mean+max into per-node repr
        if self.use_global_pool:
            batch_idx = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(
                h_nodes.size(0), dtype=torch.long, device=h_nodes.device)
            h_mean = global_mean_pool(h_nodes, batch_idx)   # [num_graphs, hidden_dim]
            h_max = global_max_pool(h_nodes, batch_idx)     # [num_graphs, hidden_dim]
            h_global = self.global_proj(torch.cat([h_mean, h_max], dim=-1))  # [num_graphs, hidden_dim]
            h_nodes = h_nodes + h_global[batch_idx]  # additive residual injection


        # Node heads
        # Predictions are made using the output of the last graph convolution layer

        if self.head_mode=="standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]: node pred is [N,3]
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 3]
            else:
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                vang_pred = self.vang_pred(h_nodes)
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, vang_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 4]
        elif self.head_mode=="with_encoder":
            pq_mask = data.pq_mask.bool()
            pv_mask = data.pv_mask.bool()
            slack_mask = data.slack_mask.bool()
            if self.angle_mode == "edge_delta":  # [STSI 250526]: [N,3]=[Vmag,P,Q]
                node_pred = torch.zeros(h_nodes.size(0), 3, device=h_nodes.device)
                if pq_mask.any():
                    raw_vmag = self.pq_head(h_nodes[pq_mask]).squeeze(-1)  # Vmag
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        node_pred[pq_mask, 0] = raw_vmag + 1.0
                    else:
                        node_pred[pq_mask, 0] = raw_vmag
                if pv_mask.any():
                    node_pred[pv_mask, 2] = self.pv_head(h_nodes[pv_mask]).squeeze(-1)  # Q
                if slack_mask.any():
                    slack_out = self.slack_head(h_nodes[slack_mask])
                    node_pred[slack_mask, 1] = slack_out[:, 0]  # P
                    node_pred[slack_mask, 2] = slack_out[:, 1]  # Q
            else:
                node_pred = torch.zeros(h_nodes.size(0), 4, device=h_nodes.device)
                if pq_mask.any():
                    pq_out = self.pq_head(h_nodes[pq_mask])  # [N_pq, 2] = [Vmag, Vang]
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        pq_out = pq_out.clone()
                        pq_out[:, 0] = pq_out[:, 0] + 1.0
                    node_pred[pq_mask, 0:2] = pq_out
                if pv_mask.any():
                    pv_out=self.pv_head(h_nodes[pv_mask])  # Predict Vmag, Vang for PV nodes
                    node_pred[pv_mask,1]=pv_out[:,0]  # Vmag from PV head
                    node_pred[pv_mask,3]=pv_out[:,1]  # Q from PV head
                if slack_mask.any():
                    slack_out=self.slack_head(h_nodes[slack_mask])  # Predict Vmag, Vang for Slack nodes
                    node_pred[slack_mask, 2] = slack_out[:, 0]  # P from Slack head
                    node_pred[slack_mask, 3] = slack_out[:, 1]  # Q from Slack head

        # Edge embeddings
        row, col = data.edge_index
        h_src = h_nodes[row]   # [num_edges, hidden_dim]
        h_dst = h_nodes[col]   # [num_edges, hidden_dim]
        edge_input = torch.cat([h_src, h_dst, edge_attr], dim=-1)  # [num_edges, 2*hidden_dim + edge_features]
        h_edges = self.edge_mlp(edge_input)  # [num_edges, hidden_dim]

        # [STSI 250526]:edge angle prediction for edge_delta mode
        delta_theta_pred = None
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:dual-head produces Δθ in both modes
            # All forward edges (lines + transformers) — even indices
            all_fwd = torch.zeros(h_edges.size(0), dtype=torch.bool, device=h_edges.device)
            all_fwd[::2] = True
            h_edges_fwd = h_edges[all_fwd]    # [E_all_fwd, hidden_dim]
            delta_theta_pred = self.edge_angle_pred(h_edges_fwd).squeeze(-1)  # [E_fwd]

        if return_embeddings:
            return node_pred, h_nodes, h_edges, delta_theta_pred

        # Bilinear PTDF: H_W = h_edges @ W,  ptdf_pred = H_W @ h_nodes.T
        H_W = h_edges @ self.ptdf_W                   # [num_edges, hidden_dim]
        ptdf_pred = H_W @ h_nodes.T                   # [num_edges, num_nodes]
        return node_pred, ptdf_pred, delta_theta_pred

    def _initialize_weights(self, gain=1.0):
        """
        STSI 24.09.25: Enhanced weight initialization for the entire network.
        """
        # 1. Initialize node embedding layer (Added 24.09.25)
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)

        # 2. Initialize GAT convolution layers (Added 24.09.25)
        for conv in self.convs:
            if hasattr(conv, "lin_l") and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, "lin_r") and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, "lin_edge") and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
            # GAT layers have multiple linear transformations

        # 3. Initialize output layers
        _vmag_bias = 0.0 if self.vmag_mode == "residual" else 1.0  # [STSI 280526]
        if self.head_mode == "standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]no angle head, only Vmag, P, Q
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            else:
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.vang_pred, 0.0),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            for layer, bias_init in prediction_layers:
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.constant_(layer.bias, bias_init)
            # Zero-init vmag_pred weights so output starts exactly at bias [STSI 280526]
            nn.init.zeros_(self.vmag_pred.weight)
        elif self.head_mode == "with_encoder":
            for layer in (self.pq_head, self.pv_head, self.slack_head):
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.zeros_(layer.bias)
            # Vmag outputs (col 0 of pq_head) should start near 1.0 pu
            if self.angle_mode != "edge_delta":  # [STSI 250526]:only for 2-output pq_head
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)
            else:
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)  # single Vmag output
            nn.init.zeros_(self.pq_head.weight[0])  # Zero-init Vmag row [STSI 280526]

        # [STSI 250526]:initialize edge angle head  # [STSI 020626]:Fix — "both" also needs zero init
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:was == "edge_delta" only
            nn.init.zeros_(self.edge_angle_pred.weight)
            nn.init.zeros_(self.edge_angle_pred.bias)

        # [STSI 290526]: init global pooling projection
        if self.use_global_pool:
            nn.init.xavier_uniform_(self.global_proj.weight)


# =============================================================================
# SECTION 6: PHYSICS-INFORMED LOSS
# =============================================================================



def compute_power_flow_residual_from_pred(
    pred, x, Y_matrix, network, bus_masks=None,
    use_q_partial_mode: bool = False,
    w_P: float = 0.5,  # [STSI 210526]: weight on P residual (guides v_ang)
    w_Q: float = 0.5,  # [STSI 210526]: weight on Q residual (guides v_mag)
):
    """
    Compute AC power-flow residual.

    use_q_partial_mode=False (default/old behaviour):
        All bus types contribute both P and Q residuals.
    use_q_partial_mode=True:
        PV buses contribute only P residual (Q is a free variable for them).
    """
    Y_real, Y_imag = Y_matrix
    n_nodes = pred.size(0)
    device  = pred.device

    if bus_masks is None:
        slack_mask = (x[:, 0] == 1.0)
        pv_mask    = (x[:, 1] == 1.0)
        pq_mask    = (x[:, 2] == 1.0)
    else:
        slack_mask, pv_mask, pq_mask = bus_masks

    # ── Assemble injections ─────────────────────────────────────
    p_inj = torch.zeros(n_nodes, device=device)
    q_inj = torch.zeros(n_nodes, device=device)

    p_inj[pq_mask]    = x[pq_mask, 3]       # known P at PQ buses
    q_inj[pq_mask]    = x[pq_mask, 4]       # known Q at PQ buses
    p_inj[pv_mask]    = x[pv_mask, 3]       # known P at PV buses
    q_inj[pv_mask]    = pred[pv_mask, 3]    # predicted Q at PV buses
    p_inj[slack_mask] = pred[slack_mask, 2] # predicted P at slack
    q_inj[slack_mask] = pred[slack_mask, 3] # predicted Q at slack

    # ── Assemble voltages ────────────────────────────────────────
    v_mag = torch.zeros(n_nodes, device=device)
    v_ang = torch.zeros(n_nodes, device=device) #

    v_mag[pq_mask]    = pred[pq_mask, 0]    # predicted Vmag at PQ
    v_ang[pq_mask]    = pred[pq_mask, 1]    # predicted Vang at PQ
    v_mag[pv_mask]    = x[pv_mask, 5]       # known Vmag at PV
    v_ang[pv_mask]    = pred[pv_mask, 1]    # predicted Vang at PV
    v_mag[slack_mask] = x[slack_mask, 5]    # known Vmag at slack
    v_ang[slack_mask] = x[slack_mask, 6]    # known Vang (= 0) at slack

    # ── AC power calculation via Y-bus ──────────────────────────
    v_real = v_mag * torch.cos(v_ang)
    v_imag = v_mag * torch.sin(v_ang)

    I_real = torch.matmul(Y_real, v_real) - torch.matmul(Y_imag, v_imag)
    I_imag = torch.matmul(Y_real, v_imag) + torch.matmul(Y_imag, v_real)

    p_calc = v_real * I_real + v_imag * I_imag
    q_calc = v_imag * I_real - v_real * I_imag

    # ── Residuals ────────────────────────────────────────────────
    b_diag = torch.diag(Y_imag).abs().clamp(min=1e-6)  # [STSI 210526]: avoid division by zero for isolated buses
    # [STSI 270526]: divide by b_diag to normalize by typical power flow magnitudes at each bus, avoiding domination of high-injection buses in the loss
    p_residual = ((p_calc - p_inj) / b_diag) ** 2

    if use_q_partial_mode:
        # exclude PV buses from Q residual — they are Q-free by definition
        q_mask = pq_mask | slack_mask
        q_residual = torch.zeros(n_nodes, device=device)
        q_residual[q_mask] = ((q_calc[q_mask] - q_inj[q_mask]) / b_diag[q_mask]) ** 2
    else:
        #[STSI 270526]: Divide by b_diag for all buses in Q residual as well, for consistent normalization across P and Q and to prevent domination by high-injection buses.
        q_residual = ((q_calc - q_inj) / b_diag) ** 2 # include all bus types in the Q residual calculation

    p_res_mean = p_residual.mean().detach()  # [STSI 210526]: diagnostic, no grad
    q_res_mean = q_residual.mean().detach()  # [STSI 210526]: diagnostic, no grad
    physics_loss = torch.mean(w_P * p_residual + w_Q * q_residual)
    return physics_loss, p_res_mean, q_res_mean  # [STSI 210526]: 3-tuple return


@dataclass
class PhysicsConfig:
    """
    Toggles for every physics guidance term.
    Designed to be swept as a hyperparameter in run_hparam_sweep.
    """
    w_phys: float = 0.0                     # static weight on physics loss (used in "fixed" mode)

    # Dynamic loss weighting [STSI 200526]
    loss_weight_mode: str = "fixed"         # "fixed" | "adaptive"
    fraction_physics: float = 0.1           # target: physics = 10% of MSE (adaptive only)
    fraction_ptdf: float = 0.1              # target: PTDF = 10% of MSE (adaptive only)
    max_w_phys: float = 100.0               # cap for adaptive physics weight
    max_w_ptdf: float = 100.0               # cap for adaptive PTDF weight

    # Variant selection
    physics_mode: str = "rich"            # reserved — "rich" only; "simple" removed 2026-05-21  # [STSI 210526]

    # Shared options
    use_power_balance: bool = True          # AC residual on P,Q
    use_angle_ref_penalty: bool = False     # slack angle mean^2

    # P/Q residual weights [STSI 210526]
    w_P: float = 0.5    # weight on P residual in physics loss (guides v_ang)
    w_Q: float = 0.5    # weight on Q residual in physics loss (guides v_mag)

    # Rich-mode options (full PV/slack handling, partial Q)
    use_q_partial_mode: bool = False        # exclude PV buses from Q residual
    
    # PTDF supervision
    use_ptdf_loss: bool = False    # DC PTDF head on/off
    weight_ptdf: float = 0.0      # weight applied to the PTDF loss term
    ptdf_loss_mode: str = "matrix" # "matrix", "flows", "mixed", or "learned_edge" (STSI 26.04.07)
    ptdf_alpha: float = 0.5       # mix ratio for mode="mixed" (matrix share)
    ptdf_branch_mode: str = "lines"  # branches in PTDF: "lines" or "all"
    ptdf_changepoint: int = None     # [STSI 290526] epoch offset (from warmup_epochs_ptdf) to switch matrix→flows in "stepwise" mode

    # ── DC/AC per-edge flow loss ────────────────────────── [STSI 020626]
    use_flow_loss: bool = False          # master on/off for per-edge flow supervision
    flow_loss_mode: str = "dc"           # "dc", "ac", or "both"
    flow_angle_mode: str = "local"       # "local" (Δθ), "global" (θ_i-θ_j), or "both"
    flow_target: str = "pq"             # "p" (P only), "q" (Q only), or "pq" (both, AC only)

    w_flow: float = 1.0                  # overall weight for flow loss term
    w_flow_local: float = 0.5            # fraction allocated to local Δθ path
    w_flow_global: float = 0.5           # fraction allocated to global θ path
    w_flow_dc: float = 0.5              # fraction allocated to DC formulation
    w_flow_ac: float = 0.5              # fraction allocated to AC formulation

    # Adaptive flow loss weighting [STSI 020626]:Task 045
    fraction_flow: float = 0.1           # target: flow = 10% of MSE (adaptive mode only)
    max_w_flow: float = 100.0            # cap for adaptive flow weight

    def label(self) -> str:
        # [STSI 200526] Distinguishes adaptive vs fixed mode in run labels
        if self.loss_weight_mode == "adaptive":
            parts = [f"adap-fp{self.fraction_physics}"]
            if self.use_ptdf_loss:
                parts.append(f"ft{self.fraction_ptdf}")
        else:
            parts = [f"w-{self.w_phys}"]
            if self.use_ptdf_loss:
                ptdf_part = f"PTDF-{self.ptdf_loss_mode}-pt{self.weight_ptdf}-{self.ptdf_branch_mode}"
                if self.ptdf_loss_mode == "mixed":
                    ptdf_part += f"-a{self.ptdf_alpha}"
                if self.ptdf_loss_mode == "stepwise" and self.ptdf_changepoint is not None:
                    ptdf_part += f"-cp{self.ptdf_changepoint}"
                parts.append(ptdf_part)
        if self.w_P != 0.5 or self.w_Q != 0.5:  # [STSI 210526]: non-default weights
            parts.append(f"wP{self.w_P}wQ{self.w_Q}")
        if self.use_flow_loss:  # [STSI 020626]
            flow_part = f"flow-{self.flow_loss_mode}-{self.flow_angle_mode}-{self.flow_target}-w{self.w_flow}"
            parts.append(flow_part)
        return "_".join(parts)


def physics_informed_loss_batch(
    pred, target, batch, networks, Y_cache,
    physics_cfg: PhysicsConfig,
    #ptdf_loss: Optional[torch.Tensor] = None, # STSI 260403:Removed as this is no longer computed in this function; it is now computed externally and passed in as an argument to the training loop, which combines it with the physics loss according to PhysicsConfig settings.
):
    """
    Modular physics-informed loss using PhysicsConfig flags.

    total = mse + w_phys * physics_terms

    STSI 26.03.25: MSE is no longer computed here — the training loop passes
    in masked MSE via _masked_mse_loss. This function returns physics loss only.
    The mse_loss return value is kept for API compatibility but is now the
    full (unmasked) MSE for logging purposes only — it is NOT used in total_loss.
    """
    # STSI 26.03.25: full MSE kept for logging/history only — not used in total_loss
    mse_loss = F.mse_loss(pred, target)

    if physics_cfg.w_phys == 0.0 and physics_cfg.loss_weight_mode != "adaptive":  # [STSI 200526] allow adaptive mode to compute physics even with w_phys=0
        zero = torch.tensor(0.0, device=pred.device)
        return mse_loss, mse_loss, zero, zero, 0.0, 0.0  # [STSI 210526]: extended to 6-tuple

    graph_ids  = batch.batch
    num_graphs = int(graph_ids.max().item()) + 1

    total_phys  = 0.0
    total_nodes = 0
    total_angle_ref = 0.0
    total_p_res = 0.0  # [STSI 210526]: per-component P residual accumulator
    total_q_res = 0.0  # [STSI 210526]: per-component Q residual accumulator

    for g in range(num_graphs):
        node_mask = (graph_ids == g) # mask for nodes in graph g
        if not node_mask.any():
            continue

        if batch.network_idx.dim() == 0:
            net_idx = int(batch.network_idx.item())
        else:
            net_idx = int(batch.network_idx[g].item())

        network  = networks[net_idx]
        Y_matrix = Y_cache[net_idx]

        x_g    = batch.x[node_mask]
        pred_g = pred[node_mask]

        # STSI 26.03.25: Fix: — assemble mixed known+predicted state vector
        # per bus type before passing to residual functions, restoring the
        # physically correct formulation from the early implementation.
        # Known values override predictions where the power flow solution
        # provides them as inputs:
        #   PQ  buses: Vmag, Vang predicted;  P, Q    known from x
        #   PV  buses: Vmag, P    known;       Vang, Q predicted
        #   Slack bus: Vmag, Vang known (=ref); P, Q  predicted
        slack_mask_g = batch.slack_mask[node_mask].bool()
        pv_mask_g    = batch.pv_mask[node_mask].bool()
        pq_mask_g    = batch.pq_mask[node_mask].bool()

        # Start from predictions for all variables
        v_mag_g = pred_g[:, 0].clone()
        v_ang_g = pred_g[:, 1].clone()
        p_g     = pred_g[:, 2].clone()
        q_g     = pred_g[:, 3].clone()

        # Override with known values where applicable
        # x layout: [is_slack, is_PV, is_PQ, P, Q, Vmag, Vang]
        v_mag_g[pv_mask_g]    = x_g[pv_mask_g,    5]   # PV:    Vmag known
        p_g[pv_mask_g]        = x_g[pv_mask_g,    3]   # PV:    P known
        p_g[pq_mask_g]        = x_g[pq_mask_g,    3]   # PQ:    P known
        q_g[pq_mask_g]        = x_g[pq_mask_g,    4]   # PQ:    Q known
        v_mag_g[slack_mask_g] = x_g[slack_mask_g, 5]   # Slack: Vmag known
        v_ang_g[slack_mask_g] = x_g[slack_mask_g, 6]   # Slack: Vang=0 known

        # Pack mixed state back into pred-shaped tensor for residual functions
        # Shape: [n_buses_g, 4] = [Vmag, Vang, P, Q]
        pred_g_mixed = torch.stack([v_mag_g, v_ang_g, p_g, q_g], dim=1)

        # [STSI 210526]: simple mode removed; rich mode is now unconditional
        bus_masks = (
            slack_mask_g,
            pv_mask_g,
            pq_mask_g,
        )
        residual, p_res_g, q_res_g = compute_power_flow_residual_from_pred(
            pred_g_mixed, x_g, Y_matrix, network,
            bus_masks=bus_masks,
            use_q_partial_mode=physics_cfg.use_q_partial_mode,
            w_P=physics_cfg.w_P,
            w_Q=physics_cfg.w_Q,
        )

        n_g = node_mask.sum().item()
        total_p_res += p_res_g.item() * n_g  # [STSI 210526]: weighted accumulation
        total_q_res += q_res_g.item() * n_g  # [STSI 210526]: weighted accumulation
        
            # angle-ref penalty (optional, both modes)
        # ── RESTORED FROM V2.1.2 ──────────────────────────────────────────
        # In V2.1.2 the angle-ref penalty was computed inside the residual
        # function, once per graph, so squaring always happened before any
        # cross-graph aggregation. When PhysicsConfig was introduced (Block 7)
        # this was lifted out to batch level — introducing a cancellation bug.
        # Fix: compute it here inside the per-graph loop, matching V2.1.2.
        angle_ref_g = torch.tensor(0.0, device=pred.device)
        if physics_cfg.use_angle_ref_penalty and slack_mask_g.any():
            angle_ref_g = pred[node_mask][slack_mask_g, 1].mean() ** 2        
        total_angle_ref += angle_ref_g * n_g          #  accumulate separatelyfor logging

        total_phys  += (residual+angle_ref_g) * n_g # include angle_ref in the physics loss for proper weighting and to avoid cross-graph cancellation
        total_nodes += n_g
    physics_loss = total_phys / total_nodes if total_nodes > 0 else torch.tensor( #[STSI110526] Before fix: works with RELU activations but can lead to scale issues with GELU. Fix: log1p of average residual for better scaling when using GELU, with zero-handling.   
    #physics_loss = torch.log1p(total_phys / total_nodes) if total_nodes > 0 else torch.tensor( #[STSI110526] Fix: log1p of average residual for better scaling when using GELU, with zero-handling
        0.0, device=pred.device
    )
    angle_ref_loss = total_angle_ref / total_nodes if total_nodes > 0 else torch.tensor(
        0.0, device=pred.device
        )


    # PTDF term (optional) # STSI 260403: Removed from here as it is now added separately during the training loop
    #ptdf_term = torch.tensor(0.0, device=pred.device)
    #if physics_cfg.use_ptdf_loss and ptdf_loss is not None:
    #    ptdf_term = ptdf_loss

    combined_physics = physics_loss #+ ptdf_term

    # STSI 26.03.25: total_loss no longer includes MSE — the training loop
    # adds masked MSE externally. Return only the physics component so the
    # caller can combine: loss = masked_mse + w_phys * physics
    p_res_mean = total_p_res / total_nodes if total_nodes > 0 else 0.0  # [STSI 210526]
    q_res_mean = total_q_res / total_nodes if total_nodes > 0 else 0.0  # [STSI 210526]
    return combined_physics, mse_loss, physics_loss, angle_ref_loss, p_res_mean, q_res_mean  # [STSI 210526]: 6-tuple

def get_effective_w_phys(cfg, epoch, warmup_epochs=10):
    if epoch < warmup_epochs:
        return cfg.w_phys * (epoch / warmup_epochs)
    return cfg.w_phys


In [ ]:
# =============================================================================
# SECTION 7: LINE FLOW CALCULATION
# =============================================================================

def calculate_line_flows(network, vmag, vang, t_idx=None):
    """
    Calculate AC line flows from predicted voltages.

    Args:
        network: PyPSA network object
        vmag:    Voltage magnitudes [num_buses], numpy array, p.u.
        vang:    Voltage angles [num_buses], numpy array, radians
        t_idx:   Snapshot index (used for nominal values only)
    Returns:
        dict with keys p0, p1, q0, q1 - each a numpy array of length num_lines (p.u.)
    """
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)

    all_buses = list(network.buses.index)
    # BUGFIX: use bus_to_idx dict instead of string parsing (topology-safe)
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(all_buses)}

    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus = network.lines.loc[line, "bus1"]

        # BUGFIX: use dict lookup instead of int(bus.split('-')[1]) - 1
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]

        r = network.lines.loc[line, "r"]
        x = network.lines.loc[line, "x"]
        b = network.lines.loc[line, "b"] if "b" in network.lines.columns else 0.0

        # Line impedance and admittance
        z = complex(r, x)
        y_series = 1.0 / z if abs(z) > 1e-12 else 0.0
        y_shunt = complex(0, b / 2.0)

        # Complex voltages at from/to buses
        V_from = vmag[from_idx] * np.exp(1j * vang[from_idx])
        V_to = vmag[to_idx] * np.exp(1j * vang[to_idx])

        # Line current from sending end
        I_from = (V_from - V_to) * y_series + V_from * y_shunt
        I_to = (V_to - V_from) * y_series + V_to * y_shunt

        S_from = V_from * np.conj(I_from)
        S_to = V_to * np.conj(I_to)

        s_nom = network.lines.loc[line, "s_nom"]
        # Convert from p.u. to MW/MVAr using system base
        #base_mva = getattr(network, "sn_mva", 100.0)

        p0[i] = S_from.real #* base_mva # removed the base_mva scaling to keep flows in p.u. for better numerical stability and consistency with the rest of the model's inputs/outputs
        q0[i] = S_from.imag #* base_mva
        p1[i] = -S_to.real #* base_mva  # convention: positive = into bus
        q1[i] = -S_to.imag #* base_mva

    return {"p0": p0, "p1": p1, "q0": q0, "q1": q1}



def calculate_line_flows_from_delta_theta(  # [STSI 260526]: direct Δθ line flows
    delta_theta,     # [E_fwd] numpy — Δθ per forward edge (θ_from - θ_to)
    vmag,            # [N] numpy — voltage magnitudes
    edge_index_fwd,  # [2, E_fwd] numpy — forward edge source/target
    edge_attr_fwd,   # [E_fwd, 6] numpy — [r, x, b_half, tap, g_ser, b_ser]
) -> dict:
    """
    Compute line flows directly from Δθ without θ reconstruction.
    Uses π-model: P_from = V_from²×(g_s+g_sh) - V_from×V_to×(g_s×cos(Δθ) + b_s×sin(Δθ))
    Returns dict with keys: 'p0', 'q0', 'p1', 'q1' (from/to in per-unit).
    """
    from_bus = edge_index_fwd[0]  # [E_fwd]
    to_bus = edge_index_fwd[1]    # [E_fwd]
    v_from = vmag[from_bus]
    v_to = vmag[to_bus]

    r = edge_attr_fwd[:, 0]
    x_imp = edge_attr_fwd[:, 1]
    b_half = edge_attr_fwd[:, 2]
    tap = edge_attr_fwd[:, 3]

    # Series admittance
    z = r + 1j * x_imp
    y_s = 1.0 / z
    g_s = y_s.real
    b_s = y_s.imag

    # Handle tap ratio for transformers
    tap_nz = np.where(np.abs(tap) > 1e-12, tap, 1.0)
    # Tap-adjusted: effective from-end voltage = V_from / tap
    v_from_eff = v_from / tap_nz

    # Shunt susceptance per end
    b_sh = b_half

    cos_dt = np.cos(delta_theta)
    sin_dt = np.sin(delta_theta)

    # From-end power (π-model with tap on from side)
    # usually b_s >>g_s and delta angle is usually small, so sin_dt ~ delta_theta and cos_dt ~ 1
    p0 = v_from_eff**2 * g_s - v_from_eff * v_to * (g_s * cos_dt + b_s * sin_dt)
    q0 = -v_from_eff**2 * (b_s + b_sh) - v_from_eff * v_to * (g_s * sin_dt - b_s * cos_dt)

    # To-end power (Δθ reversed, negated to match convention: positive = into bus)  # [STSI 260526]:fix sign convention to match calculate_line_flows
    p1 = -(v_to**2 * g_s - v_to * v_from_eff * (g_s * cos_dt - b_s * sin_dt))
    q1 = -(-v_to**2 * (b_s + b_sh) + v_to * v_from_eff * (g_s * sin_dt + b_s * cos_dt))

    return {'p0': p0, 'q0': q0, 'p1': p1, 'q1': q1}

def build_line_results_from_flows(network, flows, snapshot_label):
    """
    Helper to build a line_results-style DataFrame for one snapshot
    from a dict produced by calculate_line_flows.
    """
    idx = pd.Index([snapshot_label], name="snapshot")
    cols = pd.MultiIndex.from_product(
        [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
    )
    df = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for i, line in enumerate(network.lines.index):
        df.loc[snapshot_label, (line, "P0 pu")] = flows["p0"][i]
        df.loc[snapshot_label, (line, "P1 pu")] = flows["p1"][i]
        df.loc[snapshot_label, (line, "Q0 pu")] = flows["q0"][i]
        df.loc[snapshot_label, (line, "Q1 pu")] = flows["q1"][i]
    return df



#=============================================================================
# SECTION 7B: PTDF loss helpers
#=============================================================================

def compute_ptdf_loss_matrix(h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model):
    """Helper: compute per-graph PTDF loss from embeddings.
        Uses forward_edge_mask to select one edge per line (avoids double-counting
        the reverse duplicate edges that exist in the undirected graph)."""

    # STSI 26.02.11: PTDF loss per graph in batch (manage changing topologies)
    # STSI 26.03.10: BUGFIX: edge_batch is indexed over all edges; forward_edge_mask selects
    # only the first direction of each line to match y_ptdf shape [n_lines, n_buses]
    forward_mask_full = batch.forward_edge_mask  # [total_edges] bool
    total_ptdf_loss   = 0.0
    graphs_with_edges = 0
    for g in range(num_graphs):
        node_mask = (node_batch == g)
        edge_mask = (edge_batch == g)
        # Apply forward-only filter on top of graph filter
        fwd_edge_mask = edge_mask & forward_mask_full
        if not fwd_edge_mask.any():
            continue
        graphs_with_edges += 1
        h_nodes_g = h_nodes[node_mask]
        h_edges_g = h_edges[fwd_edge_mask]          # [n_lines_g, hidden_dim]
        y_ptdf_g  = batch.y_ptdf_list[g]            # [n_lines_g, n_buses_g]
        H_W_g     = h_edges_g @ model.ptdf_W        # [n_lines_g, hidden_dim]
        ptdf_pred_g = H_W_g @ h_nodes_g.T           # [n_lines_g, n_buses_g]
        total_ptdf_loss += F.mse_loss(ptdf_pred_g, y_ptdf_g)
    if graphs_with_edges > 0:
        return total_ptdf_loss / graphs_with_edges
    return torch.tensor(0.0, device=h_nodes.device)

def compute_ptdf_loss_flows(h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model):
    """
    PTDF loss computed in line-flow space:

    For each graph g in the batch:
        - Predict PTDF_g from edge/node embeddings.
        - Build bus injection vector ΔP_g from input features.
        - Compute line flows f_pred_g = PTDF_pred_g @ ΔP_g.
        - Compare to true line flows f_true_g from PyPSA.

    Returns:
        Scalar tensor: mean MSE over graphs with valid lines.
    """
    # 26.03.12 STSI: The original _compute_ptdf_loss computed the loss directly in PTDF space
    # (comparing predicted vs. calculated PTDF matrices). This version computes the loss in
    # line flow space, which is more directly related to physical quantities and may provide
    # a stronger training signal. It requires that the dataset includes true line flow values
    # for each graph, which can be calculated from the original PyPSA networks and included
    # during dataset construction.
    if not hasattr(batch, "forward_edge_mask"):
        raise RuntimeError("Batch missing forward_edge_mask")
    if not hasattr(batch, "y_line_p_list"):
        raise RuntimeError("Batch missing y_line_p_list")
    if not hasattr(batch, "ptdf_line_index_list"):
        raise RuntimeError("Batch missing ptdf_line_index_list")

    forward_mask_full = batch.forward_edge_mask  # [total_edges] bool
    total_loss        = 0.0
    graphs_with_edges = 0

    for g in range(num_graphs):
        # Node and edge masks for this graph
        node_mask     = (node_batch == g)
        edge_mask     = (edge_batch == g)
        fwd_edge_mask = edge_mask & forward_mask_full

        if not fwd_edge_mask.any():
            continue

        # Embeddings for this graph
        h_nodes_g = h_nodes[node_mask]        # [n_buses_g, hidden_dim]
        h_edges_g = h_edges[fwd_edge_mask]    # [n_lines_g, hidden_dim]

        # Predicted PTDF matrix: [n_lines_g, n_buses_g]
        H_W_g       = h_edges_g @ model.ptdf_W
        ptdf_pred_g = H_W_g @ h_nodes_g.T

        # Build bus injection vector ΔP_g from input features x
        # x layout: [is_slack, is_PV, is_PQ, P, Q, Vmag, Vang]
        x_g       = batch.x[node_mask]        # [n_buses_g, feat_dim]
        delta_p_g = x_g[:, 3]                 # use active power injections P as ΔP

        # 26.03.12 STSI: Added line flow loss calculation:
        # True line flows from PyPSA for this graph (must be provided in dataset)
        # Expect shape [n_lines_g]
        # STSI260402: modified to use ptdf_line_index_list to align line flows with PTDF rows, since some graphs may have fewer lines than the max and y_line_p_list is indexed by the max line count.
        # Predicted line flows from PTDF * ΔP
        # [n_lines_g] = [n_lines_g, n_buses_g] @ [n_buses_g]
        f_pred_g = ptdf_pred_g @ delta_p_g

        # True line flows aligned via ptdf_line_index
        line_idx_g = batch.ptdf_line_index_list[g]
        if line_idx_g.numel() == 0:
            continue

        f_true_all_g = batch.y_line_p_list[g]

        max_idx = int(line_idx_g.max().item())
        if max_idx >= len(f_true_all_g):
            raise RuntimeError(
                f"[PTDF flow loss] graph {g}: max line index {max_idx} "
                f">= len(y_line_p_list[{g}]) {len(f_true_all_g)}"
            )

        f_true_g = f_true_all_g[line_idx_g]

        if f_pred_g.shape != f_true_g.shape:
            raise RuntimeError(
                f"[PTDF flow loss] graph {g}: pred shape {tuple(f_pred_g.shape)} "
                f"!= true shape {tuple(f_true_g.shape)}"
            )

        # Flow-based PTDF loss for this graph
        loss_g = F.mse_loss(f_pred_g, f_true_g)
        total_loss        += loss_g
        graphs_with_edges += 1

    if graphs_with_edges > 0:
        return total_loss / graphs_with_edges
    return torch.tensor(0.0, device=h_nodes.device)


# ── STSI 26.04.07: "learned_edge" PTDF mode ─────────────────────────────────
# Learnable PTDF matrices used as (a) edge features for GAT attention and
# (b) unsupervised flow predictors.  One nn.Parameter per network, padded
# to max_buses width so all networks share the same feature dimension.

def initialize_learned_ptdf(networks, max_buses, device=None, ptdf_branch_mode="lines"):  # [STSI 260526]:added ptdf_branch_mode 
    """Create one learnable PTDF parameter per network.

    Each parameter has shape [n_branches, max_buses]:
      - columns 0..n_buses-1 initialised to 1.0 (active bus columns)
      - columns n_buses..max_buses-1 initialised to 0.0 (padding)

    When ptdf_branch_mode="all", n_branches = n_lines + n_trafos.

    Returns:
        list[nn.Parameter] indexed by position in *networks*.
    """
    ptdf_params = []
    for net in networks:
        n_branches = len(net.lines)
        if ptdf_branch_mode == "all":  # [STSI 260526]:include trafos in learned PTDF 
            n_branches += len(net.transformers)
        n_buses = len(net.buses)
        param = torch.ones(n_branches, max_buses, device=device)
        if n_buses < max_buses:
            param[:, n_buses:] = 0.0
        ptdf_params.append(nn.Parameter(param))
    return ptdf_params


def build_ptdf_augmented_edge_attr(batch, ptdf_params, max_buses, ptdf_offset=0):
    """Concatenate learned PTDF rows to batch.edge_attr for every edge.

    For each edge e in the batch:
      - if ptdf_edge_row_idx[e] >= 0 (supervised forward edge):
            append ptdf_params[net][row_idx]   (shape [max_buses])
      - otherwise: append a zero vector          (shape [max_buses])

    Args:
        batch:        PyG Batch object with .ptdf_edge_row_idx, .network_idx
        ptdf_params:  list[nn.Parameter], one per network (global indexing)
        max_buses:    int, width of every PTDF parameter
        ptdf_offset:  int, added to batch.network_idx to get the global index
                      into ptdf_params  (0 for train, num_train for val, etc.)

    Returns:
        Tensor [total_edges, original_feat + max_buses]
    """
    edge_attr   = batch.edge_attr                     # [E, F]
    total_edges = edge_attr.size(0)
    device      = edge_attr.device

    ptdf_rows = torch.zeros(total_edges, max_buses, device=device)

    node_batch     = batch.batch                       # [N]
    edge_source    = batch.edge_index[0]               # [E]
    edge_batch_ids = node_batch[edge_source]           # [E] graph id per edge
    row_idx        = batch.ptdf_edge_row_idx           # [E] line index or -1

    num_graphs = int(node_batch.max().item()) + 1

    for g in range(num_graphs):
        edge_mask_g = (edge_batch_ids == g)
        net_idx     = int(batch.network_idx[g].item()) + ptdf_offset
        row_idx_g   = row_idx[edge_mask_g]             # [E_g]

        valid = row_idx_g >= 0
        if valid.any():
            valid_line_indices = row_idx_g[valid]       # line row indices
            fetched = ptdf_params[net_idx][valid_line_indices]  # [n_valid, max_buses]

            edge_positions  = torch.where(edge_mask_g)[0]
            valid_positions = edge_positions[valid]
            ptdf_rows[valid_positions] = fetched

    return torch.cat([edge_attr, ptdf_rows], dim=-1)


def compute_ptdf_loss_learned(
    node_pred, batch, node_batch, num_graphs,
    ptdf_params, ptdf_offset=0,
):
    """Unsupervised flow loss: learned_PTDF x predicted_P vs true line flows.

    For each graph g:
      1. Retrieve learned PTDF [n_lines, max_buses], trim to n_buses_g.
      2. Multiply by model-predicted P injections -> predicted line flows.
      3. MSE against ground-truth line flows from PyPSA.

    This provides gradient to both the learned PTDF parameters and the
    model's P-prediction head.

    Args:
        node_pred:    [total_nodes, 4] model output (col 2 = P)
        batch:        PyG Batch with y_line_p_list, ptdf_line_index_list
        node_batch:   [total_nodes] graph assignment
        num_graphs:   int
        ptdf_params:  list[nn.Parameter]
        ptdf_offset:  int, local->global network index offset

    Returns:
        Scalar loss tensor (mean MSE over graphs with lines).
    """
    total_loss        = 0.0
    graphs_with_edges = 0

    for g in range(num_graphs):
        node_mask = (node_batch == g)
        if not node_mask.any():
            continue

        net_idx      = int(batch.network_idx[g].item()) + ptdf_offset
        learned_ptdf = ptdf_params[net_idx]            # [n_lines, max_buses]

        n_buses_g    = int(node_mask.sum().item())
        ptdf_trimmed = learned_ptdf[:, :n_buses_g]     # [n_lines, n_buses_g]

        # Predicted P injections from model output
        pred_p = node_pred[node_mask, 2]               # [n_buses_g]

        # Predicted flows
        f_pred = ptdf_trimmed @ pred_p                 # [n_lines]

        # True flows
        line_idx_g = batch.ptdf_line_index_list[g]
        if line_idx_g.numel() == 0:
            continue

        f_true_all = batch.y_line_p_list[g]
        f_true     = f_true_all[line_idx_g]

        if f_pred.shape != f_true.shape:
            continue

        loss_g = F.mse_loss(f_pred, f_true)
        total_loss        += loss_g
        graphs_with_edges += 1

    if graphs_with_edges > 0:
        return total_loss / graphs_with_edges
    return torch.tensor(0.0, device=node_pred.device)
# ── End STSI 26.04.07 learned_edge functions ─────────────────────────────────



def compute_physics_residual_edge_local(  # [STSI 260526]: O(E) edge-local physics
    delta_theta_pred,   # [E_fwd] — predicted Δθ for forward edges (torch, with grad)
    vmag,               # [N] — predicted voltage magnitudes (torch, with grad)
    node_pred,          # [N, 3 or 4] — full node predictions (for P/Q at slack/PV)
    edge_index,         # [2, 2E] — full bidirectional edge index
    edge_attr,          # [2E, 6] — edge features [r, x, b_half, tap, g_ser, b_ser]
    g_diag,             # [N] — precomputed diagonal conductance
    b_diag,             # [N] — precomputed diagonal susceptance
    x,                  # [N, F] — node features (cols 0-2: bus type masks, 3-4: known P/Q)
    bus_masks=None,     # optional (slack_mask, pv_mask, pq_mask)
    use_q_partial_mode: bool = False,
    w_P: float = 0.5,
    w_Q: float = 0.5,
):
    """
    Edge-local AC power flow residual using scatter operations. O(E) instead of O(N²).
    Operates directly on Δθ — no Y-bus matrix or θ reconstruction needed.

    Power flow equations in edge form:
      P_i = V_i² × G_ii - Σ_{j∈adj(i)} V_i×V_j×(g_ij×cos(Δθ_ij) + b_ij×sin(Δθ_ij))
      Q_i = -V_i² × B_ii - Σ_{j∈adj(i)} V_i×V_j×(g_ij×sin(Δθ_ij) - b_ij×cos(Δθ_ij))
    """
    from torch_geometric.utils import scatter
    N = vmag.shape[0]
    device = vmag.device

    # Build full Δθ vector [2E]: forward = +Δθ, reverse = -Δθ
    # Even indices = forward, odd = reverse (guaranteed by _create_graph_data construction)
    n_edges_total = edge_index.shape[1]
    delta_theta_full = torch.zeros(n_edges_total, device=device)
    all_fwd = torch.zeros(n_edges_total, dtype=torch.bool, device=device)
    all_fwd[::2] = True
    delta_theta_full[all_fwd] = delta_theta_pred
    delta_theta_full[~all_fwd] = -delta_theta_pred  # reverse direction

    # Edge quantities
    src, dst = edge_index[0], edge_index[1]  # [2E] each
    v_i = vmag[src]  # [2E]
    v_j = vmag[dst]  # [2E]
    g_s = edge_attr[:, 4]  # [2E] series conductance
    b_s = edge_attr[:, 5]  # [2E] series susceptance

    cos_dt = torch.cos(delta_theta_full)
    sin_dt = torch.sin(delta_theta_full)

    # Off-diagonal contribution (Y_ij_off = -y_series → negative sign in sum)
    p_edge = -v_i * v_j * (g_s * cos_dt + b_s * sin_dt)  # [2E]
    q_edge = -v_i * v_j * (g_s * sin_dt - b_s * cos_dt)  # [2E]

    # Scatter to source node
    p_off = scatter(p_edge, src, dim=0, dim_size=N, reduce='sum')  # p_off is the sum of contributions from all edges originating at node i
    q_off = scatter(q_edge, src, dim=0, dim_size=N, reduce='sum')  # q_off is the sum of contributions from all edges originating at node i

    # Diagonal (self-admittance)
    p_calc = vmag**2 * g_diag + p_off  # [N]
    q_calc = -(vmag**2) * b_diag + q_off  # [N]

    # ── Assemble specified injections (same logic as compute_power_flow_residual_from_pred)
    if bus_masks is None:
        slack_mask = (x[:, 0] == 1.0)
        pv_mask = (x[:, 1] == 1.0)
        pq_mask = (x[:, 2] == 1.0)
    else:
        slack_mask, pv_mask, pq_mask = bus_masks

    p_inj = torch.zeros(N, device=device)
    q_inj = torch.zeros(N, device=device)
    p_inj[pq_mask] = x[pq_mask, 3]        # known P at PQ buses
    q_inj[pq_mask] = x[pq_mask, 4]        # known Q at PQ buses
    p_inj[pv_mask] = x[pv_mask, 3]        # known P at PV buses
    # node_pred layout for edge_delta: [Vmag(0), P(1), Q(2)]
    q_inj[pv_mask] = node_pred[pv_mask, 2]    # predicted Q at PV
    p_inj[slack_mask] = node_pred[slack_mask, 1]  # predicted P at slack
    q_inj[slack_mask] = node_pred[slack_mask, 2]  # predicted Q at slack

    # ── Residuals
    b_scale=b_diag.abs().clamp(min=1e-6)  # avoid division by zero, keep scaling consistent
    #[STSI 260526]: Normalize by division of b_diag to avoid domination by high-injection buses.
    p_residual = ((p_calc - p_inj) / b_scale) ** 2
    if use_q_partial_mode:
        q_mask = pq_mask | slack_mask
        q_residual = torch.zeros(N, device=device)
        q_residual[q_mask] = ((q_calc[q_mask] - q_inj[q_mask]) / b_scale[q_mask]) ** 2
    else:
        #[STSI 260526]: Normalize by division of b_diag to avoid domination by high-injection buses.
        q_residual = ((q_calc - q_inj) / b_scale) ** 2

    p_res_mean = p_residual.mean().detach()
    q_res_mean = q_residual.mean().detach()
    physics_loss = torch.mean(w_P * p_residual + w_Q * q_residual)
    return physics_loss, p_res_mean, q_res_mean

def compute_ptdf_loss(
    h_nodes,
    h_edges,
    batch,
    node_batch,
    edge_batch,
    num_graphs,
    model,
    ptdf_loss_mode: str = "matrix",
    ptdf_alpha: float = 0.5,
    # STSI 26.04.07: additional kwargs for learned_edge mode
    node_pred=None,
    ptdf_params=None,
    ptdf_offset=0,
):
    if ptdf_loss_mode == "matrix":
        return compute_ptdf_loss_matrix(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
    elif ptdf_loss_mode == "flows":
        return compute_ptdf_loss_flows(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
    elif ptdf_loss_mode == "mixed":
        loss_mat = compute_ptdf_loss_matrix(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
        loss_flow = compute_ptdf_loss_flows(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
        return ptdf_alpha * loss_mat + (1.0 - ptdf_alpha) * loss_flow
        
    elif ptdf_loss_mode == "learned_edge":  # STSI 26.04.07
        return compute_ptdf_loss_learned(
            node_pred, batch, node_batch, num_graphs,
            ptdf_params, ptdf_offset=ptdf_offset,
        )
    else:
        raise ValueError(f"Unknown ptdf_loss_mode: {ptdf_loss_mode!r}")


# ═══════════════════════════════════════════════════════════════════════════════
# DC/AC Per-Edge Flow Loss Functions — [STSI 020626]
# ═══════════════════════════════════════════════════════════════════════════════

def compute_flow_loss_dc_local(
    delta_theta_pred,   # [E_fwd] — predicted Δθ for forward edges
    edge_attr_fwd,      # [E_fwd, 6] — forward-edge features [r, x, b_half, tap, g_ser, b_ser]
    y_line_p,           # [E_supervised] — true line flows from AC power flow
    line_mask=None,     # [E_fwd] bool — which forward edges have flow supervision (optional)
):
    """
    DC per-edge flow loss: f_pred = Δθ_pred / x, compared to y_line_p.
    
    Uses DC approximation (r=0, |V|=1): P_ij = (θ_i - θ_j) / x_ij
    Consistent with PTDF matrix computation (which uses 1/x).
    
    Returns: (loss_scalar, flow_mae_detached)
    """
    x_react = edge_attr_fwd[:, 1]  # [E_fwd] — reactance from col 1

    # DC flow prediction: Δθ / x
    f_pred_dc = delta_theta_pred / x_react.clamp(min=1e-8)

    # Align dimensions: y_line_p may cover fewer branches than E_fwd
    if line_mask is not None:
        f_pred_dc = f_pred_dc[line_mask]
    else:
        # Default: supervise first len(y_line_p) edges (lines come first in edge_index)
        f_pred_dc = f_pred_dc[:len(y_line_p)]

    loss = F.mse_loss(f_pred_dc, y_line_p)
    mae = (f_pred_dc - y_line_p).abs().mean().detach()
    return loss, mae


def compute_flow_loss_dc_global(
    theta_pred,         # [N] — predicted node voltage angles (from node_pred[:, 1])
    edge_index_fwd,     # [2, E_fwd] — forward edge indices (source, target)
    edge_attr_fwd,      # [E_fwd, 6] — forward-edge features [r, x, b_half, tap, g_ser, b_ser]
    y_line_p,           # [E_supervised] — true line flows
    line_mask=None,     # [E_fwd] bool — which forward edges have flow supervision
):
    """
    DC per-edge flow loss using node θ: f_pred = (θ_i - θ_j) / x.
    Requires angle_mode="both" or "node" (node_pred has θ column).
    
    Returns: (loss_scalar, flow_mae_detached)
    """
    src, dst = edge_index_fwd[0], edge_index_fwd[1]
    delta_from_node = theta_pred[src] - theta_pred[dst]  # [E_fwd]

    x_react = edge_attr_fwd[:, 1]  # [E_fwd] reactance
    f_pred_dc = delta_from_node / x_react.clamp(min=1e-8)

    if line_mask is not None:
        f_pred_dc = f_pred_dc[line_mask]
    else:
        f_pred_dc = f_pred_dc[:len(y_line_p)]

    loss = F.mse_loss(f_pred_dc, y_line_p)
    mae = (f_pred_dc - y_line_p).abs().mean().detach()
    return loss, mae


def compute_flow_loss_ac_local(
    delta_theta_pred,   # [E_fwd] — predicted Δθ
    vmag,               # [N] — predicted voltage magnitudes
    edge_index_fwd,     # [2, E_fwd] — forward edge indices (source, target)
    edge_attr_fwd,      # [E_fwd, 6] — forward-edge features [r, x, b_half, tap, g_ser, b_ser]
    y_line_p,           # [E_supervised] — true P line flows
    y_line_q,           # [E_supervised] — true Q line flows
    flow_target="pq",   # "p", "q", or "pq"
    line_mask=None,     # [E_fwd] bool
):
    """
    AC per-edge flow loss using Δθ head.
    P_ij = V_i²×g_s - V_i×V_j×(g_s×cos(Δθ) + b_s×sin(Δθ))
    Q_ij = -V_i²×b_s - V_i×V_j×(g_s×sin(Δθ) - b_s×cos(Δθ))
    
    The V_i²×g_s / -V_i²×b_s self-admittance terms account for series losses.
    Q flow provides direct per-edge gradient to the Vmag head.
    
    Returns: (loss_scalar, diag_dict)
        diag_dict: {p_mae, q_mae} (detached)
    """
    src_fwd = edge_index_fwd[0]  # [E_fwd]
    dst_fwd = edge_index_fwd[1]  # [E_fwd]

    v_i = vmag[src_fwd]  # [E_fwd]
    v_j = vmag[dst_fwd]  # [E_fwd]

    g_s = edge_attr_fwd[:, 4]  # series conductance
    b_s = edge_attr_fwd[:, 5]  # series susceptance

    cos_dt = torch.cos(delta_theta_pred)
    sin_dt = torch.sin(delta_theta_pred)

    loss = torch.tensor(0.0, device=vmag.device)
    diag = {}

    # AC P flow: P_ij = V_i²×g_s - V_i×V_j×(g_s×cos(Δθ) + b_s×sin(Δθ))
    if flow_target in ("p", "pq"):
        p_pred = v_i**2 * g_s - v_i * v_j * (g_s * cos_dt + b_s * sin_dt)
        p_pred_sup = p_pred[line_mask] if line_mask is not None else p_pred[:len(y_line_p)]
        loss = loss + F.mse_loss(p_pred_sup, y_line_p)
        diag["p_mae"] = (p_pred_sup - y_line_p).abs().mean().detach().item()

    # AC Q flow: Q_ij = -V_i²×b_s - V_i×V_j×(g_s×sin(Δθ) - b_s×cos(Δθ))
    if flow_target in ("q", "pq"):
        q_pred = -v_i**2 * b_s - v_i * v_j * (g_s * sin_dt - b_s * cos_dt)
        q_pred_sup = q_pred[line_mask] if line_mask is not None else q_pred[:len(y_line_q)]
        loss = loss + F.mse_loss(q_pred_sup, y_line_q)
        diag["q_mae"] = (q_pred_sup - y_line_q).abs().mean().detach().item()

    return loss, diag


def compute_flow_loss_ac_global(
    theta_pred,         # [N] — predicted node voltage angles
    vmag,               # [N] — predicted voltage magnitudes
    edge_index_fwd,     # [2, E_fwd] — forward edge indices (source, target)
    edge_attr_fwd,      # [E_fwd, 6] — forward-edge features [r, x, b_half, tap, g_ser, b_ser]
    y_line_p,           # [E_supervised] — true P line flows
    y_line_q,           # [E_supervised] — true Q line flows
    flow_target="pq",   # "p", "q", or "pq"
    line_mask=None,     # [E_fwd] bool
):
    """
    AC per-edge flow loss using node θ.
    P_ij = V_i²×g_s - V_i×V_j×(g_s×cos(θ_i-θ_j) + b_s×sin(θ_i-θ_j))
    Q_ij = -V_i²×b_s - V_i×V_j×(g_s×sin(θ_i-θ_j) - b_s×cos(θ_i-θ_j))
    Requires angle_mode="both" or "node".
    
    Returns: (loss_scalar, diag_dict)
    """
    src_fwd = edge_index_fwd[0]  # [E_fwd]
    dst_fwd = edge_index_fwd[1]  # [E_fwd]

    v_i = vmag[src_fwd]
    v_j = vmag[dst_fwd]
    delta_from_node = theta_pred[src_fwd] - theta_pred[dst_fwd]  # [E_fwd]

    g_s = edge_attr_fwd[:, 4]
    b_s = edge_attr_fwd[:, 5]

    cos_dt = torch.cos(delta_from_node)
    sin_dt = torch.sin(delta_from_node)

    loss = torch.tensor(0.0, device=vmag.device)
    diag = {}

    if flow_target in ("p", "pq"):
        p_pred = v_i**2 * g_s - v_i * v_j * (g_s * cos_dt + b_s * sin_dt)
        p_pred_sup = p_pred[line_mask] if line_mask is not None else p_pred[:len(y_line_p)]
        loss = loss + F.mse_loss(p_pred_sup, y_line_p)
        diag["p_mae"] = (p_pred_sup - y_line_p).abs().mean().detach().item()

    if flow_target in ("q", "pq"):
        q_pred = -v_i**2 * b_s - v_i * v_j * (g_s * sin_dt - b_s * cos_dt)
        q_pred_sup = q_pred[line_mask] if line_mask is not None else q_pred[:len(y_line_q)]
        loss = loss + F.mse_loss(q_pred_sup, y_line_q)
        diag["q_mae"] = (q_pred_sup - y_line_q).abs().mean().detach().item()

    return loss, diag


def compute_flow_loss(
    cfg,                # PhysicsConfig
    delta_theta_pred,   # [E_fwd] or None — from Δθ head
    theta_pred,         # [N] or None — from node_pred[:, 1]
    vmag,               # [N] — from node_pred[:, 0]
    edge_index,         # [2, 2E] — full bidirectional edge index
    edge_attr,          # [2E, 6] — full bidirectional edge attributes
    y_line_p,           # [E_supervised] — true P flows
    y_line_q=None,      # [E_supervised] or None — true Q flows (needed for AC)
    line_mask=None,     # [E_fwd] bool
    dc_line_mask=None,  # [STSI 020626]:lines-only mask for DC (excludes transformers where P=Δθ/x blows up)
    active_local=True,  # controlled by warmup (Task 045)
    active_global=True, # controlled by warmup (Task 045)
    active_ac=True,     # controlled by warmup (Task 045)
):
    """
    Unified dispatcher for DC/AC × local/global flow losses.
    Returns: (total_flow_loss, diagnostics_dict)
    
    Precomputes edge_index_fwd and edge_attr_fwd once for all sub-functions.
    """
    device = y_line_p.device
    total = torch.tensor(0.0, device=device)
    diag = {}

    use_dc = cfg.flow_loss_mode in ("dc", "both")
    use_ac = cfg.flow_loss_mode in ("ac", "both") and active_ac
    use_local = cfg.flow_angle_mode in ("local", "both") and active_local
    use_global = cfg.flow_angle_mode in ("global", "both") and active_global

    # Precompute forward-only views (used by all sub-functions)
    edge_index_fwd = edge_index[:, ::2]   # [2, E_fwd]
    edge_attr_fwd = edge_attr[::2]        # [E_fwd, 6]

    # [STSI 020626]:DC uses lines-only mask (excludes transformers where P=Δθ/x is ill-conditioned)
    _dc_mask = dc_line_mask if dc_line_mask is not None else line_mask
    _y_dc = y_line_p[_dc_mask] if _dc_mask is not None else y_line_p

    # DC local
    if use_dc and use_local and delta_theta_pred is not None:
        loss_dc_l, mae_dc_l = compute_flow_loss_dc_local(
            delta_theta_pred, edge_attr_fwd, _y_dc, _dc_mask)  # [STSI 020626]:DC uses lines-only mask
        total = total + cfg.w_flow_dc * cfg.w_flow_local * loss_dc_l
        diag["flow_loss_dc_local"] = loss_dc_l.item()
        diag["flow_mae_dc_local"] = mae_dc_l.item()

    # DC global
    if use_dc and use_global and theta_pred is not None:
        loss_dc_g, mae_dc_g = compute_flow_loss_dc_global(
            theta_pred, edge_index_fwd, edge_attr_fwd, _y_dc, _dc_mask)  # [STSI 020626]:DC uses lines-only mask
        total = total + cfg.w_flow_dc * cfg.w_flow_global * loss_dc_g
        diag["flow_loss_dc_global"] = loss_dc_g.item()
        diag["flow_mae_dc_global"] = mae_dc_g.item()

    # AC local (P + Q)
    if use_ac and use_local and delta_theta_pred is not None:
        _y_q = y_line_q if y_line_q is not None else torch.zeros_like(y_line_p)
        loss_ac_l, diag_ac_l = compute_flow_loss_ac_local(
            delta_theta_pred, vmag, edge_index_fwd, edge_attr_fwd,
            y_line_p, _y_q, cfg.flow_target, line_mask)
        total = total + cfg.w_flow_ac * cfg.w_flow_local * loss_ac_l
        diag["flow_loss_ac_local"] = loss_ac_l.item()
        if "p_mae" in diag_ac_l:
            diag["flow_mae_ac_local_p"] = diag_ac_l["p_mae"]
        if "q_mae" in diag_ac_l:
            diag["flow_mae_ac_local_q"] = diag_ac_l["q_mae"]

    # AC global (P + Q)
    if use_ac and use_global and theta_pred is not None:
        _y_q = y_line_q if y_line_q is not None else torch.zeros_like(y_line_p)
        loss_ac_g, diag_ac_g = compute_flow_loss_ac_global(
            theta_pred, vmag, edge_index_fwd, edge_attr_fwd,
            y_line_p, _y_q, cfg.flow_target, line_mask)
        total = total + cfg.w_flow_ac * cfg.w_flow_global * loss_ac_g
        diag["flow_loss_ac_global"] = loss_ac_g.item()
        if "p_mae" in diag_ac_g:
            diag["flow_mae_ac_global_p"] = diag_ac_g["p_mae"]
        if "q_mae" in diag_ac_g:
            diag["flow_mae_ac_global_q"] = diag_ac_g["q_mae"]

    total = cfg.w_flow * total
    return total, diag



## Training Loop

In [ ]:

# =============================================================================
# SECTION 8: TRAINING
# =============================================================================
def train_power_flow_gnn(
    networks,
    num_epochs=200,
    batch_size=1,
    lr=0.001,
    conv_type="gatv2",
    hidden_dim=64,
    num_layers=3,
    head_mode: str = "standard",
    use_residual: bool = False,          # [STSI 120526]:independent conv block flags
    norm_type: str = None,               # None | "layer" | "graph"
    activation: str = "leaky_relu",      # "leaky_relu" | "gelu" | "relu" | "elu" | "silu"
    use_dropout: bool = False,
    drop_rate: float = 0.1,
    conv_mode: str = None,              # deprecated shorthand
    physics_cfg: "PhysicsConfig | None" = None,   # ← replaces weight_physics + q_residual_mode
    q_residual_mode="full", 
    use_pnom_share: bool = False,        # [STSI070526] append p_nom participation weight as 8th node feature — [STSI100526] removed use_pnet_balance param (dead after col7 removal)
    weight_ptdf=0.1,                 # STSI260402: removed and included in PhysicsConfig
    ptdf_loss_mode: str = "flows",  # "matrix", "flows","mixed" or "stepwise" STSI260402: removed and included in PhysicsConfig 
    ptdf_alpha: float = 0.5,         # only used if mode == "mixed" # STSI260402: removed and included in PhysicsConfig
    ptdf_changepoint:int = None,          # epoch at which to switch from matrix to flow loss in "stepwise" mode
    use_edge_features=True,
    max_n_test=None,
    # DISTSLACK: propagate distribute_slack flag to dataset construction
    distribute_slack=False,
    y_matrix_source: str = "auto",   # STSI260320: controls Y source for physics loss
    warmup_epochs: int = 0,            # STSI 28.05.25: keept for backwards compatibility
    warmup_epochs_ptdf: int = 0,           # STSI 28.05.25: exposed for sweep control
    warmup_epochs_phys: int = 0,           # STSI 28.05.25: exposed for sweep control
    angle_mode: str = "node",          # [STSI 250526] "node", "edge_delta", or "both"  # [STSI 020626]:added "both"
    vmag_mode: str = "absolute",       # [STSI 280526] "absolute" or "residual"
    warmup_epochs_flow: int = 0,         # [STSI 020626]:epochs before DC-local flow loss activates (Task 045)
    warmup_epochs_flow_global: int = 0,  # [STSI 020626]:additional delay before global flow activates
    warmup_epochs_flow_ac: int = 0,      # [STSI 020626]:additional delay before AC flow activates
    use_global_pool: bool = False,   # [STSI 290526] global graph context injection
    seed=42,
    return_debug_objects: bool = False,# STSI260402 added for enabling return of extra debug info without affecting main return values
    tqdm_file=None,  # file handle for tqdm output (e.g. real stderr when verbose=False)
):
    """
    Train power flow GNN with proper network-based train/val/test split.
    Extended version that returns per-epoch histories and aggregated test metrics.

    DISTSLACK: pass distribute_slack=True to enable distributed slack handling
               in the physics loss and masking.

    Returns:
        model, history (dict of per-epoch lists), test_metrics (dict)
    """
    if physics_cfg is None:
        physics_cfg = PhysicsConfig(w_phys=0.0)

    # [STSI 260526]: edge_delta BFS requires all branches in forward_edge_mask
    if angle_mode in ("edge_delta", "both") and physics_cfg.ptdf_branch_mode != "all":  # [STSI 020626]:extended to "both" (needs BFS)
        physics_cfg.ptdf_branch_mode = "all"
        print(f"  [auto] ptdf_branch_mode overridden to 'all' for angle_mode='{angle_mode}'")


    # STSI 16.02.26: Extended train_power_flow_gnn to return history & test metrics
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    networks_copy = networks.copy()
    random.shuffle(networks_copy)
    num_train_networks = int(0.7 * len(networks_copy))
    num_val_networks   = int(0.85 * len(networks_copy)) - num_train_networks
    train_networks = networks_copy[:num_train_networks]
    val_networks   = networks_copy[num_train_networks:num_train_networks + num_val_networks]
    test_networks  = networks_copy[num_train_networks + num_val_networks:]

    # Debug: Verify split
    print(f"Network split: Train={len(train_networks)}, Val={len(val_networks)}, Test={len(test_networks)}")
    print(f"Total {len(train_networks)+len(val_networks)+len(test_networks)} of {len(networks)}")

    # Create datasets from split networks
    train_dataset = PowerFlowDataset(
        train_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=use_pnom_share,
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
        angle_mode=angle_mode,  # [STSI 250526]: angle_mode is needed for dataset construction since it controls whether the dataset includes Δθ targets for forward edges (edge_delta mode) or not (node mode), which affects the shape of the model output and physics loss.
    )
    val_dataset = PowerFlowDataset(
        val_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=use_pnom_share,
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
        angle_mode=angle_mode,  # [STSI 250526]: angle_mode is needed for dataset construction since it controls whether the dataset includes Δθ targets for forward edges (edge_delta mode) or not (node mode), which affects the shape of the model output and physics loss.
    )
    test_dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=use_pnom_share,
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
        angle_mode=angle_mode,  # [STSI 250526]: angle_mode is needed for dataset construction since it controls whether the dataset includes Δθ targets for forward edges (edge_delta mode) or not (node mode), which affects the shape of the model output and physics loss.
    )

    # Debug: Verify dataset sizes
    print(f"Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
    sample = train_dataset[0]
    print(f"Feature dim: {sample.x.size(1)}")        # must be 8

        # ── Dataset diagnostic (compare with v2.1.2 output) ──────────────────
    _diag = train_dataset[0]
    _sm = _diag.slack_mask.bool()
    _pvm = _diag.pv_mask.bool()
    _pqm = _diag.pq_mask.bool()
    print(f"\n--- DATASET DIAGNOSTIC (sample 0) ---")
    print(f"x shape: {_diag.x.shape}  y shape: {_diag.y.shape}")
    print(f"edge_index shape: {_diag.edge_index.shape}  ({_diag.edge_index.shape[1]} edges)")
    print(f"bus counts: slack={_sm.sum()}, PV={_pvm.sum()}, PQ={_pqm.sum()}")
    if _sm.any():
        print(f"Slack x[6] (Vang, should be ~0): {_diag.x[_sm, 6]}")
        print(f"Slack x[5] (Vmag):               {_diag.x[_sm, 5]}")
        print(f"Slack x[3:5] (P,Q, should be 0): {_diag.x[_sm, 3:5]}")
        print(f"Slack y (target):                {_diag.y[_sm]}")
    if _pvm.any():
        print(f"PV x[5] (Vmag, should be ~1.0):  {_diag.x[_pvm, 5][:3]}")
        print(f"PV x[6] (Vang, should be 0):     {_diag.x[_pvm, 6][:3]}")
        print(f"PV x[3] (P, known):              {_diag.x[_pvm, 3][:3]}")
        print(f"PV x[4] (Q, should be 0):        {_diag.x[_pvm, 4][:3]}")
        print(f"PV y (target):                   {_diag.y[_pvm][:3]}")
    if _pqm.any():
        print(f"PQ x[5:7] (Vmag,Vang, should be 0): {_diag.x[_pqm, 5:7][:3]}")
        print(f"PQ x[3:5] (P,Q, known):             {_diag.x[_pqm, 3:5][:3]}")
        print(f"PQ y (target):                       {_diag.y[_pqm][:3]}")
    if hasattr(_diag, 'edge_attr'):
        print(f"edge_attr shape: {_diag.edge_attr.shape}")
        print(f"edge_attr sample (first edge): {_diag.edge_attr[0]}")
    print(f"--- END DIAGNOSTIC ---\n")


    # Cache Y-matrices to reduce computation during training
    logger.info("Precomputing admittance matrices...")
    train_Y_cache = precompute_Y_matrices(train_networks, y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    val_Y_cache   = precompute_Y_matrices(val_networks,   y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    test_Y_cache  = precompute_Y_matrices(test_networks,  y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    logger.info("Y-matrices cached successfully")

    # ── STSI 26.04.07: learned_edge PTDF initialisation ─────────────────
    _is_learned_edge = (physics_cfg.ptdf_loss_mode == "learned_edge"
                        and physics_cfg.use_ptdf_loss
                        and physics_cfg.weight_ptdf > 0.0)
    ptdf_params = None
    max_buses = 0
    train_ptdf_offset = 0
    val_ptdf_offset = num_train_networks
    test_ptdf_offset = num_train_networks + len(val_networks)
    if _is_learned_edge:
        max_buses = max(len(net.buses) for net in networks_copy)
        ptdf_params = initialize_learned_ptdf(networks_copy, max_buses, ptdf_branch_mode=physics_cfg.ptdf_branch_mode)  # [STSI 260526]:pass branch mode
        print(f"[learned_edge] Initialized {len(ptdf_params)} PTDF params, max_buses={max_buses}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_with_ptdf
    )  # STSI 26.02.10: variable graph sizes + PTDF
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_with_ptdf
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_with_ptdf
    )

    # Initialize model based on sample
    sample_data = train_dataset[0]


    # STSI 26.04.07: augment edge feature dim when learned_edge PTDF rows are appended
    _edge_feat_dim = sample_data.edge_attr.size(1) + (max_buses if _is_learned_edge else 0)

    model = PowerFlowGNN(
        node_features=sample_data.x.size(1),
        edge_features=_edge_feat_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        conv_type=conv_type,
        head_mode=head_mode,
        use_residual=use_residual,  # [STSI 120526]:independent conv block flags
        norm_type=norm_type,
        activation=activation,
        use_dropout=use_dropout,
        drop_rate=drop_rate,
        conv_mode=conv_mode,
        angle_mode=angle_mode,  # [STSI 250526]: pass angle_mode to model for correct output head sizing and Δθ handling
        vmag_mode=vmag_mode,    # [STSI 280526]: residual V_mag prediction mode
        use_global_pool=use_global_pool,  # [STSI 290526]
    )
    print(model)
    #check_model_input_dim(model, train_dataset, label="train") # removed for now as it caused error and was not really needed now...
    # Optimizer & scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10, verbose=True
    )

    # STSI 26.04.07: add learned PTDF params to optimizer
    if ptdf_params is not None:
        optimizer.add_param_group({'params': ptdf_params})

    # STSI 16.02.26: Extended tracking of per-epoch components
    train_total_list, val_total_list = [], []
    train_mse_list, train_phys_list, train_ptdf_list = [], [], []
    val_mse_list,   val_phys_list,   val_ptdf_list   = [], [], []
    val_p_res_list,  val_q_res_list  = [], []          # [STSI 220526]: symmetric val diagnostics
    train_p_res_list, train_q_res_list = [], []  # [STSI 210526]: P/Q residual diagnostics
    train_angle_ref_list, val_angle_ref_list = [], []
    train_delta_theta_list, val_delta_theta_list = [], []  # [STSI 250526]: Δθ MSE tracking
    train_ptdf_matrix_list, train_ptdf_flows_list = [], []  # [STSI 290526] stepwise dual-log
    val_ptdf_matrix_list,   val_ptdf_flows_list   = [], []  # [STSI 290526] stepwise dual-log
    train_flow_loss_list, val_flow_loss_list = [], []  # [STSI 020626]:flow loss tracking (Task 045)
    train_w_flow_list = []  # [STSI 020626]:effective flow weight history
    best_val_loss   = float("inf")
    best_state_dict = None  # 26.03.12 STSI: track best model weights in memory

    # [STSI 200526] Replaces _effective_physics_cfg — unified weight computation for fixed/adaptive modes
    def _compute_loss_weights(
        cfg: PhysicsConfig,
        epoch: int,
        warmup_epochs: int,
        warmup_epochs_phys: int,
        warmup_epochs_ptdf: int,
        mse: torch.Tensor,
        raw_phys: torch.Tensor,
        raw_ptdf: torch.Tensor,
    ) -> tuple:
        """
        Compute effective weights for physics and PTDF losses.
        Handles 'fixed' and 'adaptive' modes, with symmetric linear warmup ramp.

        Returns:
            (eff_w_phys, eff_w_ptdf) — scalar floats to multiply raw losses
        """
        # Warmup toggle (symmetric for both losses, both modes)
        if warmup_epochs_ptdf == 0 and warmup_epochs > 0: # If only warmup_epochs is set, apply it to both losses for backwards compatibility
            warmup_epochs_ptdf = warmup_epochs
        if warmup_epochs_ptdf > 0 and epoch < warmup_epochs_ptdf:
            warmup_ptdf_scale = 0  #(epoch + 1) / warmup_epochs
        else:
            warmup_ptdf_scale = 1.0
            
        if warmup_epochs_phys == 0 and warmup_epochs > 0: # If only warmup_epochs is set, apply it to both losses for backwards compatibility
            warmup_epochs_phys = warmup_epochs
        if warmup_epochs_phys > 0 and epoch < warmup_epochs_phys:
            warmup_phys_scale = 0  #(epoch + 1) / warmup_epochs
        else:
            warmup_phys_scale = 1.0

        if cfg.loss_weight_mode == "adaptive":
            mse_val = mse.detach().item()

            # Physics weight
            phys_val = raw_phys.detach().item()
            if phys_val > 1e-8 and cfg.fraction_physics > 0:
                eff_w_phys = min(
                    cfg.fraction_physics * mse_val / phys_val,
                    cfg.max_w_phys,
                )
            else:
                eff_w_phys = 0.0

            # PTDF weight
            ptdf_val = raw_ptdf.detach().item()
            if ptdf_val > 1e-8 and cfg.fraction_ptdf > 0:
                eff_w_ptdf = min(
                    cfg.fraction_ptdf * mse_val / ptdf_val,
                    cfg.max_w_ptdf,
                )
            else:
                eff_w_ptdf = 0.0

        else:  # "fixed" mode — same as legacy behavior
            eff_w_phys = cfg.w_phys
            eff_w_ptdf = cfg.weight_ptdf

        # Apply warmup ramp
        eff_w_phys *= warmup_phys_scale
        eff_w_ptdf *= warmup_ptdf_scale

        return eff_w_phys, eff_w_ptdf

    # STSI 26.03.25: Restored masked MSE from early implementation — only penalise
    # unknown variables per bus type to avoid gradient dilution and shared-head conflicts.
    # PQ  → predict Vmag (0), Vang (1)
    # PV  → predict Vang (1), Q (3)
    # Slack → predict P (2), Q (3)

    def _masked_mse_loss(
        node_pred: torch.Tensor,
        target: torch.Tensor,
        batch,
        debug: bool = False,
        angle_mode: str = "node",  # [STSI 250526]: controls whether to use original masked MSE logic (angle_mode="node") or edge_delta-specific logic that maps the model output and target to the relevant variables for comparison (angle_mode="edge_delta")
    ) -> torch.Tensor:
        """
        MSE loss restricted to physically unknown variables per bus type.

        Bus type  | Predict (unknown)    | Skip (known input)
        ----------|----------------------|--------------------
        PQ        | Vmag (0), Vang (1)   | P (3), Q (4)
        PV        | Vang (1), Q (3)      | Vmag (5), P (3)
        Slack     | P (2), Q (3)         | Vmag (5), Vang (6)

        Reads masks from batch.slack_mask / batch.pv_mask / batch.pq_mask
        (bool tensors, node-level, set by PowerFlowDataset and propagated by
        collate_with_ptdf via Batch.from_data_list auto-concatenation).

        Returns 0 if all masks are missing (with a warning) to fail loudly.
        """
        # ── Source masks from batch attributes (preferred) or x columns ──────────
        if hasattr(batch, "slack_mask") and hasattr(batch, "pv_mask") and hasattr(batch, "pq_mask"):
            slack_mask = batch.slack_mask.bool()
            pv_mask    = batch.pv_mask.bool()
            pq_mask    = batch.pq_mask.bool()
            mask_source = "batch_attr"
        else:
            # Fallback: derive from x feature columns [is_slack=0, is_PV=1, is_PQ=2]
            logger.warning(
                "_masked_mse_loss: batch.slack_mask/pv_mask/pq_mask not found. "
                "Falling back to x[:,0:3] — check collate_with_ptdf and PowerFlowDataset."
            )
            slack_mask = batch.x[:, 0].bool()
            pv_mask    = batch.x[:, 1].bool()
            pq_mask    = batch.x[:, 2].bool()
            mask_source = "x_cols"

        if debug:
            print(f"[_masked_mse_loss] mask_source={mask_source}")
            print(f"  slack_mask sum={slack_mask.sum().item()}, "
                f"pv_mask sum={pv_mask.sum().item()}, "
                f"pq_mask sum={pq_mask.sum().item()}")

        # [STSI 250526]:edge_delta mode — node_pred is [N,3]=[Vmag,P,Q], angle handled by Δθ MSE
        # Note: angle_mode="both" produces [N,4] and falls to the else branch below (standard path)  # [STSI 020626]
        if angle_mode == "edge_delta":
            # Map target cols: y=[Vmag(0),Vang(1),P(2),Q(3)] → compare with pred=[Vmag(0),P(1),Q(2)]
            y_mapped = torch.stack([target[:, 0], target[:, 2], target[:, 3]], dim=-1)  # [N, 3]
            mask = torch.zeros_like(node_pred, dtype=torch.bool)
            mask[pq_mask, 0] = True    # Vmag unknown at PQ
            mask[pv_mask, 2] = True    # Q unknown at PV
            mask[slack_mask, 1] = True # P unknown at Slack
            mask[slack_mask, 2] = True # Q unknown at Slack
            diff = (node_pred - y_mapped) ** 2
            if mask.any():
                return diff[mask].mean()
            else:
                return torch.tensor(0.0, device=node_pred.device, requires_grad=True)

        loss = torch.tensor(0.0, device=node_pred.device)
        n    = 0

        # PQ buses: predict Vmag (col 0) and Vang (col 1)
        if pq_mask.any():
            loss = loss + F.mse_loss(node_pred[pq_mask][:, 0:2], target[pq_mask][:, 0:2])
            n += 1

        # PV buses: predict Vang (col 1) and Q (col 3)
        if pv_mask.any():
            pv_cols = torch.tensor([1, 3], device=node_pred.device)
            loss = loss + F.mse_loss(
                node_pred[pv_mask][:, pv_cols],
                target[pv_mask][:, pv_cols],
            )
            n += 1

        # Slack buses: predict P (col 2) and Q (col 3)
        if slack_mask.any():
            loss = loss + F.mse_loss(node_pred[slack_mask][:, 2:4], target[slack_mask][:, 2:4])
            n += 1

        if n == 0:
            logger.warning(
                "_masked_mse_loss: all masks are empty — returning 0. "
                "Training gradients will not flow. Check mask propagation."
            )
            return torch.tensor(0.0, device=node_pred.device, requires_grad=True)

        return loss / max(n, 1)
# STSI260402  removed nested helpers for ptdf losses (_compute_ptdf_loss, _compute_ptdf_loss_flows, _compute_ptdf_loss_matrix) to expose these to unit testing and smoke test
# ── Move _epoch0_debug definition here — BEFORE the epoch loop ──────────

    def _epoch0_debug(batch, node_pred):
        print("\n" + "="*60)
        print("EPOCH 0 VERIFICATION CHECKS")
        print("="*60)

        print("\n[P1] Mask propagation through collate_with_ptdf:")
        for attr in ("slack_mask", "pv_mask", "pq_mask"):
            if hasattr(batch, attr):
                s = getattr(batch, attr).sum().item()
                print(f"  [OK] batch.{attr} present -- sum={int(s)}")
            else:
                print(f"  [FAIL] batch.{attr} MISSING -- _masked_mse_loss will return 0!")

        print("\n[P2] Masked vs full MSE:")
        # [STSI 250526]:edge_delta mode produces [N,3] (no angle col) vs batch.y [N,4]
        if node_pred.shape[1] == batch.y.shape[1]:
            _y_cmp = batch.y
        else:
            # edge_delta: node_pred=[Vmag,P,Q], batch.y=[Vmag,Vang,P,Q] → compare cols 0,2,3
            _y_cmp = batch.y[:, [0, 2, 3]]
        full_mse   = F.mse_loss(node_pred, _y_cmp)
        masked_mse = _masked_mse_loss(node_pred, batch.y, batch, angle_mode=angle_mode)
        print(f"  Full MSE:   {full_mse.item():.6f}")
        print(f"  Masked MSE: {masked_mse.item():.6f}")
        if abs(full_mse.item() - masked_mse.item()) < 1e-8:
            print("  [WARN] Full MSE == Masked MSE -- masks may be inactive!")
        else:
            print("  [OK] Masked MSE differs from full MSE -- masks are active.")

        print("\n[P3] Physics state assembly:")
        slack_mask = batch.slack_mask.bool()
        pv_mask    = batch.pv_mask.bool()
        if slack_mask.any():
            print(f"  Slack P predicted: {node_pred[slack_mask, 2].detach()[:3]}")
            print(f"  Slack Vang input (should be ~0): {batch.x[slack_mask, 6].detach()[:3]}")
        else:
            print("  [WARN] No slack buses found in this batch!")
        if pv_mask.any():
            print(f"  PV Vmag input (should be ~1.0-1.05): {batch.x[pv_mask, 5].detach()[:3]}")

        print(f"\n[Info] batch.x shape: {batch.x.shape}  (expect [N,7] or [N,8])")
        print("="*60 + "\n")

    # ── Move flag here — BEFORE the epoch loop ───────────────────────────────
    _first_batch_debug_done = False
    print(f"train_loader length: {len(train_loader)}")
    print(f"train_dataset length: {len(train_dataset)}")
    print(f"train_networks length: {len(train_networks)}") 
    _tqdm_kw = dict(desc="Training", unit="epoch")
    if tqdm_file is not None:
        _tqdm_kw["file"] = tqdm_file
    train_w_phys_list, train_w_ptdf_list = [], []  # [STSI 200526] effective weight history
    # [STSI 290526] Stepwise PTDF mode resolution
    _ptdf_changepoint = physics_cfg.ptdf_changepoint if ptdf_changepoint is None else ptdf_changepoint
    if physics_cfg.ptdf_loss_mode == "stepwise":
        if not warmup_epochs_ptdf or warmup_epochs_ptdf <= 0:
            raise ValueError("ptdf_loss_mode='stepwise' requires warmup_epochs_ptdf > 0")
        if not _ptdf_changepoint or _ptdf_changepoint <= 0:
            raise ValueError("ptdf_loss_mode='stepwise' requires ptdf_changepoint > 0")

    def _resolve_ptdf_mode(cfg: PhysicsConfig, epoch: int, warmup_ptdf: int) -> str:
        """Return effective ptdf_loss_mode for this epoch (never returns 'stepwise')."""
        if cfg.ptdf_loss_mode != "stepwise":
            return cfg.ptdf_loss_mode
        if epoch < warmup_ptdf + _ptdf_changepoint:
            return "matrix"
        return "flows"

    for epoch in tqdm(range(num_epochs), **_tqdm_kw):

        # ---- Training ----
        model.train()
        epoch_loss = epoch_mse = epoch_physics = epoch_angle_ref = epoch_ptdf = 0.0
        epoch_p_res = epoch_q_res = 0.0  # [STSI 210526]: per-component residual diagnostics
        epoch_eff_w_phys = epoch_eff_w_ptdf = 0.0  # [STSI 200526] accumulate effective weights
        epoch_delta_theta = 0.0  # [STSI 250526]: Δθ MSE accumulator
        epoch_ptdf_matrix = epoch_ptdf_flows = 0.0  # [STSI 290526] dual-log for stepwise
        epoch_flow_loss = 0.0  # [STSI 020626]:flow loss accumulator (Task 045)
        epoch_eff_w_flow = 0.0  # [STSI 020626]:effective flow weight accumulator

        # ── Flow loss warmup gates ──────────────────────────── [STSI 020626]:Task 045
        use_flow_this_epoch = (physics_cfg.use_flow_loss and epoch >= warmup_epochs_flow)
        flow_active_global = (use_flow_this_epoch and epoch >= warmup_epochs_flow_global)
        flow_active_ac = (use_flow_this_epoch and epoch >= warmup_epochs_flow_ac)

        for batch in train_loader:
            optimizer.zero_grad()

            # STSI 26.04.07: augment edge_attr with learned PTDF rows
            if ptdf_params is not None:
                batch.edge_attr = build_ptdf_augmented_edge_attr(
                    batch, ptdf_params, max_buses, ptdf_offset=train_ptdf_offset)

            node_pred, h_nodes, h_edges, delta_theta_pred = model(batch, return_embeddings=True)  # [STSI 250526]:4-tuple

            #=====================Debug block
            if epoch == 0 and not _first_batch_debug_done:
                _epoch0_debug(batch, node_pred)
                _first_batch_debug_done = True
            #=====================End Debug block
            # ── MSE loss — masked to unknown variables per bus type ──────────
            # STSI 26.03.25: replaced full F.mse_loss(node_pred, batch.y) with
            # _masked_mse_loss to avoid gradient dilution from trivially-known
            # outputs (e.g. slack Vmag/Vang, PQ P/Q) and shared-head conflicts.
            mse = _masked_mse_loss(node_pred, batch.y, batch, angle_mode=angle_mode)  # [STSI 250526]:pass angle_mode

            # ── Δθ edge MSE loss (angle_mode="edge_delta") [STSI 250526]
            if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:  # [STSI 020626]:Δθ MSE for both modes
                y_delta_theta_batch = torch.cat(batch.y_delta_theta, dim=0).to(node_pred.device)# concatenate list of Δθ targets from batch into single tensor
                loss_delta_theta = F.mse_loss(delta_theta_pred, y_delta_theta_batch)
            else:
                loss_delta_theta = torch.tensor(0.0, device=node_pred.device)

            # ── Step 1: Compute raw losses ──────────────────────────────────  [STSI 200526]
            # [STSI 260526]:edge-local physics OR θ-reconstruction dispatch 
            use_edge_local_physics = (angle_mode in ("edge_delta", "both") and delta_theta_pred is not None)  # [STSI 020626]:edge-local physics for both modes
            if use_edge_local_physics:
                # Edge-local path: direct O(E) computation, no θ reconstruction
                import itertools as _itertools
                _dt_sizes = [t.size(0) for t in batch.y_delta_theta]
                _dt_offsets = [0] + list(_itertools.accumulate(_dt_sizes))
                _node_batch = batch.batch
                _num_graphs = int(_node_batch.max().item()) + 1
                physics_parts = []
                p_res_parts = []
                q_res_parts = []
                for _g in range(_num_graphs):
                    _nm = (_node_batch == _g)
                    _dt_s = _dt_offsets[_g]
                    _dt_e = _dt_offsets[_g + 1]
                    _dtp_g = delta_theta_pred[_dt_s:_dt_e]
                    _vmag_g = node_pred[_nm, 0]  # Vmag from node output
                    _edge_mask_g = (batch.batch[batch.edge_index[0]] == _g)
                    _phys_g, _pr_g, _qr_g = compute_physics_residual_edge_local(
                        delta_theta_pred=_dtp_g,
                        vmag=_vmag_g,
                        node_pred=node_pred[_nm],
                        edge_index=batch.edge_index[:, _edge_mask_g] - int(_nm.nonzero()[0][0]),
                        edge_attr=batch.edge_attr[_edge_mask_g],
                        g_diag=batch.g_diag[_nm],
                        b_diag=batch.b_diag[_nm],
                        x=batch.x[_nm],
                        bus_masks=(batch.slack_mask[_nm], batch.pv_mask[_nm], batch.pq_mask[_nm]),
                        use_q_partial_mode=physics_cfg.use_q_partial_mode,
                        w_P=physics_cfg.w_P, w_Q=physics_cfg.w_Q,
                    )
                    physics_parts.append(_phys_g)
                    p_res_parts.append(_pr_g)
                    q_res_parts.append(_qr_g)
                _pred_for_phys = None  # signal: edge-local path used
                _edge_local_physics = torch.stack(physics_parts).mean()
                _edge_local_p_res = torch.stack(p_res_parts).mean()
                _edge_local_q_res = torch.stack(q_res_parts).mean()
            elif angle_mode == "edge_delta" and delta_theta_pred is not None:
                # Fallback: reconstruct θ from Δθ for Y-bus physics (should not normally be used)
                import itertools as _itertools
                _dt_sizes = [t.size(0) for t in batch.y_delta_theta]
                _dt_offsets = [0] + list(_itertools.accumulate(_dt_sizes))
                _node_batch = batch.batch
                _num_graphs = int(_node_batch.max().item()) + 1
                node_pred_full = torch.zeros(node_pred.size(0), 4, device=node_pred.device)
                for _g in range(_num_graphs):
                    _nm = (_node_batch == _g) # node mask for graph g
                    _n_i = int(_nm.sum().item()) # number of nodes in graph g
                    _dt_s = _dt_offsets[_g] # start index for graph g in delta_theta_pred
                    _dt_e = _dt_offsets[_g + 1] # end index for graph g in delta_theta_pred
                    _dtp_g = delta_theta_pred[_dt_s:_dt_e].detach() # Δθ for graph g (detached to avoid backprop through reconstruction)
                    _theta_g = reconstruct_theta_from_delta(
                        _dtp_g,
                        batch.bfs_edge_idx[_g].to(node_pred.device),
                        batch.bfs_signs[_g].to(node_pred.device),
                        batch.bfs_node_order[_g].to(node_pred.device),
                        batch.bfs_parent[_g].to(node_pred.device),
                        _n_i,
                    )
                    node_pred_full[_nm, 0] = node_pred[_nm, 0]  # Vmag
                    node_pred_full[_nm, 1] = _theta_g            # θ reconstructed
                    node_pred_full[_nm, 2] = node_pred[_nm, 1]  # P (col1 in 3-col)
                    node_pred_full[_nm, 3] = node_pred[_nm, 2]  # Q (col2 in 3-col)
                _pred_for_phys = node_pred_full
                _edge_local_physics = None
            elif angle_mode == "both":  # [STSI 020626]: node_pred is already [N,4] with theta
                _pred_for_phys = node_pred
                _edge_local_physics = None
            else:
                _pred_for_phys = node_pred
                _edge_local_physics = None


            if _edge_local_physics is not None:  # [STSI 260526]: edge-local physics
                physics = _edge_local_physics
                angle_ref = torch.tensor(0.0, device=node_pred.device)
                p_res = _edge_local_p_res
                q_res = _edge_local_q_res
            elif physics_cfg.w_phys > 0.0 or physics_cfg.loss_weight_mode == "adaptive":
                _, _, physics, angle_ref, p_res, q_res = physics_informed_loss_batch(  # [STSI 210526]: unpack 6-tuple
                    _pred_for_phys, batch.y, batch,
                    networks=train_networks,
                    Y_cache=train_Y_cache,
                    physics_cfg=physics_cfg,
                )
            else:
                physics   = torch.tensor(0.0, device=node_pred.device)
                angle_ref = torch.tensor(0.0, device=node_pred.device)
                p_res = q_res = 0.0  # [STSI 210526]: no physics computed

            node_batch = batch.batch
            row, _     = batch.edge_index
            edge_batch = node_batch[row]
            num_graphs = int(node_batch.max().item()) + 1

            if physics_cfg.use_ptdf_loss:
                eff_ptdf_mode = _resolve_ptdf_mode(physics_cfg, epoch, warmup_epochs_ptdf)  # [STSI 290526]
                ptdf_loss = compute_ptdf_loss(
                    h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model,
                    ptdf_loss_mode=eff_ptdf_mode,
                    ptdf_alpha=physics_cfg.ptdf_alpha,
                    node_pred=node_pred, ptdf_params=ptdf_params,
                    ptdf_offset=train_ptdf_offset,
                )
                # [STSI 290526] dual-log: compute both modes for diagnostics
                with torch.no_grad():
                    _ptdf_mat = compute_ptdf_loss(
                        h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model,
                        ptdf_loss_mode="matrix", node_pred=node_pred,
                        ptdf_params=ptdf_params, ptdf_offset=train_ptdf_offset,
                    )
                    _ptdf_flw = compute_ptdf_loss(
                        h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model,
                        ptdf_loss_mode="flows", node_pred=node_pred,
                        ptdf_params=ptdf_params, ptdf_offset=train_ptdf_offset,
                    )
            else:
                ptdf_loss = torch.tensor(0.0, device=node_pred.device)
                _ptdf_mat = _ptdf_flw = ptdf_loss  # [STSI 290526]

            # ── Step 2: Compute effective weights ───────────────────────────  [STSI 200526]
            eff_w_phys, eff_w_ptdf = _compute_loss_weights(
                physics_cfg, epoch, warmup_epochs=warmup_epochs, warmup_epochs_phys=warmup_epochs_phys, warmup_epochs_ptdf=warmup_epochs_ptdf,
                mse=mse, raw_phys=physics, raw_ptdf=ptdf_loss,
            )

            # ── Step 3: Combine ─────────────────────────────────────────────  [STSI 200526]

            # ── Per-edge flow loss (Task 045) ──────────────────────── [STSI 020626]
            flow_loss_val = torch.tensor(0.0, device=node_pred.device)
            eff_w_flow = 0.0
            if use_flow_this_epoch and delta_theta_pred is not None:
                # Extract theta from node predictions (col 1) for global mode
                theta_pred_flow = None
                if angle_mode in ('node', 'both') and node_pred.shape[1] >= 4:
                    theta_pred_flow = node_pred[:, 1]
                # Extract vmag from node predictions (col 0)
                vmag_pred_flow = node_pred[:, 0]

                _y_lp = torch.cat(batch.y_line_p_list, dim=0).to(node_pred.device)
                _y_lq = torch.cat(batch.y_line_q_list, dim=0).to(node_pred.device) if hasattr(batch, 'y_line_q_list') else None
                flow_loss_raw, flow_diag = compute_flow_loss(
                    cfg=physics_cfg,
                    delta_theta_pred=delta_theta_pred,
                    theta_pred=theta_pred_flow,
                    vmag=vmag_pred_flow,
                    edge_index=batch.edge_index,
                    edge_attr=batch.edge_attr,
                    y_line_p=_y_lp,
                    y_line_q=_y_lq,
                    line_mask=batch.forward_edge_mask[::2],  # [STSI 020626]:Fix — supervised-line mask skips trafos in batch
                    dc_line_mask=batch.dc_flow_mask[::2],  # [STSI 020626]:lines-only mask for DC flow loss
                    active_local=True,
                    active_global=flow_active_global,
                    active_ac=flow_active_ac,
                )

                # Apply weighting: fixed or adaptive
                if physics_cfg.loss_weight_mode == 'adaptive':
                    mse_val_det = mse.detach().item()
                    flow_val_det = flow_loss_raw.detach().item()
                    if flow_val_det > 1e-10 and physics_cfg.fraction_flow > 0:
                        eff_w_flow = min(
                            physics_cfg.fraction_flow * mse_val_det / flow_val_det,
                            physics_cfg.max_w_flow,
                        )
                    else:
                        eff_w_flow = 0.0
                else:  # fixed mode — w_flow is already applied inside compute_flow_loss
                    eff_w_flow = 1.0  # flow_loss_raw already includes cfg.w_flow scaling

                flow_loss_val = eff_w_flow * flow_loss_raw

            loss = mse + loss_delta_theta + eff_w_phys * physics + eff_w_ptdf * ptdf_loss + flow_loss_val  # [STSI 020626]:+flow  # [STSI 250526]:added Δθ MSE

            loss.backward()
            optimizer.step()

            epoch_loss    += loss.item()
            epoch_mse     += mse.item()
            epoch_physics += physics.item()
            epoch_angle_ref += angle_ref.item()
            epoch_ptdf    += ptdf_loss.item()
            epoch_ptdf_matrix += _ptdf_mat.item()   # [STSI 290526]
            epoch_ptdf_flows  += _ptdf_flw.item()   # [STSI 290526]
            epoch_delta_theta += loss_delta_theta.item()  # [STSI 250526]: Δθ MSE
            epoch_p_res   += p_res  # [STSI 210526]: accumulate P residual diagnostic
            epoch_q_res   += q_res  # [STSI 210526]: accumulate Q residual diagnostic
            epoch_eff_w_phys += eff_w_phys  # [STSI 200526]
            epoch_eff_w_ptdf += eff_w_ptdf  # [STSI 200526]
            epoch_flow_loss += flow_loss_val.item()  # [STSI 020626]:flow loss accumulator
            epoch_eff_w_flow += eff_w_flow  # [STSI 020626]:effective flow weight

        # Backward step averages
        train_loss = epoch_loss / len(train_loader)
        train_total_list.append(train_loss)
        train_mse_list.append(epoch_mse      / len(train_loader))
        train_phys_list.append(epoch_physics / len(train_loader))
        train_angle_ref_list.append(epoch_angle_ref / len(train_loader))
        train_ptdf_list.append(epoch_ptdf    / len(train_loader))
        train_ptdf_matrix_list.append(epoch_ptdf_matrix / len(train_loader))  # [STSI 290526]
        train_ptdf_flows_list.append(epoch_ptdf_flows   / len(train_loader))  # [STSI 290526]
        train_delta_theta_list.append(epoch_delta_theta / len(train_loader))  # [STSI 250526]
        train_p_res_list.append(epoch_p_res  / len(train_loader))  # [STSI 210526]
        train_q_res_list.append(epoch_q_res  / len(train_loader))  # [STSI 210526]
        train_w_phys_list.append(epoch_eff_w_phys / len(train_loader))  # [STSI 200526]
        train_w_ptdf_list.append(epoch_eff_w_ptdf / len(train_loader))  # [STSI 200526]
        train_flow_loss_list.append(epoch_flow_loss / len(train_loader))  # [STSI 020626]:flow loss
        train_w_flow_list.append(epoch_eff_w_flow / len(train_loader))  # [STSI 020626]:eff flow weight

        # ---- Validation ----
        model.eval()
        val_loss = val_mse = val_physics = val_angle_ref = val_ptdf = 0.0
        val_p_res = val_q_res = 0.0  # [STSI 220526]
        val_delta_theta = 0.0  # [STSI 250526]
        val_flow_loss = 0.0  # [STSI 020626]:validation flow loss (Task 045)
        val_ptdf_matrix = val_ptdf_flows = 0.0  # [STSI 290526] dual-log for stepwise
        with torch.no_grad():
            for batch in val_loader:
                # STSI 26.04.07: augment edge_attr with learned PTDF rows
                if ptdf_params is not None:
                    batch.edge_attr = build_ptdf_augmented_edge_attr(
                        batch, ptdf_params, max_buses, ptdf_offset=val_ptdf_offset)

                node_pred, h_nodes, h_edges, delta_theta_pred = model(batch, return_embeddings=True)  # [STSI 250526]:4-tuple

                # STSI 26.03.25: use masked MSE for validation consistency
                mse = _masked_mse_loss(node_pred, batch.y, batch, angle_mode=angle_mode)  # [STSI 250526]:pass angle_mode

                # [STSI 250526]:reconstruct θ for val physics in edge_delta mode
                if angle_mode == "edge_delta" and delta_theta_pred is not None:
                    import itertools as _itertools
                    _dt_sizes = [t.size(0) for t in batch.y_delta_theta]
                    _dt_offsets = [0] + list(_itertools.accumulate(_dt_sizes))
                    _node_batch = batch.batch
                    _num_graphs = int(_node_batch.max().item()) + 1
                    node_pred_full = torch.zeros(node_pred.size(0), 4, device=node_pred.device)
                    for _g in range(_num_graphs):
                        _nm = (_node_batch == _g)
                        _n_i = int(_nm.sum().item())
                        _dtp_g = delta_theta_pred[_dt_offsets[_g]:_dt_offsets[_g + 1]]
                        _theta_g = reconstruct_theta_from_delta(
                            _dtp_g,
                            batch.bfs_edge_idx[_g].to(node_pred.device),
                            batch.bfs_signs[_g].to(node_pred.device),
                            batch.bfs_node_order[_g].to(node_pred.device),
                            batch.bfs_parent[_g].to(node_pred.device),
                            _n_i,
                        )
                        node_pred_full[_nm, 0] = node_pred[_nm, 0]
                        node_pred_full[_nm, 1] = _theta_g
                        node_pred_full[_nm, 2] = node_pred[_nm, 1]
                        node_pred_full[_nm, 3] = node_pred[_nm, 2]
                    _pred_for_phys = node_pred_full
                elif angle_mode == "both":  # [STSI 020626]: node_pred is already [N,4]
                    _pred_for_phys = node_pred
                else:
                    _pred_for_phys = node_pred

                # ── Step 1: Compute raw losses (val) ────────────────────────  [STSI 200526]
                if physics_cfg.w_phys > 0.0 or physics_cfg.loss_weight_mode == "adaptive":
                    _, _, physics, angle_ref, p_res, q_res = physics_informed_loss_batch(  # [STSI 220526]: capture p/q residuals
                        _pred_for_phys, batch.y, batch,
                        networks=val_networks,
                        Y_cache=val_Y_cache,
                        physics_cfg=physics_cfg,
                    )
                else:
                    physics   = torch.tensor(0.0, device=node_pred.device)
                    angle_ref = torch.tensor(0.0, device=node_pred.device)
                    p_res = q_res = 0.0  # [STSI 220526]

                node_batch = batch.batch
                row, _     = batch.edge_index
                edge_batch = node_batch[row]
                num_graphs = int(node_batch.max().item()) + 1

                if physics_cfg.use_ptdf_loss:
                    eff_ptdf_mode = _resolve_ptdf_mode(physics_cfg, epoch, warmup_epochs_ptdf)  # [STSI 290526]
                    ptdf_loss = compute_ptdf_loss(
                        h_nodes, h_edges, batch,
                        node_batch, edge_batch, num_graphs,
                        model=model,
                        ptdf_loss_mode=eff_ptdf_mode,
                        ptdf_alpha=physics_cfg.ptdf_alpha,
                        node_pred=node_pred, ptdf_params=ptdf_params,
                        ptdf_offset=val_ptdf_offset,
                    )
                    # [STSI 290526] dual-log: both modes for diagnostics
                    _ptdf_mat = compute_ptdf_loss(
                        h_nodes, h_edges, batch,
                        node_batch, edge_batch, num_graphs, model=model,
                        ptdf_loss_mode="matrix", node_pred=node_pred,
                        ptdf_params=ptdf_params, ptdf_offset=val_ptdf_offset,
                    )
                    _ptdf_flw = compute_ptdf_loss(
                        h_nodes, h_edges, batch,
                        node_batch, edge_batch, num_graphs, model=model,
                        ptdf_loss_mode="flows", node_pred=node_pred,
                        ptdf_params=ptdf_params, ptdf_offset=val_ptdf_offset,
                    )
                else:
                    ptdf_loss = torch.tensor(0.0, device=node_pred.device)
                    _ptdf_mat = _ptdf_flw = ptdf_loss  # [STSI 290526]

                # ── Step 2: Compute effective weights (val: no warmup) ──────  [STSI 200526]
                eff_w_phys, eff_w_ptdf = _compute_loss_weights(
                    physics_cfg, epoch=warmup_epochs, warmup_epochs=warmup_epochs,warmup_epochs_phys=warmup_epochs, warmup_epochs_ptdf=warmup_epochs,
                    mse=mse, raw_phys=physics, raw_ptdf=ptdf_loss,
                )


                # ── Δθ MSE for val (symmetric with training) ────────────────  [STSI 250526]
                if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:  # [STSI 020626]:Δθ metric for both
                    y_delta_theta_batch = torch.cat(batch.y_delta_theta, dim=0).to(node_pred.device)
                    loss_delta_theta_val = F.mse_loss(delta_theta_pred, y_delta_theta_batch)
                else:
                    loss_delta_theta_val = torch.tensor(0.0, device=node_pred.device)

                # ── Step 3: Combine ─────────────────────────────────────────  [STSI 200526]

                # ── Validation flow loss (Task 045) ─────────────── [STSI 020626]
                val_flow_loss_batch = torch.tensor(0.0, device=node_pred.device)
                if use_flow_this_epoch and delta_theta_pred is not None:
                    theta_pred_val = None
                    if angle_mode in ('node', 'both') and node_pred.shape[1] >= 4:
                        theta_pred_val = node_pred[:, 1]
                    vmag_pred_val = node_pred[:, 0]
                    _y_lp_val = torch.cat(batch.y_line_p_list, dim=0).to(node_pred.device)
                    _y_lq_val = torch.cat(batch.y_line_q_list, dim=0).to(node_pred.device) if hasattr(batch, 'y_line_q_list') else None
                    val_flow_raw, _ = compute_flow_loss(
                        cfg=physics_cfg,
                        delta_theta_pred=delta_theta_pred,
                        theta_pred=theta_pred_val,
                        vmag=vmag_pred_val,
                        edge_index=batch.edge_index,
                        edge_attr=batch.edge_attr,
                        y_line_p=_y_lp_val,
                        y_line_q=_y_lq_val,
                        line_mask=batch.forward_edge_mask[::2],  # [STSI 020626]:Fix — supervised-line mask
                        dc_line_mask=batch.dc_flow_mask[::2],  # [STSI 020626]:lines-only mask for DC flow loss
                        active_local=True,
                        active_global=flow_active_global,
                        active_ac=flow_active_ac,
                    )
                    val_flow_loss_batch = val_flow_raw

                loss = mse + loss_delta_theta_val + eff_w_phys * physics + eff_w_ptdf * ptdf_loss + val_flow_loss_batch  # [STSI 020626]:+flow
                val_loss    += loss.item()
                val_mse     += mse.item()
                val_physics += physics.item()
                val_angle_ref += angle_ref.item()
                val_ptdf    += ptdf_loss.item()
                val_ptdf_matrix += _ptdf_mat.item()   # [STSI 290526]
                val_ptdf_flows  += _ptdf_flw.item()   # [STSI 290526]
                val_p_res   += p_res   # [STSI 220526]
                val_q_res   += q_res   # [STSI 220526]
                val_delta_theta += loss_delta_theta_val.item()  # [STSI 250526]
                val_flow_loss += val_flow_loss_batch.item()  # [STSI 020626]

        val_loss /= len(val_loader)
        val_total_list.append(val_loss)
        val_mse_list.append(val_mse       / len(val_loader))
        val_phys_list.append(val_physics  / len(val_loader))
        val_angle_ref_list.append(val_angle_ref / len(val_loader)) # STSI 260403: log angle reference error separately
        val_ptdf_list.append(val_ptdf     / len(val_loader))
        val_ptdf_matrix_list.append(val_ptdf_matrix / len(val_loader))  # [STSI 290526]
        val_ptdf_flows_list.append(val_ptdf_flows   / len(val_loader))  # [STSI 290526]
        val_p_res_list.append(val_p_res   / len(val_loader))  # [STSI 220526]
        val_q_res_list.append(val_q_res   / len(val_loader))  # [STSI 220526]
        val_delta_theta_list.append(val_delta_theta / len(val_loader))  # [STSI 250526]
        val_flow_loss_list.append(val_flow_loss / len(val_loader))  # [STSI 020626]

        scheduler.step(val_loss)

        # Save best model — use physics label in filename
        save_name = (
            f"training_results_saved/best_model"
            f"_{physics_cfg.label()}"
            f"_pt{physics_cfg.weight_ptdf}_bs{batch_size}_lr{lr}.pt"
        )

        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            safe_torch_save(model.state_dict(), save_name)

        print(
            f"Epoch {epoch+1}/{num_epochs}, "
            f"Train Loss {train_loss:.6f}, "
            f"Val Loss {val_loss:.6f}, "
            f"LR {optimizer.param_groups[0]['lr']:.6f}"
        )
        # Debug to detect data leakage (epoch 0 only)
        if epoch == 0:
            print("DATA LEAKAGE CHECK")
            print(f"Input features batch.x shape {batch.x.shape}")
            print(f"Input features sample: {batch.x[:5]}")
            print(f"Target features batch.y shape {batch.y.shape}")
            print(f"Target features sample: {batch.y[:5]}")
            print("CHECKING FOR LEAKAGE")
            pq_indices = torch.where(batch.pq_mask)[0]
            if len(pq_indices) > 0:
                print(f"PQ bus indices: {pq_indices}")
                print(f"Input at PQ bus 0: {batch.x[pq_indices[0]]}")
                print(f"Target at PQ bus 0: {batch.y[pq_indices[0]]}")

    # ---- Test evaluation ----
    # after training loop, once you've tracked best_state_dict
    # 26.03.12 STSI: ensure we evaluate the best model, not the last epoch
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    model.eval()

    test_total = test_mse = test_physics = test_angle_ref = test_ptdf = 0.0
    with torch.no_grad():
        for batch in test_loader:
            # STSI 26.04.07: augment edge_attr with learned PTDF rows
            if ptdf_params is not None:
                batch.edge_attr = build_ptdf_augmented_edge_attr(
                    batch, ptdf_params, max_buses, ptdf_offset=test_ptdf_offset)

            node_pred, h_nodes, h_edges, delta_theta_pred = model(batch, return_embeddings=True)  # [STSI 250526]:4-tuple
            mse = _masked_mse_loss(node_pred, batch.y, batch, angle_mode=angle_mode)

            # [STSI 250526]:reconstruct θ for test physics in edge_delta mode
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                import itertools as _itertools
                _dt_sizes = [t.size(0) for t in batch.y_delta_theta]
                _dt_offsets = [0] + list(_itertools.accumulate(_dt_sizes))
                _node_batch = batch.batch
                _num_graphs = int(_node_batch.max().item()) + 1
                node_pred_full = torch.zeros(node_pred.size(0), 4, device=node_pred.device)
                for _g in range(_num_graphs):
                    _nm = (_node_batch == _g)
                    _n_i = int(_nm.sum().item())
                    _dtp_g = delta_theta_pred[_dt_offsets[_g]:_dt_offsets[_g + 1]]
                    _theta_g = reconstruct_theta_from_delta(
                        _dtp_g,
                        batch.bfs_edge_idx[_g].to(node_pred.device),
                        batch.bfs_signs[_g].to(node_pred.device),
                        batch.bfs_node_order[_g].to(node_pred.device),
                        batch.bfs_parent[_g].to(node_pred.device),
                        _n_i,
                    )
                    node_pred_full[_nm, 0] = node_pred[_nm, 0]
                    node_pred_full[_nm, 1] = _theta_g
                    node_pred_full[_nm, 2] = node_pred[_nm, 1]
                    node_pred_full[_nm, 3] = node_pred[_nm, 2]
                _pred_for_phys = node_pred_full
            elif angle_mode == "both":  # [STSI 020626]: node_pred is already [N,4] with theta
                _pred_for_phys = node_pred
            else:
                _pred_for_phys = node_pred

            # ── Step 1: Compute raw losses (test) ───────────────────────────  [STSI 200526]
            if physics_cfg.w_phys > 0.0 or physics_cfg.loss_weight_mode == "adaptive":
                _, _, physics, angle_ref, _, _ = physics_informed_loss_batch(  # [STSI 210526]: unpack 6-tuple
                    _pred_for_phys, batch.y, batch,
                    networks=test_networks,
                    Y_cache=test_Y_cache,
                    physics_cfg=physics_cfg,
                )
            else:
                physics   = torch.tensor(0.0, device=node_pred.device)
                angle_ref = torch.tensor(0.0, device=node_pred.device)

            node_batch = batch.batch
            row, _     = batch.edge_index
            edge_batch = node_batch[row]
            num_graphs = int(node_batch.max().item()) + 1

            if physics_cfg.use_ptdf_loss:
                eff_ptdf_mode = _resolve_ptdf_mode(physics_cfg, epoch, warmup_epochs_ptdf)  # [STSI 290526]
                ptdf_loss = compute_ptdf_loss(
                    h_nodes, h_edges, batch,
                    node_batch, edge_batch, num_graphs, model,
                    ptdf_loss_mode=eff_ptdf_mode,
                    ptdf_alpha=physics_cfg.ptdf_alpha,
                    node_pred=node_pred, ptdf_params=ptdf_params,
                    ptdf_offset=test_ptdf_offset,
                )
            else:
                ptdf_loss = torch.tensor(0.0, device=node_pred.device)

            # ── Step 2: Compute effective weights (test: no warmup) ─────────  [STSI 200526]
            eff_w_phys, eff_w_ptdf = _compute_loss_weights(
                physics_cfg, epoch=warmup_epochs, warmup_epochs=warmup_epochs,warmup_epochs_ptdf=warmup_epochs,warmup_epochs_phys=warmup_epochs_phys,
                mse=mse, raw_phys=physics, raw_ptdf=ptdf_loss,
            )

            # [STSI 250526]:Δθ loss for test
            if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:  # [STSI 020626]:Δθ metric for both
                y_delta_theta_batch = torch.cat(batch.y_delta_theta, dim=0).to(node_pred.device)
                loss_delta_theta_test = F.mse_loss(delta_theta_pred, y_delta_theta_batch)
            else:
                loss_delta_theta_test = torch.tensor(0.0, device=node_pred.device)

            # ── Step 3: Combine ─────────────────────────────────────────────  [STSI 200526]
            loss = mse + loss_delta_theta_test + eff_w_phys * physics + eff_w_ptdf * ptdf_loss  # [STSI 250526]:added Δθ MSE
            test_total   += loss.item()
            test_mse     += mse.item()
            test_physics += physics.item() # physics loss logged but includes both power residual and angle reference components
            test_angle_ref += angle_ref.item() # STSI 260403: log angle reference error separately 
            test_ptdf    += ptdf_loss.item()

    test_total   /= len(test_loader)
    test_mse     /= len(test_loader)
    test_physics /= len(test_loader)
    test_angle_ref /= len(test_loader) # STSI 260403: log angle reference error separately
    test_ptdf    /= len(test_loader)

    print("TEST METRICS")
    print(f"Test total loss: {test_total:.6f}")
    print(f"Test MSE loss:   {test_mse:.6f}")
    print(f"Test physics:    {test_physics:.6f}")
    print(f"Test angle ref:  {test_angle_ref:.6f}") # STSI 260403: log angle reference error separately
    print(f"Test PTDF:       {test_ptdf:.6f}")

    history = {
        "train_total": train_total_list, "val_total": val_total_list,
        "train_mse":   train_mse_list,   "val_mse":   val_mse_list,
        "train_phys":  train_phys_list,  "val_phys":  val_phys_list,
        "train_angle_ref": train_angle_ref_list, "val_angle_ref": val_angle_ref_list,
        "train_ptdf":  train_ptdf_list,  "val_ptdf":  val_ptdf_list,
        "train_ptdf_matrix": train_ptdf_matrix_list, "train_ptdf_flows": train_ptdf_flows_list,  # [STSI 290526]
        "val_ptdf_matrix":   val_ptdf_matrix_list,   "val_ptdf_flows":   val_ptdf_flows_list,    # [STSI 290526]
        "train_delta_theta_mse": train_delta_theta_list, "val_delta_theta_mse": val_delta_theta_list,  # [STSI 250526]
        "train_eff_w_phys": train_w_phys_list,  # [STSI 200526] effective weight history
        "train_eff_w_ptdf": train_w_ptdf_list,  # [STSI 200526] effective weight history
        "train_p_res": train_p_res_list,  # [STSI 210526]: P-residual diagnostic
        "train_q_res": train_q_res_list,  # [STSI 210526]: Q-residual diagnostic
        "val_p_res":   val_p_res_list,    # [STSI 220526]: symmetric val diagnostic
        "val_q_res":   val_q_res_list,    # [STSI 220526]: symmetric val diagnostic
        "train_flow_loss": train_flow_loss_list,  # [STSI 020626]:flow loss (Task 045)
        "val_flow_loss":   val_flow_loss_list,    # [STSI 020626]
        "train_eff_w_flow": train_w_flow_list,    # [STSI 020626]:effective flow weight
    }

    test_metrics = {
        "test_total":   test_total,
        "test_mse":     test_mse,
        "test_physics": test_physics,
        "test_angle_ref": test_angle_ref, # STSI 260403: added angle reference error to test metrics
        "test_ptdf":    test_ptdf,
    }

    # Optional: add node-wise metrics over all test networks
    nodewise = evaluate_gnn_on_test_set(
        model, test_networks, use_edge_features=use_edge_features,
        ptdf_params=ptdf_params, max_buses=max_buses,       # STSI 26.04.07
        ptdf_offset=test_ptdf_offset,                       # STSI 26.04.07
        angle_mode=angle_mode,  # [STSI 250526]: pass angle_mode to test-time evaluation for consistency with training
    )
    test_metrics.update(nodewise)

    # STSI 26.04.07: store learned_edge artefacts so callers can augment edge_attr
    test_metrics["_ptdf_params"] = ptdf_params   # list[nn.Parameter] or None
    test_metrics["_max_buses"]   = max_buses      # int (0 when not learned_edge)

    if return_debug_objects: #STSI260402: return debug objects if flag is set
        debug_objects = {
            "train_networks": train_networks,
            "val_networks": val_networks,
            "test_networks": test_networks,
            "train_dataset": train_dataset,
            "val_dataset": val_dataset,
            "test_dataset": test_dataset,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "test_loader": test_loader,
            "train_Y_cache": train_Y_cache,
            "val_Y_cache": val_Y_cache,
            "test_Y_cache": test_Y_cache,
        }
        return model, history, test_metrics, debug_objects

    return model, history, test_metrics

def safe_torch_save(state_dict, path, max_retries=5, delay=0.5):
    import time, os, torch
    tmp_path = f"{path}.tmp_{os.getpid()}"
    for attempt in range(1, max_retries + 1):
        try:
            torch.save(state_dict, tmp_path)
            if os.path.exists(path):
                os.remove(path)
            os.replace(tmp_path, path)
            return
        except Exception as e:
            if attempt == max_retries:
                print(f"[WARN] Failed to save model to {path} after {max_retries} attempts: {e}")
                try:
                    if os.path.exists(tmp_path):
                        os.remove(tmp_path)
                except Exception:
                    pass
                return
            time.sleep(delay)

#debug helpers
# %%
# Replace _epoch0_debug with ASCII-safe version (Windows cp1252 compatible)

def check_model_input_dim(model, dataset, label="train"):
    """Verify model input dim matches dataset node feature dim."""
    sample = dataset[0]
    expected_dim = sample.x.size(1)

    # GATv2Conv projects inputs via a linear layer before attention.
    # The correct place to read input dim is the weight matrix of that
    # linear layer, not in_channels (which stores hidden/output dim).
    model_in_dim = None
    try:
        # PowerFlowGNN typically has an input projection: self.input_proj or
        # the first conv layer's lin_src / lin_l weight
        first_conv = model.convs[0]
        if hasattr(first_conv, "lin_l"):
            model_in_dim = first_conv.lin_l.weight.size(1)
        elif hasattr(first_conv, "lin_src"):
            model_in_dim = first_conv.lin_src.weight.size(1)
        elif hasattr(model, "input_proj"):
            model_in_dim = model.input_proj.weight.size(1)
    except Exception:
        model_in_dim = None

    if model_in_dim is None:
        print(f"⚠️  [{label}] Could not determine model input dim — skipping check. "
              f"Dataset node feature dim = {expected_dim}.")
        return

    if model_in_dim != expected_dim:
        raise ValueError(
            f"Model input dim ({model_in_dim}) != dataset node feature dim ({expected_dim}). "
            f"Rebuild model with node_features=sample_data.x.size(1)."
        )

    print(f"✅ [{label}] Model input dim {model_in_dim} matches dataset dim {expected_dim}.")

In [ ]:
# =============================================================================
# SECTION 9: INFERENCE & PREDICTION
# =============================================================================

def _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset=0, device=None):
    """Augment data.edge_attr with learned PTDF rows for learned_edge models.

    Shared helper used by evaluate_model, compare_pf_conventional_vs_gnn,
    predict_network_results_with_masks, and evaluate_gnn_on_test_set.

    Args:
        data:          single PyG Data object (already moved to device)
        ptdf_params:   list[nn.Parameter] or None
        max_buses:     int, PTDF parameter width
        net_idx:       int, network index within the dataset
        ptdf_offset:   int, added to net_idx to get global ptdf_params index
        device:        torch device (inferred from data.edge_attr if None)

    Returns:
        data with edge_attr augmented in-place. No-op if ptdf_params is None.
    """
    if ptdf_params is None or max_buses <= 0:
        return data
    if device is None:
        device = data.edge_attr.device
    row_idx = data.ptdf_edge_row_idx  # [E] line index or -1
    ptdf_rows = torch.zeros(data.edge_attr.size(0), max_buses, device=device)
    valid = (row_idx >= 0) & (row_idx < ptdf_params[net_idx + ptdf_offset].size(0))
    if valid.any():
        ptdf_rows[valid] = ptdf_params[net_idx + ptdf_offset][row_idx[valid]].to(device)
    data.edge_attr = torch.cat([data.edge_attr, ptdf_rows], dim=-1)
    return data


def model_results(network, print_results=False, plot_results=False, compute_ptdf=False):
    """
    Run PyPSA power flow and store results in DataFrames.
    Print if print_results=True, plot if plot_results=True.

    DISTSLACK: pass distribute_slack=True to use PyPSA's distributed slack solver.
    """
    # DISTSLACK: use distribute_slack=True to properly handle distributed slack
    network.pf(use_seed=True, distribute_slack=False)  # DISTSLACK: change to True when using distributed slack

    snapshots = network.snapshots
    buses = network.buses.index
    lines = network.lines.index

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )

    for t in snapshots:
        for bus in buses:
            bus_results.loc[t, (bus, "P pu")]    = network.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, "Q pu")]  = network.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, "V pu")]    = network.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])
        for line in lines:
            line_results.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    if print_results:
        print(bus_results)
        print(line_results)
    if plot_results:
        plot_network_results(network, bus_results)

    ptdf_df = None
    if compute_ptdf:
        ptdf_df = compute_ptdf_matrix(network)

    return bus_results, line_results, ptdf_df


def batched_gnn_forward(model, data_list):
    """Run GNN on a batch of graphs in a single forward pass."""
    exclude_keys = ['y_ptdf', 'ptdf_line_index', 'y_line_p',
                    'bfs_edge_idx', 'bfs_signs', 'bfs_node_order',
                    'bfs_parent', 'y_delta_theta']  # [STSI 250526]:added BFS keys
    batch = Batch.from_data_list(data_list, exclude_keys=exclude_keys)
    out = model(batch)
    if isinstance(out, tuple):
        out = out[0]
    results = []
    for g in range(len(data_list)):
        mask = (batch.batch == g)
        results.append(out[mask])
    return results


def predict_network_results_with_masks(
    model, network, use_edge_features=True, debug=False, use_pnom_share=None,  # [STSI100526] added; None=auto-detect
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None, max_buses=0, ptdf_offset=0,
    angle_mode=None,  # [STSI 250526]: None=auto-detect from model.angle_mode
):
    """
    Run GNN prediction on a network and return bus/line results with prediction masks.

    DISTSLACK: prediction masks now also flag distributed slack buses correctly.
    Distributed slack buses predict P and Q (same as single slack).
    STSI 26.04.07: Augments edge_attr when ptdf_params is provided (learned_edge).

    Args:
        model:   Trained PowerFlowGNN model
        network: PyPSA network object (must have solved power flow for feature extraction)
    Returns:
        bus_results, line_results, prediction_masks
    """
    dataset = PowerFlowDataset(
        [network],
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
    )
    snapshots = network.snapshots
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product(
            [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
        ),
        dtype=float,
    )
    # Prediction masks: which properties were predicted (not taken from input)
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    model.eval()
    with torch.no_grad():
        for t_idx, t in enumerate(snapshots):
            data = dataset[t_idx]

            # STSI 26.04.07: augment edge_attr for learned_edge models
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx=0, ptdf_offset=ptdf_offset)

            # Fallback: zero-pad if model expects wider edge_attr (old runs without stored ptdf_params)
            if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
                _expected = model.convs[0].lin_edge.in_channels
                if data.edge_attr.size(1) < _expected:
                    data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)

            node_pred, *rest = model(data)  # [STSI 250526]:unpack (node_pred, ptdf_pred, delta_theta_pred)

            # [STSI 250526]: reconstruct θ for edge_delta mode
            _am = angle_mode if angle_mode is not None else getattr(model, "angle_mode", "node")
            if _am == "edge_delta" and len(rest) >= 2 and rest[-1] is not None:
                delta_theta_pred = rest[-1]
                _theta = reconstruct_theta_from_delta(
                    delta_theta_pred,
                    data.bfs_edge_idx.to(node_pred.device),
                    data.bfs_signs.to(node_pred.device),
                    data.bfs_node_order.to(node_pred.device),
                    data.bfs_parent.to(node_pred.device),
                    node_pred.size(0),
                )
                # Assemble [N,4]: Vmag, θ, P, Q
                node_pred_full = torch.zeros(node_pred.size(0), 4, device=node_pred.device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = _theta            # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P (was col 1 in [N,3])
                node_pred_full[:, 3] = node_pred[:, 2]  # Q (was col 2 in [N,3])
                node_pred = node_pred_full

            pred_vmag = node_pred[:, 0]
            pred_vang = node_pred[:, 1]
            pred_p    = node_pred[:, 2]
            pred_q    = node_pred[:, 3]

            for i, bus in enumerate(buses):
                bus_type = "PQ"  # default
                connected_gens = network.generators[network.generators.bus == bus]
                if len(connected_gens) > 0:
                    ctrl = connected_gens.iloc[0]["control"]
                    if ctrl in ("Slack", "PV"):
                        bus_type = ctrl

                if bus_type == "PQ":
                    # For PQ buses: predict vmag, vang
                    pred_vmag_i = pred_vmag[i]
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]   # Known
                    pred_q_i    = network.buses_t.q.loc[t, bus]   # Known
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = False
                    prediction_masks["V"].loc[t, bus]     = True
                    prediction_masks["Angle"].loc[t, bus] = True

                elif bus_type == "PV":
                    # For PV buses: known vmag, predict vang, q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]         # Known
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = True

                else:
                    # Slack (single or distributed): known vmag, vang; predict p, q
                    # DISTSLACK: for distributed slack, vmag is known but angle
                    # reference is only soft-fixed; we still use the known angle
                    # from the PF solution as input and predict P, Q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = network.buses_t.v_ang.loc[t, bus]     # Known (reference)
                    pred_p_i    = pred_p[i]
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = True
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = False

                bus_results.loc[t, (bus, "P pu")]      = pred_p_i.item() if torch.is_tensor(pred_p_i) else pred_p_i
                bus_results.loc[t, (bus, "Q pu")]      = pred_q_i.item() if torch.is_tensor(pred_q_i) else pred_q_i
                bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i.item() if torch.is_tensor(pred_vmag_i) else pred_vmag_i
                bus_results.loc[t, (bus, "Angle deg")] = np.degrees(
                    pred_vang_i.item() if torch.is_tensor(pred_vang_i) else pred_vang_i
                )

            if debug:
                print(f"Predicted angles should be small radians: {pred_vang[:3].numpy()}")
                print(f"Predicted angles in degrees: {np.degrees(pred_vang[:3].numpy())}")
                print(f"Angle range: {pred_vang.min():.6f} to {pred_vang.max():.6f} radians")

            # Calculate line flows from predicted voltages
            vmag_np = np.array([
                bus_results.loc[t, (bus, "V pu")] for bus in buses
            ], dtype=float)
            vang_np = np.radians(np.array([
                bus_results.loc[t, (bus, "Angle deg")] for bus in buses
            ], dtype=float))
            pred_line_flows = calculate_line_flows(network, vmag_np, vang_np, t_idx)

            for i, line in enumerate(network.lines.index):
                line_results.loc[t, (line, "P0 pu")]   = pred_line_flows["p0"][i]
                line_results.loc[t, (line, "P1 pu")]   = pred_line_flows["p1"][i]
                line_results.loc[t, (line, "Q0 pu")] = pred_line_flows["q0"][i]
                line_results.loc[t, (line, "Q1 pu")] = pred_line_flows["q1"][i]

    return bus_results, line_results, prediction_masks



## Hyperparameter Sweep Helpers

In [ ]:
def evaluate_gnn_on_test_set(
    model,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnom_share: bool = None,  # None = auto-detect from model.node_embedding.in_features
    device=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None,
    max_buses: int = 0,
    ptdf_offset: int = 0,
    # [STSI260502]: batch_size for batched timing measurement
    batch_size: int = 64,
    angle_mode: str = None,  # [STSI 250526]:None=auto-detect from model
) -> dict:
    """
    Evaluate GNN on test networks snapshot-by-snapshot.

    KEY FIX (2026-03-25): After inference, override known variables with ground
    truth inputs — otherwise scatter plots show model predictions for variables
    the model was never asked to predict (Vmag/Vang at slack/PV), making results
    appear much worse than they are.

    Override rules (mirror of _masked_mse_loss):
        PV    → vmag_eval[pv]    = x[pv,    5]  (known Vmag)
        Slack → vmag_eval[slack] = x[slack,  5]  (known Vmag = v_set)
        Slack → vang_eval[slack] = x[slack,  6]  (known Vang = 0)

    STSI 26.04.07: When ptdf_params is not None (learned_edge mode), augments
    data.edge_attr with learned PTDF rows before model forward pass so the
    model receives the same edge feature width it was trained with.

    [STSI260502]: Added multi-level timing:
      solve_time_ms:       pure forward pass per snapshot
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
      batch_solve_time_ms: batched forward pass amortized per snapshot
    Dataset construction and data.to(device) excluded from all timers.

    Returns dict with MAE/RMSE for Vmag, Vang, P, Q + line flow MAE + timing.
    """
    if device is None:
        device = next(model.parameters()).device

    # [STSI 250526]:auto-detect angle_mode from model attribute
    if angle_mode is None:
        angle_mode = getattr(model, 'angle_mode', 'node')

    model.eval()
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
        angle_mode=angle_mode,  # [STSI 250526]: pass angle_mode to dataset for consistency with training and to ensure correct handling of edge_delta mode
        ptdf_branch_mode="all" if angle_mode in ("edge_delta", "both") else "lines",  # [STSI 260526]: edge_delta needs all branches
    )

    vmag_errors, vang_errors, p_errors, q_errors = [], [], [], []
    vmag_true_all, vang_true_all, p_true_all, q_true_all = [], [], [], []

    lflow_errors = []
    all_delta_theta_pred = []  # [STSI 260526]: Δθ metrics accumulator 
    lflow_q0_errors = []
    all_true_flows, all_pred_flows = [], []
    all_true_q0_flows, all_pred_q0_flows = [], []
    n_snapshots = 0

    # [STSI260502]: per-snapshot timing lists (solve and postproc measured separately)
    solve_times = []
    postproc_times = []
    theta_recon_times = []  # [STSI 250526]: θ reconstruction timing for edge_delta mode
    # [STSI260502]: collect prepared data for batched timing later
    all_data_for_batch = []

    with torch.no_grad():
        delta_theta_pred_np = None  # [STSI 260526]: initialized for line flow dispatch
        for i in range(len(dataset)):
            data = dataset[i].to(device)

            # STSI 26.04.07: augment edge_attr for learned_edge models
            net_idx, _ = dataset._index[i]
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset, device)

            # [STSI260502]: save a copy for batched timing (before forward modifies anything)
            all_data_for_batch.append(data.clone())

            # [STSI260502]: Level 1 — time pure forward pass only
            t_solve_start = time.perf_counter()
            node_pred, h_nodes_eval, h_edges_eval, delta_theta_pred = model(data, return_embeddings=True)  # [STSI 250526]:4-tuple
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t_solve_start

            # [STSI 250526]:θ reconstruction for edge_delta mode
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                t_recon_start = time.perf_counter()
                theta_recon = reconstruct_theta_from_delta(
                    delta_theta_pred,
                    data.bfs_edge_idx.to(device),
                    data.bfs_signs.to(device),
                    data.bfs_node_order.to(device),
                    data.bfs_parent.to(device),
                    data.x.size(0),
                )
                theta_recon_times.append((time.perf_counter() - t_recon_start) * 1000)
                # Assemble [N, 4] = [Vmag, θ, P, Q] from [N, 3] + reconstructed θ
                node_pred_full = torch.zeros(data.x.size(0), 4, device=device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = theta_recon       # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P (col1 in 3-col)
                node_pred_full[:, 3] = node_pred[:, 2]  # Q (col2 in 3-col)
                node_pred = node_pred_full
                # [STSI 260526]: expose delta_theta_pred as numpy for line flow + metrics
                delta_theta_pred_np = delta_theta_pred.detach().cpu().numpy()
            elif angle_mode == "both" and delta_theta_pred is not None:  # [STSI 020626]: no reconstruction needed
                delta_theta_pred_np = delta_theta_pred.detach().cpu().numpy()


            # [STSI260502]: Level 2 — time post-processing (masks + line flows)
            t_post_start = time.perf_counter()

            # ── Extract masks ─────────────────────────────────────────────────
            slack_mask = data.slack_mask.bool()
            pv_mask    = data.pv_mask.bool()
            pq_mask    = data.pq_mask.bool()

            # ── Raw predictions ───────────────────────────────────────────────
            vmag_eval = node_pred[:, 0].clone()
            vang_eval = node_pred[:, 1].clone()
            p_eval    = node_pred[:, 2].clone()
            q_eval    = node_pred[:, 3].clone()

            # ── Override known variables with ground-truth inputs ─────────────
            # PV: Vmag is known (set by voltage regulator)
            if pv_mask.any():
                vmag_eval[pv_mask] = data.x[pv_mask, 5]

            # Slack: both Vmag and Vang=0 are known
            if slack_mask.any():
                vmag_eval[slack_mask] = data.x[slack_mask, 5]
                vang_eval[slack_mask] = data.x[slack_mask, 6]  # should be ~0

            # PQ: P and Q are known inputs
            if pq_mask.any():
                p_eval[pq_mask] = data.x[pq_mask, 3]
                q_eval[pq_mask] = data.x[pq_mask, 4]

            # PV: P is known input
            if pv_mask.any():
                p_eval[pv_mask] = data.x[pv_mask, 3]

            # ── Ground truth ──────────────────────────────────────────────────
            vmag_true = data.y[:, 0]
            vang_true = data.y[:, 1]
            p_true    = data.y[:, 2]
            q_true    = data.y[:, 3]

            vmag_errors.append((vmag_eval - vmag_true).abs().mean().item())
            vang_errors.append((vang_eval - vang_true).abs().mean().item())
            p_errors.append((p_eval - p_true).abs().mean().item())
            q_errors.append((q_eval - q_true).abs().mean().item())
            vmag_true_all.append(vmag_true.detach())
            vang_true_all.append(vang_true.detach())
            p_true_all.append(p_true.detach())
            q_true_all.append(q_true.detach())
            n_snapshots += 1

            # ── Step 4 (partial): line flow MAE ──────────────────────────────
            if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                try:
                    net_idx, t_idx = dataset._index[i]
                    network  = test_networks[net_idx]
                    snapshot = network.snapshots[t_idx]
                    if not network.lines_t.p0.empty:
                        true_flows = network.lines_t.p0.loc[snapshot].values
                        vmag_np = vmag_eval.cpu().numpy()
                        vang_np = vang_eval.cpu().numpy()
                        # [STSI 260526]: direct line flows from Δθ when available and in edge_delta mode
                        if angle_mode == "edge_delta" and hasattr(data, 'y_delta_theta'):  # [STSI 260526]:use loop var `data` not dataset[0] 
                            # Use Δθ predictions directly for line flow computation
                            _data_cpu = data.cpu()  # [STSI 260526]:ensure CPU for .numpy() calls
                            _all_fwd_mask_np = np.zeros(_data_cpu.edge_index.shape[1], dtype=bool)
                            _all_fwd_mask_np[::2] = True
                            _ei_fwd_np = _data_cpu.edge_index[:, _all_fwd_mask_np].numpy()
                            _ea_fwd_np = _data_cpu.edge_attr[_all_fwd_mask_np].numpy()
                            # delta_theta_pred for this snapshot
                            _dt_pred_np = delta_theta_pred_np if delta_theta_pred_np is not None else vang_np[_ei_fwd_np[0]] - vang_np[_ei_fwd_np[1]]
                            flows = calculate_line_flows_from_delta_theta(
                                delta_theta=_dt_pred_np,
                                vmag=vmag_np,
                                edge_index_fwd=_ei_fwd_np,
                                edge_attr_fwd=_ea_fwd_np,
                            )
                        else:
                            flows = calculate_line_flows(network, vmag_np, vang_np)
                        n_lines = len(network.lines)
                        pred_flows = np.array(flows["p0"])[:n_lines]  # lines only (excl transformers)
                        err = np.abs(pred_flows - true_flows).mean()
                        # [STSI 260526]: expose Δθ arrays in metrics when in edge_delta mode for deeper analysis
                        if angle_mode in ("edge_delta", "both") and delta_theta_pred_np is not None:
                            all_delta_theta_pred.append(delta_theta_pred_np)
                        lflow_errors.append(err)
                        all_true_flows.append(true_flows)
                        all_pred_flows.append(pred_flows)
                    if not network.lines_t.q0.empty:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])[:n_lines]  # lines only
                        lflow_q0_errors.append(np.abs(pred_q0 - true_q0).mean())
                        all_true_q0_flows.append(true_q0)
                        all_pred_q0_flows.append(pred_q0)
                except Exception as _lfe:
                    if not lflow_errors:  # only warn once
                        print(f"[line_flow_mae] skipped: {_lfe}")

            # [STSI260502]: end post-processing timer (includes masks + line flows)
            t_post = time.perf_counter() - t_post_start
            solve_times.append(t_solve)
            postproc_times.append(t_post)

    # [STSI260502]: Phase 2 — batched timing measurement
    # Warmup batched forward
    n_batch_warmup = 3
    with torch.no_grad():
        if all_data_for_batch:
            for _ in range(n_batch_warmup):
                _ = batched_gnn_forward(model, all_data_for_batch[:batch_size])
            
            batch_times = []
            batch_counts = []
            for start in range(0, len(all_data_for_batch), batch_size):
                chunk = all_data_for_batch[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))
            batch_ms_per_snap = sum(batch_times) * 1000 / max(1, sum(batch_counts))
        else:
            batch_ms_per_snap = float('nan')

    metrics = {
        "vmag_mae":      float(np.mean(vmag_errors)),
        "vang_mae":      float(np.mean(vang_errors)),
        "p_mae":         float(np.mean(p_errors)),
        "q_mae":         float(np.mean(q_errors)),
        "vmag_rmse":     float(np.sqrt(np.mean([e**2 for e in vmag_errors]))),
        "vang_rmse":     float(np.sqrt(np.mean([e**2 for e in vang_errors]))),
        "vmag_true_std": float(torch.cat(vmag_true_all).std().item()),
        "vang_true_std": float(torch.cat(vang_true_all).std().item()),
        "p_true_std":    float(torch.cat(p_true_all).std().item()),
        "q_true_std":    float(torch.cat(q_true_all).std().item()),
    }
    if lflow_errors:
        metrics["line_flow_mae"] = float(np.mean(lflow_errors))
    if lflow_q0_errors:
        metrics["line_flow_q0_mae"] = float(np.mean(lflow_q0_errors))

    if all_true_flows:
        raw = {
            "p0_true": np.concatenate(all_true_flows),
            "p0_pred": np.concatenate(all_pred_flows),
        }
        if all_true_q0_flows:
            raw["q0_true"] = np.concatenate(all_true_q0_flows)
            raw["q0_pred"] = np.concatenate(all_pred_q0_flows)
        metrics["line_flow_raw"] = raw
        
    print("----------------\n Evaluation Metrics (known variables overridden) ---------------")
    for k, v in metrics.items():
        if isinstance(v, (int, float)):
            print(f"  {k:20s}: {v:.6f}")

    # [STSI260502]: monolithic timing kept for backward compat
    inference_time_s = sum(solve_times) + sum(postproc_times)
    metrics["inference_time_s"] = inference_time_s
    metrics["inference_time_per_snapshot_ms"] = (
        1000.0 * inference_time_s / max(1, n_snapshots)
    )
    
    # [STSI 250526]:θ reconstruction timing
    if theta_recon_times:
        metrics["theta_reconstruction_time_ms"] = float(np.mean(theta_recon_times))

    # [STSI260502]: new multi-level timing metrics
    solve_ms    = float(np.mean(solve_times)) * 1000
    postproc_ms = float(np.mean(postproc_times)) * 1000
    metrics["solve_time_ms"]       = solve_ms
    metrics["postproc_time_ms"]    = postproc_ms
    metrics["total_time_ms"]       = solve_ms + postproc_ms
    metrics["batch_solve_time_ms"] = batch_ms_per_snap

    print(f"\n  --- Timing breakdown (per snapshot) ---")
    print(f"  {'solve_time_ms':20s}: {solve_ms:.3f}")
    print(f"  {'postproc_time_ms':20s}: {postproc_ms:.3f}")
    print(f"  {'total_time_ms':20s}: {solve_ms + postproc_ms:.3f}")
    print(f"  {'batch_solve_time_ms':20s}: {batch_ms_per_snap:.3f}")

    # [STSI 260526]: expose Δθ predictions for analysis
    if all_delta_theta_pred:
        metrics["delta_theta_pred_all"] = np.concatenate(all_delta_theta_pred)

    return metrics




def load_sweep_results(pkl_path: str) -> list:
    """ed
    Load a full sweep result list from a pickle file.
    Returns the runs list as saved by save_sweep_results or run_hparam_sweep.
    """
    with open(pkl_path, "rb") as f:
        runs = pickle.load(f)
    print(f"Loaded {len(runs)} runs from {pkl_path}")
    return runs




In [ ]:
# =============================================================================
# SECTION 11: HYPERPARAMETER SWEEP
# =============================================================================

def run_hparam_sweep(
    networks,
    physics_configs: List[PhysicsConfig] = None,
    batch_sizes=(32,),
    learning_rates=(1e-3,),
    num_epochs=50,
    hidden_dims=(64,),
    num_layers_list=(3,),
    conv_types=("gatv2", "transformer"),  # [STSI120526] Added transformer to default sweep
    head_modes=("standard",), # or "with encoder"
    use_residuals=(False,),         # [STSI 120526]:independent conv block flag tuples
    norm_types=(None,),             # None | "layer" | "graph"
    activations=("leaky_relu",),    # "leaky_relu" | "gelu" | "relu" | "elu" | "silu"
    use_dropouts=(False,),
    drop_rates=(0.1,),
    use_pnom_share: bool = False,
    angle_modes: tuple = ("node",),  # [STSI 030626]: "node" | "edge_delta" | "both"
    vmag_modes: tuple = ("absolute",),  # [STSI 030626]: "absolute" | "residual"
    use_global_pools: tuple = (False,),   # [STSI 290526] global graph context
    seeds=(42,),
    warmup_epochs_ptdf_list=(10,),
    warmup_epochs_phys_list=(10,),
    warmup_epochs_flow_list: tuple = ((0, 0, 0),),  # [STSI 030626]:list of (flow, flow_global, flow_ac) triplets
    y_matrix_sources=("manual",),
    tag="run",
    weight_ptdf=0.0,        # added for v2.1.2 compatibility; now should be set via PhysicsConfig
    ptdf_loss_mode="matrix",# added for v2.1.2 compatibility; now should be set via PhysicsConfig
    ptdf_alpha=0.5,# added for v2.1.2 compatibility; now should be set via PhysicsConfig
    verbose=False,
    save_path=None,
):
    """
    Run a hyperparameter sweep over PhysicsConfig and model/training settings.

    Resume / checkpoint behavior
    ----------------------------
    If save_path is provided and exists, previously completed runs are loaded and
    skipped using a stable run_key. After each successful run, the full runs list
    is checkpointed back to save_path.

    Also stores training_time per run and uses previously completed runtimes
    (including resumed runs) to estimate remaining time and finish clock time.
    """
    import itertools
    import os
    import pickle
    import sys
    import time
    from datetime import datetime, timedelta

    if physics_configs is None:
        physics_configs = [PhysicsConfig(w_phys=0.0)]

    combos = list(itertools.product(
        physics_configs,
        batch_sizes,
        learning_rates,
        hidden_dims,
        num_layers_list,
        conv_types,
        y_matrix_sources,
        warmup_epochs_ptdf_list,
        warmup_epochs_phys_list,
        seeds,
        head_modes,
        use_residuals,
        norm_types,
        activations,
        use_dropouts,
        drop_rates,
        use_global_pools,
        angle_modes,
        vmag_modes,
        warmup_epochs_flow_list,
    ))

    total_runs = len(combos)
    runs = []
    done_keys = set()
    runtimes = []

    sweep_start_time = time.time()

    # ------------------------------------------------------------------
    # Resume from checkpoint if available
    # ------------------------------------------------------------------
    if save_path and os.path.isfile(save_path):
        with open(save_path, "rb") as f:
            runs = pickle.load(f)

        backfilled = 0
        restored_runtimes = 0

        for r in runs:
            if not r.get("run_key"):
                physics_label = r.get("physics_label", "baseline")
                bs = r.get("batch_size")
                lr = r.get("lr")
                hidden_dim = r.get("hidden_dim")
                num_layers = r.get("num_layers")
                conv_type = r.get("conv_type")
                ysrc = r.get("y_matrix_source")
                warmup_epochs_ptdf = r.get("warmup_epochs_ptdf", 0)# returns 0 if key not present, which is correct default for old runs without this key
                warmup_epochs_phys = r.get("warmup_epochs_phys", 0)# returns 0 if key not present, which is correct default for old runs without this key
                seed = r.get("seed", 42)
                head_modes = r.get("head_mode", "standard")

                # [STSI 120526]:Map legacy conv_mode to new flags for run_key
                _legacy_map = {
                    "old":            (False, None,    "leaky_relu", False),
                    "residual":       (True,  None,    "gelu",       False),
                    "res_norm":       (True,  "layer", "gelu",       False),
                    "res_norm_drop":  (True,  "layer", "gelu",       True),
                }
                _cm = r.get("conv_mode", "old")
                _ur, _nt, _act, _ud = _legacy_map.get(_cm, (False, None, "leaky_relu", False))
                _dr = r.get("drop_rate", 0.1)
                _cb = []
                if _ur: _cb.append("res")
                if _nt is not None: _cb.append(f"n{_nt[:3]}")
                if _act != "leaky_relu": _cb.append(f"a{_act[:4]}")
                if _ud: _cb.append(f"do{_dr}")
                _cbk = "_".join(_cb) if _cb else "plain"

                r["run_key"] = (
                    f"{physics_label}_"
                    f"bs{bs}_"
                    f"lr{lr:.2e}_"
                    f"h{hidden_dim}_"
                    f"L{num_layers}_"
                    f"{conv_type}_"
                    f"{ysrc}_"
                    f"wu{warmup_epochs}_"
                    f"s{seed}_"
                    f"hm{r.get('head_mode', 'standard')}_"
                    f"pn{int(r.get('use_pnom_share', False))}_"
                    f"cb-{_cbk}"
                )
                backfilled += 1

            done_keys.add(r["run_key"])

            t = r.get("training_time", None)
            if isinstance(t, (int, float)) and t > 0:
                runtimes.append(float(t))
                restored_runtimes += 1

        print(
            f"[checkpoint] Loaded {len(runs)} completed runs; "
            f"{len(done_keys)} keys will be skipped."
        )
        if backfilled:
            print(f"[checkpoint] Added run_key to {backfilled} legacy runs.")
        if restored_runtimes:
            avg_loaded = sum(runtimes) / len(runtimes)
            print(
                f"[checkpoint] Restored {restored_runtimes} prior training_time values "
                f"(avg {avg_loaded/60:.1f} min/run)."
            )

    def fmt_seconds(seconds):
        if seconds is None:
            return "calculating..."
        seconds = int(max(0, seconds))
        m, s = divmod(seconds, 60)
        h, m = divmod(m, 60)
        if h > 0:
            return f"{h}h {m}m {s}s"
        elif m > 0:
            return f"{m}m {s}s"
        else:
            return f"{s}s"

    def fmt_clock_time(unix_ts):
        return datetime.fromtimestamp(unix_ts).strftime("%Y-%m-%d %H:%M:%S")

    for current_run, (
        pcfg,
        bs,
        lr,
        hidden_dim,
        num_layers,
        conv_type,
        ysrc,
        warmup_epochs_ptdf,
        warmup_epochs_phys,
        seed,
        head_mode,
        use_residual,
        norm_type,
        activation,
        use_dropout,
        drop_rate,
        use_global_pool,
        angle_mode,
        vmag_mode,
        _flow_warmup_triplet,
    ) in enumerate(combos, 1):

        warmup_epochs_flow, warmup_epochs_flow_global, warmup_epochs_flow_ac = _flow_warmup_triplet

        key = make_run_key(
            pcfg=pcfg,
            bs=bs,
            lr=lr,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            conv_type=conv_type,
            ysrc=ysrc,
            warmup_epochs_ptdf=warmup_epochs_ptdf,
            warmup_epochs_phys=warmup_epochs_phys,
            seed=seed,
            head_mode=head_mode,
            use_pnom_share=use_pnom_share,
            use_residual=use_residual,
            norm_type=norm_type,
            activation=activation,
            use_dropout=use_dropout,
            drop_rate=drop_rate,
            angle_mode=angle_mode,  # [STSI 250526]: added angle_mode to pass through to training for consistency and to enable edge_delta mode in sweep
            vmag_mode=vmag_mode,    # [STSI 280526]: residual V_mag prediction mode
            use_global_pool=use_global_pool,  # [STSI 290526]
            warmup_epochs_flow=warmup_epochs_flow,  # [STSI 020626]:Task 045
            warmup_epochs_flow_global=warmup_epochs_flow_global,  # [STSI 020626]:Fix
            warmup_epochs_flow_ac=warmup_epochs_flow_ac,          # [STSI 020626]:Fix
        )

        if key in done_keys:
            print(f"[skip {current_run}/{total_runs}] already done: {key}")
            continue

        run_start = time.time()

        if runtimes:
            avg_time = sum(runtimes) / len(runtimes)
            median_time = sorted(runtimes)[len(runtimes) // 2]
            remaining_runs_including_this = total_runs - current_run + 1
            eta_seconds = median_time * remaining_runs_including_this
            estimated_total_seconds = median_time * total_runs
            estimated_finish_ts = time.time() + eta_seconds
        else:
            avg_time = None
            median_time = None
            eta_seconds = None
            estimated_total_seconds = None
            estimated_finish_ts = None

        mode = getattr(pcfg, "physics_mode", "rich")
        parts = [
            f"{current_run}/{total_runs}",
            f"mode={mode}",
            f"cfg={pcfg.label()}",
            f"wphys={pcfg.w_phys}",
            f"losswm={pcfg.loss_weight_mode}",
            f"ptdfw={pcfg.weight_ptdf}",
            f"ptdfmode={pcfg.ptdf_loss_mode}",
            f"ptdfbr={pcfg.ptdf_branch_mode}",
            f"ptdf_cp={pcfg.ptdf_changepoint}",
            f"fpt={pcfg.fraction_physics:.2f}",
            f"fptdf={pcfg.fraction_ptdf:.2f}",
            f"bs={bs}",
            f"lr={lr:.0e}",
            f"conv={conv_type}",
            f"{'res' if use_residual else ''}{'|n=' + norm_type if norm_type else ''}{'|do' + str(drop_rate) if use_dropout else ''}" if (use_residual or norm_type or use_dropout) else "cb=plain",
            f"gpool={'ON' if use_global_pool else 'off'}",
            f"warmup_ptdf={warmup_epochs_ptdf}",
            f"warmup_phys={warmup_epochs_phys}",
            f"warmup_flow={warmup_epochs_flow}|g{warmup_epochs_flow_global}|a{warmup_epochs_flow_ac}",  # [STSI 020626]:Fix — show flow warmups
            f"am={angle_mode}",   # [STSI 020626]:Fix — show angle_mode
            f"vm={vmag_mode}",    # [STSI 020626]:Fix — show vmag_mode
            f"seed={seed}",
        ]
        parts.append(f"Starting time={fmt_clock_time(time.time())}")
        if eta_seconds is not None:
            parts.append(f"remaining={fmt_seconds(eta_seconds)}")
            parts.append(f"est_total={fmt_seconds(estimated_total_seconds)}")
            parts.append(f"finish≈{fmt_clock_time(estimated_finish_ts)}")
        else:
            parts.append("remaining=calculating...")
            parts.append("finish≈calculating...")

        print(" | ".join(parts))

        if not verbose:
            old_stdout, old_stderr = sys.stdout, sys.stderr
            sys.stdout = open(os.devnull, "w")
            sys.stderr = open(os.devnull, "w")

        try:
            model, history, test_metrics, debug_objects = train_power_flow_gnn(
                networks=networks,
                num_epochs=num_epochs,
                batch_size=bs,
                lr=lr,
                physics_cfg=pcfg,
                conv_type=conv_type,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                use_edge_features=True,
                seed=seed,
                warmup_epochs_ptdf=warmup_epochs_ptdf,
                warmup_epochs_phys=warmup_epochs_phys,
                y_matrix_source=ysrc,
                weight_ptdf=weight_ptdf,          # ← scalar from your grid added for testing only
                ptdf_loss_mode=ptdf_loss_mode,    # ← e.g. "matrix"
                ptdf_alpha=ptdf_alpha,            # ← used only for "mixed"
                tqdm_file=old_stderr if not verbose else None,
                return_debug_objects=True,
                head_mode=head_mode, # [STSI060526] Added to toggle between standard and multi-head output modes for node types
                use_pnom_share=use_pnom_share,
                use_residual=use_residual,
                norm_type=norm_type,
                activation=activation,
                use_dropout=use_dropout,
                drop_rate=drop_rate,
                angle_mode=angle_mode,  # [STSI 250526]: added angle_mode to pass through to training for consistency and to enable edge_delta mode in sweep
                vmag_mode=vmag_mode,    # [STSI 280526]: residual V_mag prediction mode
                use_global_pool=use_global_pool,  # [STSI 290526]
                warmup_epochs_flow=warmup_epochs_flow,  # [STSI 020626]:Task 045
                warmup_epochs_flow_global=warmup_epochs_flow_global,
                warmup_epochs_flow_ac=warmup_epochs_flow_ac,
            )
        finally:
            if not verbose:
                sys.stdout.close()
                sys.stderr.close()
                sys.stdout, sys.stderr = old_stdout, old_stderr

        runtime = time.time() - run_start
        runtimes.append(runtime)

        final_train_loss = (
            history["train_total"][-1]
            if "train_total" in history and len(history["train_total"]) > 0
            else float("nan")
        )
        final_val_loss = (
            history["val_total"][-1]
            if "val_total" in history and len(history["val_total"]) > 0
            else float("nan")
        )

        run_info = {
            "tag": tag,

            # physics / PTDF config
            "physics_mode": pcfg.physics_mode,
            "w_P": pcfg.w_P,           # [STSI 220526] P-residual weight
            "w_Q": pcfg.w_Q,           # [STSI 220526] Q-residual weight
            "physics_label": pcfg.label(),
            "weight_physics": pcfg.w_phys,
            "loss_weight_mode": pcfg.loss_weight_mode,      # [STSI 200526]
            "fraction_physics": pcfg.fraction_physics,      # [STSI 200526]
            "fraction_ptdf": pcfg.fraction_ptdf,            # [STSI 200526]
            # [STSI100526] Removed use_power_balance/use_angle_ref_penalty/use_q_partial_mode (always-on, dead discriminators)
            "use_ptdf_loss_flag": pcfg.use_ptdf_loss,
            "weight_ptdf": pcfg.weight_ptdf,
            "ptdf_loss_mode": pcfg.ptdf_loss_mode,
            "ptdf_alpha": pcfg.ptdf_alpha,
            "ptdf_branch_mode": pcfg.ptdf_branch_mode,
            "ptdf_changepoint": pcfg.ptdf_changepoint,  # [STSI 290526]

            # Flow loss config [STSI 020626]:Task 045
            "use_flow_loss": pcfg.use_flow_loss,
            "flow_loss_mode": pcfg.flow_loss_mode,
            "flow_angle_mode": pcfg.flow_angle_mode,
            "w_flow": pcfg.w_flow,
            "w_flow_local": pcfg.w_flow_local,      # [STSI 020626]:Fix — store sub-weights
            "w_flow_global": pcfg.w_flow_global,
            "w_flow_dc": pcfg.w_flow_dc,
            "w_flow_ac": pcfg.w_flow_ac,
            "flow_target": pcfg.flow_target,
            "fraction_flow": pcfg.fraction_flow,
            "max_w_flow": pcfg.max_w_flow,
            "warmup_epochs_flow": warmup_epochs_flow,
            "warmup_epochs_flow_global": warmup_epochs_flow_global,
            "warmup_epochs_flow_ac": warmup_epochs_flow_ac,

            # run hyperparameters
            "batch_size": bs,
            "lr": lr,
            "conv_type": conv_type,
            "hidden_dim": hidden_dim,
            "num_layers": num_layers,
            "warmup_epochs_ptdf": warmup_epochs_ptdf,
            "warmup_epochs_phys": warmup_epochs_phys,
            "seed": seed,
            "y_matrix_source": ysrc,
            "head_mode": head_mode, # [STSI060526] Added to toggle between standard and multi-head output modes for node types
            "use_pnom_share": use_pnom_share,  # [STSI070526] participation-weighted balance as 8th feature
            "angle_mode": angle_mode,  # [STSI 250526]: "node" | "edge_delta" | "both"
            "vmag_mode": vmag_mode,    # [STSI 280526]: "absolute" | "residual"
            "use_global_pool": use_global_pool,  # [STSI 290526]
            "node_feature_mode": "pnom_weighted" if use_pnom_share else "base",  # [STSI070526] discriminator for evaluate auto-detect
            "use_residual": use_residual,  # [STSI 120526]:independent conv block flags
            "norm_type": norm_type,
            "activation": activation,
            "use_dropout": use_dropout,
            "drop_rate": drop_rate,

            # outputs
            "history": history,
            "test_metrics": test_metrics,
            "model": model,
            "training_time": runtime,
            "final_train_loss": final_train_loss,
            "final_val_loss": final_val_loss,

            # STSI 26.04.07: learned_edge artefacts for inference
            "_ptdf_params": test_metrics.get("_ptdf_params"),
            "_max_buses":   test_metrics.get("_max_buses", 0),

            # resume / dedup
            "run_key": key,

            # test split for later evaluation
            "test_networks": debug_objects.get("test_networks"),
            "num_scenarios":   len(networks),
            "bus_count_min":   min(len(n.buses) for n in networks),
            "bus_count_max":   max(len(n.buses) for n in networks),
            "bus_count_range": max(len(n.buses) for n in networks) - min(len(n.buses) for n in networks),
        }

        runs.append(run_info)
        done_keys.add(key)

        tm = test_metrics
        print(
            f"→ done in {runtime/60:.1f} min | "
            f"Train {final_train_loss:.4f} "
            f"Val {final_val_loss:.4f} "
            f"Test {tm.get('test_total', float('nan')):.4f} "
        )

        # ------------------------------------------------------------------
        # Checkpoint save after every completed run
        # ------------------------------------------------------------------
        if save_path:
            # Strip unpicklable objects (PyPSA networks contain weakrefs)
            _stashed = []
            for r in runs:
                _stashed.append(r.pop("test_networks", None))
            # Atomic write: dump to .tmp first, then rename so an interrupt
            # never leaves the real checkpoint file truncated/empty
            _tmp_path = save_path + ".tmp"
            with open(_tmp_path, "wb") as f:
                pickle.dump(runs, f)
            os.replace(_tmp_path, save_path)
            for r, tn in zip(runs, _stashed):
                if tn is not None:
                    r["test_networks"] = tn

    return runs

def make_run_key(
    pcfg,
    bs,
    lr,
    hidden_dim,
    num_layers,
    conv_type,
    ysrc,
    warmup_epochs_ptdf,
    warmup_epochs_phys,
    seed,
    head_mode: str = "standard",
    use_pnom_share: bool = False,
    use_residual: bool = False,  # [STSI 120526]:independent conv block flags
    norm_type: str = None,
    activation: str = "leaky_relu",
    use_dropout: bool = False,
    drop_rate: float = 0.1,
    angle_mode: str = "node",  # [STSI 250526]:added angle_mode to run_key for consistency and to enable edge_delta mode in sweep
    vmag_mode: str = "absolute",  # [STSI 280526]: include in key for deduplication
    use_global_pool: bool = False,  # [STSI 290526]
    warmup_epochs_flow: int = 0,
    warmup_epochs_flow_global: int = 0,
    warmup_epochs_flow_ac: int = 0,
) -> str:
    """
    Stable string key identifying one combo used for checkpoint deduplication
    in run_hparam_sweep.
    """
    # Compact conv-block encoding: only include non-default values [STSI 120526]:new run_key format
    cb_parts = []
    if use_residual: cb_parts.append("res")
    if norm_type is not None: cb_parts.append(f"n{norm_type[:3]}")
    if activation != "leaky_relu": cb_parts.append(f"a{activation[:4]}")
    if use_dropout: cb_parts.append(f"do{drop_rate}")
    _conv_block_key = "_".join(cb_parts) if cb_parts else "plain"

    # [STSI 250526]:include angle_mode in key only if non-default
    _am_part = f"_am{angle_mode}" if angle_mode != "node" else ""
    _vm_part = f"_vm{vmag_mode}" if vmag_mode != "absolute" else ""  # [STSI 280526]
    _gp_part = f"_gp" if use_global_pool else ""  # [STSI 290526]
    _fl_part = (f"_fl{warmup_epochs_flow}g{warmup_epochs_flow_global}a{warmup_epochs_flow_ac}"  # [STSI 020626]:Fix — all 3 flow warmups in key
              if (warmup_epochs_flow > 0 or warmup_epochs_flow_global > 0 or warmup_epochs_flow_ac > 0) else "")
    return (
        f"{pcfg.label()}_"
        f"bs{bs}_"
        f"lr{lr:.2e}_"
        f"h{hidden_dim}_"
        f"L{num_layers}_"
        f"{conv_type}_"
        f"{ysrc}_"
        f"wu_ptdf{warmup_epochs_ptdf}_"
        f"wu_phys{warmup_epochs_phys}_"
        f"s{seed}_"
        f"hm{head_mode}_"
        f"pn{int(use_pnom_share)}_"
        f"cb-{_conv_block_key}"
        f"{_am_part}"
        f"{_vm_part}"
        f"{_gp_part}"
        f"{_fl_part}"
    )

def create_comparison_dataframe(runs_list):
    """Create a comprehensive comparison DataFrame from sweep results."""
    rows = []
    for r in runs_list:
        tm = r["test_metrics"]
        rows.append({
            "tag":           r["tag"],
            "w_phys":        r["weight_physics"],
            "w_ptdf":        r["weight_ptdf"],
            "bs":            r["batch_size"],
            "lr":            r["lr"],
            "train_loss":    r["final_train_loss"],
            "val_loss":      r["final_val_loss"],
            "test_total":    tm["test_total"],
            "test_mse":      tm["test_mse"],
            "test_physics":  tm["test_physics"],
            "test_ptdf":     tm["test_ptdf"],
            "training_time": r["training_time"],
            # [STSI100526] New mode columns for comparison
            "head_mode":     r.get("head_mode", "standard"),
            "use_residual":  r.get("use_residual", False),  # [STSI 120526]:independent flags
            "norm_type":     r.get("norm_type", None),
            "activation":    r.get("activation", "leaky_relu"),
            "use_dropout":   r.get("use_dropout", False),
            "drop_rate":     r.get("drop_rate", 0.1),
            "physics_label": r.get("physics_label", ""),
            "warmup_epochs_ptdf": r.get("warmup_epochs_ptdf", 0),
            "warmup_epochs_phys": r.get("warmup_epochs_phys", 0),
        })
    return pd.DataFrame(rows)

def _get_run_styles(runs):
    # [STSI100526] Removed physics_mode (always-on); added head_mode/conv_mode/drop_rate
    varying = [
        k for k in ("physics_label", "weight_ptdf",
                    "head_mode", "use_residual", "norm_type",
                    "activation", "use_dropout", "drop_rate",
                    "batch_size", "lr", "conv_type", "y_matrix_source")
        if len({str(x.get(k)) for x in runs}) > 1
    ]
    if not varying:
        varying = ["physics_label"]

    dim_values = {}
    for k in varying:
        seen = {}
        for r in runs:
            v = str(r.get(k, ""))
            if v not in seen:
                seen[v] = len(seen)
        dim_values[k] = seen

    styles = {}
    for r in runs:
        rid = id(r)
        c_idx  = dim_values[varying[0]][str(r.get(varying[0], ""))] if len(varying) >= 1 else 0
        ls_idx = dim_values[varying[1]][str(r.get(varying[1], ""))] if len(varying) >= 2 else 0
        mk_idx = dim_values[varying[2]][str(r.get(varying[2], ""))] if len(varying) >= 3 else 0

        styles[rid] = (
            _COLOR_CYCLE[c_idx  % len(_COLOR_CYCLE)],
            _LINE_STYLES[ls_idx % len(_LINE_STYLES)],
            _MARKERS[mk_idx     % len(_MARKERS)],
        )
    return styles

def build_legend_label(r, runs):
    # [STSI100526] Removed physics_mode (dead); added head_mode/conv_mode/drop_rate
    varying = [
        k for k in [
            "physics_label",
            "weight_ptdf",
            "ptdf_loss_mode",
            "ptdf_branch_mode",
            "ptdf_alpha",
            "ptdf_changepoint",
            "head_mode",
            "use_residual",
            "norm_type",
            "activation",
            "use_dropout",
            "drop_rate",
            "batch_size",
            "lr",
            "conv_type",
            "y_matrix_source",
            "warmup_epochs_ptdf",
            "warmup_epochs_phys",
            "hidden_dim",
            "num_layers",
        ]
        if len({str(x.get(k)) for x in runs}) > 1
    ]

    if not varying:
        varying = ["physics_label"]

    parts = []

    # [STSI100526] Removed physics_mode label (always-on)
    if "physics_label" in varying:
        parts.append(r.get("physics_label", ""))

    if "weight_ptdf" in varying:
        parts.append(f"ptdf={r.get('weight_ptdf')}")
    if "ptdf_loss_mode" in varying:
        mode_str = r.get('ptdf_loss_mode')
        cp = r.get('ptdf_changepoint')
        if mode_str == 'stepwise' and cp is not None:
            mode_str = f'stepwise(cp={cp})'
        parts.append(f'mode={mode_str}')
    if "ptdf_changepoint" in varying and "ptdf_loss_mode" not in varying:
        parts.append(f"cp={r.get('ptdf_changepoint')}")
    if "ptdf_branch_mode" in varying:
        parts.append(f"br={r.get('ptdf_branch_mode')}")
    if "ptdf_alpha" in varying and r.get("ptdf_loss_mode") == "mixed":
        parts.append(f"a={r.get('ptdf_alpha')}")

    if "y_matrix_source" in varying:
        parts.append(f"y={r.get('y_matrix_source')}")
    if "warmup_epochs_ptdf" in varying:
        parts.append(f"wu_ptdf={r.get('warmup_epochs_ptdf')}")
    if "warmup_epochs_phys" in varying:
        parts.append(f"wu_phys={r.get('warmup_epochs_phys')}")
    if "hidden_dim" in varying:
        parts.append(f"h={r.get('hidden_dim')}")
    if "num_layers" in varying:
        parts.append(f"L={r.get('num_layers')}")
    if "batch_size" in varying:
        parts.append(f"bs={r.get('batch_size')}")
    if "lr" in varying:
        parts.append(f"lr={r.get('lr'):.0e}")
    if "conv_type" in varying:
        parts.append(r.get("conv_type", ""))
    # [STSI100526] New mode labels for head_mode/conv_mode/drop_rate
    if "head_mode" in varying:
        parts.append(f"hm={r.get('head_mode', 'standard')}")
    # Conv block flags — compact notation [STSI 120526]:independent flags in legend
    conv_parts = []
    if "use_residual" in varying and r.get("use_residual", False):
        conv_parts.append("res")
    if "norm_type" in varying and r.get("norm_type") is not None:
        conv_parts.append(f"{r['norm_type'][:3]}N")
    if "activation" in varying and r.get("activation", "leaky_relu") != "leaky_relu":
        conv_parts.append(r["activation"][:4])
    if "use_dropout" in varying and r.get("use_dropout", False):
        conv_parts.append(f"do{r.get('drop_rate', 0.1)}")
    if conv_parts:
        parts.append("+".join(conv_parts))
    elif "drop_rate" in varying:
        parts.append(f"dr={r.get('drop_rate', 0.1)}")

    return " | ".join(parts)


def plot_all_runs_training_curves(runs, title_prefix="All runs", marker_every=10):
    """
    Plot train/val loss for all runs.
    - Color    separates the first varying hyperparameter (e.g. physics_label).
    - Linestyle separates the second varying hyperparameter (e.g. weight_ptdf).
    - Marker   separates the third varying hyperparameter (e.g. batch_size / lr).
    - marker_every: place a marker every N epochs (keeps plots readable).
    """
    styles = _get_run_styles(runs)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

    for r in runs:
        label = build_legend_label(r, runs)
        h     = r["history"]
        color, ls, mk = styles[id(r)]
        n = len(h["train_total"])
        markevery = max(1, n // marker_every) if mk else None

        ax1.plot(h["train_total"], label=label, alpha=0.85,
                 color=color, linestyle=ls, marker=mk,
                 markevery=markevery, markersize=5)
        ax2.plot(h["val_total"],   label=label, alpha=0.85,
                 color=color, linestyle=ls, marker=mk,
                 markevery=markevery, markersize=5)

    for ax, title, ylabel in [
        (ax1, f"{title_prefix} — Training Loss",   "Training Loss"),
        (ax2, f"{title_prefix} — Validation Loss", "Validation Loss"),
    ]:
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8, ncol=max(1, len(runs) // 8))
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_loss_components(runs, title_prefix="Loss Components", marker_every=10):
    """
    Plot train/val loss for each component (MSE, Physics, Angle-ref, PTDF)
    in a 2×2 grid of subplots, with each subplot showing train (solid) and
    val (dashed) curves for all runs.
    """
    styles = _get_run_styles(runs)

    components = [
        ("mse",       "MSE Loss"),
        ("phys",      "Physics Loss"),
        ("angle_ref", "Angle-Ref Loss"),
        ("ptdf",      "PTDF Loss"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    axes = axes.flatten()

    for ax, (key, comp_title) in zip(axes, components):
        for r in runs:
            label = build_legend_label(r, runs)
            h = r["history"]
            train_key = f"train_{key}"
            val_key   = f"val_{key}"

            if train_key not in h or not h[train_key]:
                continue

            color, ls, mk = styles[id(r)]
            n = len(h[train_key])
            markevery = max(1, n // marker_every) if mk else None

            ax.plot(h[train_key], label=f"{label} (train)", alpha=0.85,
                    color=color, linestyle="-", marker=mk,
                    markevery=markevery, markersize=5)
            ax.plot(h[val_key],   label=f"{label} (val)", alpha=0.85,
                    color=color, linestyle="--", marker=mk,
                    markevery=markevery, markersize=5)

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"{title_prefix} — {comp_title}")
        ax.legend(fontsize=7, ncol=max(1, len(runs) // 4))
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_all_runs_accuracy_metrics(runs, title_prefix="All runs"):
    """
    Bar chart comparison of test accuracy metrics across runs.
    Reads from r["test_metrics"]["overall_metrics"] for mean/std per metric.
    Metrics: vmag_mae, vang_rmse, p_mae, q_mae.
    """
    metrics = [
        ("vmag_mae",  "Vmag MAE  [p.u.]"),
        ("vang_rmse", "Vang RMSE [rad]"),
        ("p_mae",     "P MAE     [p.u.]"),
        ("q_mae",     "Q MAE     [p.u.]"),
        ("line_flow_mae", "Line Flow MAE [p.u.]"),
        ("line_flow_q0_mae", "Line Flow Q0 MAE [p.u.]"),
    ]
    x      = list(range(len(runs)))
    labels = [build_legend_label(r, runs) for r in runs]
    styles = _get_run_styles(runs)   # reuse the same color assignment as training curves
    colors = [styles[id(r)][0] for r in runs]

    fig, axes = plt.subplots(1, len(metrics), figsize=(max(16, 2.5 * len(metrics) + len(runs)), 6))
    fig.suptitle(f"{title_prefix} — Test Accuracy Metrics", fontsize=13)
    short_labels = [str(i + 1) for i in range(len(runs))]   # "1", "2", …
    for ax, (metric_key, metric_label) in zip(axes, metrics):
        means, stds = [], []
        for r in runs:
            tm = r.get("test_metrics", {})
            val = tm.get(metric_key, float("nan"))
            # support both scalar and dict-with-mean
            if isinstance(val, dict):
                mean = float(val.get("mean", float("nan")))
                std  = float(val.get("std",  0.0))
            else:
                mean = float(val)
                std  = 0.0
            means.append(mean)
            stds.append(std)

        bars = ax.bar(x, means, yerr=stds, capsize=5, alpha=0.8, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(short_labels, fontsize=9)
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.grid(True, axis="y", alpha=0.3)

        valid_means = [m for m in means if m == m]  # filter nan
        top = max(valid_means) if valid_means else 1.0
        for bar, mean in zip(bars, means):
            if mean == mean:  # not nan
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + top * 0.02,
                    f"{mean:.4f}",
                    ha="center", va="bottom", fontsize=7
                )

    # ── Shared legend below all subplots ─────────────────────────────────────
    legend_lines = [
        plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                   label=f"{i+1}: {labels[i]}")
        for i in range(len(runs))
    ]
    fig.legend(
        handles=legend_lines,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=max(1, min(3, len(runs))),
        fontsize=8,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    plt.show()

def print_sweep_summary(runs_list, top_n=5):
    """
    Print a ranked summary table of hyperparameter sweep results.
    Shows top_n configurations by test loss.
    """
    sorted_runs = sorted(runs_list, key=lambda r: r["test_metrics"]["test_total"])
    # [STSI100526] Added head_mode/conv_mode columns to summary table
    print(f"{'Rank':<5}{'w_phys':<9}{'w_ptdf':<9}{'head':<14}{'res':<6}{'norm':<8}{'act':<12}{'do':<5}"  # [STSI 120526]
          f"{'bs':<6}{'lr':<11}"
          f"{'Train':<11}{'Val':<11}{'Test':<11}{'Time(s)':<9}")
    print("-" * 120)
    for i, run in enumerate(sorted_runs[:top_n], 1):
        tm = run["test_metrics"]
        print(
            f"{i:<5}{run['weight_physics']:<9}{run['weight_ptdf']:<9}"
            f"{run.get('head_mode','standard'):<14}"
            f"{str(run.get('use_residual', False)):<6}{str(run.get('norm_type', None)):<8}{run.get('activation', 'leaky_relu'):<12}{str(run.get('use_dropout', False)):<5}"  # [STSI 120526]
            f"{run['batch_size']:<6}{run['lr']:<11.2e}"
            f"{run['final_train_loss']:<11.6f}{run['final_val_loss']:<11.6f}"
            f"{tm['test_total']:<11.6f}{run['training_time']:<9.1f}"
        )

def _sanitise_filename(s: str) -> str:
    """Replace characters illegal on Windows filesystems with safe alternatives."""
    for ch in r'\/:*?"<>|=':
        s = s.replace(ch, "-")
    return s


def save_sweep_results(
    runs: list,
    save_dir: str = "training_results_saved",
    tag: str = None,
    save_models: bool = True,
    save_csv: bool = True,
    save_histories: bool = True,
):
    """
    Save all results from run_hparam_sweep to disk.
    # SECTION: save_sweep_results — updated for current run_info dict structure
    # Changes from old version:
    #   - weight_physics → w_phys (PhysicsConfig field name)
    #   - Added warmup_epochs, physics_mode, seed to CSV
    #   - Filename sanitisation (removes |, =, illegal Windows chars)
    #   - Flattened overall_metrics access matches evaluate_gnn_on_test_set output
    #   - Model label uses physics_label instead of weight_physics
    # 2026-03-26
    # =============================================================================

    Saves:
      - runs_{tag}_{ts}.pkl            : full run list (models + histories + metrics)
      - runs_{tag}_{ts}_metrics.csv    : flat metrics table
      - runs_{tag}_{ts}_histories/     : per-run training curves as CSV
      - models/runs_{tag}_{ts}/        : per-run model .pt files
      - runs_{tag}_{ts}_manifest.json  : metadata
    """
    os.makedirs(save_dir, exist_ok=True)

    if tag is None:
        tag = runs[0].get("tag", "sweep") if runs else "sweep"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = _sanitise_filename(f"runs_{tag}_{timestamp}")

    # ── 1. Full pickle ────────────────────────────────────────────────────────
    pkl_path = os.path.join(save_dir, f"{base_name}.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(runs, f)
    print(f"  [OK] Full run list  -> {pkl_path}")

    # ── 2. Metrics CSV ────────────────────────────────────────────────────────
    csv_path = None
    if save_csv:
        rows = []
        for i, r in enumerate(runs):
            tm  = r.get("test_metrics", {})
            om  = tm.get("overall_metrics", {})   # from evaluate_gnn_on_test_set

            rows.append({
                # Identifiers
                "run_idx":              i,
                "tag":                  r.get("tag", ""),
                "seed":                 r.get("seed"),
                # Physics config
                "physics_label":        r.get("physics_label"),
                "physics_mode":         r.get("physics_mode"),
                "w_phys":               r.get("weight_physics"),   # stored as weight_physics in run_info
                # [STSI100526] Removed use_power_balance/use_angle_ref/use_q_partial (always-on dead cols)
                "use_ptdf_loss_flag":   r.get("use_ptdf_loss_flag"),
                # Other hparams
                "w_ptdf":               r.get("weight_ptdf"),
                "batch_size":           r.get("batch_size"),
                "lr":                   r.get("lr"),
                "conv_type":            r.get("conv_type"),
                "ptdf_loss_mode":       r.get("ptdf_loss_mode"),
                "hidden_dim":           r.get("hidden_dim"),
                "num_layers":           r.get("num_layers"),
                "warmup_epochs":        r.get("warmup_epochs"),
                "y_matrix_source":      r.get("y_matrix_source"),
                # [STSI100526] New hparam columns for sweep discriminators
                "head_mode":            r.get("head_mode", "standard"),
                "use_residual":         r.get("use_residual", False),  # [STSI 120526]:independent flags
                "norm_type":            r.get("norm_type", None),
                "activation":           r.get("activation", "leaky_relu"),
                "use_dropout":          r.get("use_dropout", False),
                "drop_rate":            r.get("drop_rate", 0.1),
                "use_pnom_share":       r.get("use_pnom_share", False),
                "node_feature_mode":    r.get("node_feature_mode", "base"),
                "angle_mode":           r.get("angle_mode", "node"),  # [STSI 270526]: "node" | "edge_delta" | "both"
                # Training outcomes
                "final_train_loss":     r.get("final_train_loss"),
                "final_val_loss":       r.get("final_val_loss"),
                "training_time_s":      r.get("training_time"),
                # Test losses
                "test_total":           tm.get("test_total"),
                "test_mse":             tm.get("test_mse"),
                "test_physics":         tm.get("test_physics"),
                "test_ptdf":            tm.get("test_ptdf"),
                # Node-wise metrics from evaluate_gnn_on_test_set
                "vmag_mae":             tm.get("vmag_mae"),
                "vang_mae":             tm.get("vang_mae"),
                "p_mae":                tm.get("p_mae"),
                "q_mae":                tm.get("q_mae"),
                "vmag_rmse":            tm.get("vmag_rmse"),
                "vang_rmse":            tm.get("vang_rmse"),
                "line_flow_mae":        tm.get("line_flow_mae"),   # Step 4 — None if not computed
                "inference_time_s":     tm.get("inference_time_s"),
                "infer_ms_per_snap":    tm.get("inference_time_per_snapshot_ms"),
            })

        csv_path = os.path.join(save_dir, f"{base_name}_metrics.csv")
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f"  [OK] Metrics CSV    -> {csv_path}")

    # ── 3. Per-run training histories ─────────────────────────────────────────
    if save_histories:
        hist_dir = os.path.join(save_dir, f"{base_name}_histories")
        os.makedirs(hist_dir, exist_ok=True)
        for i, r in enumerate(runs):
            h = r.get("history", {})
            if not h:
                continue
            hist_label = _sanitise_filename(
                f"run{i:03d}"
                f"_{r.get('physics_label', 'nophys')}"
                f"_pt{r.get('weight_ptdf')}"
                f"_bs{r.get('batch_size')}"
                f"_lr{r.get('lr'):.0e}"
                f"_s{r.get('seed')}"
            )
            pd.DataFrame(h).to_csv(
                os.path.join(hist_dir, f"{hist_label}.csv"),
                index_label="epoch"
            )
        print(f"  [OK] Histories      -> {hist_dir}/  ({len(runs)} files)")

    # ── 4. Model state dicts ──────────────────────────────────────────────────
    if save_models:
        model_dir = os.path.join(save_dir, "models", base_name)
        os.makedirs(model_dir, exist_ok=True)
        for i, r in enumerate(runs):
            model = r.get("model")
            if model is None:
                continue
            model_label = _sanitise_filename(
                f"run{i:03d}"
                f"_{r.get('physics_label', 'nophys')}"
                f"_pt{r.get('weight_ptdf')}"
                f"_bs{r.get('batch_size')}"
                f"_lr{r.get('lr'):.0e}"
                f"_{r.get('conv_type')}"
                f"_{r.get('ptdf_loss_mode')}"
                f"_s{r.get('seed')}"
                f".pt"
            )
            safe_torch_save(
                model.state_dict(),
                os.path.join(model_dir, model_label)
            )
        print(f"  [OK] Model weights  -> {model_dir}/  ({len(runs)} files)")

    # ── 5. Manifest ───────────────────────────────────────────────────────────
    manifest = {
        "timestamp":  timestamp,
        "tag":        tag,
        "num_runs":   len(runs),
        "save_dir":   save_dir,
        "files": {
            "pickle":    os.path.basename(pkl_path),
            "metrics":   os.path.basename(csv_path) if csv_path else None,
            "histories": f"{base_name}_histories/" if save_histories else None,
            "models":    f"models/{base_name}/" if save_models else None,
        },
        "hyperparams": {
            "physics_labels": sorted(set(r.get("physics_label", "") for r in runs)),
            "w_phys":         sorted(set(r.get("weight_physics", 0.0) for r in runs)),
            "w_ptdf":         sorted(set(r.get("weight_ptdf", 0.0) for r in runs)),
            "batch_sizes":    sorted(set(r.get("batch_size") for r in runs)),
            "learning_rates": sorted(set(r.get("lr") for r in runs)),
            "conv_types":     sorted(set(r.get("conv_type", "") for r in runs)),
            "ptdf_modes":     sorted(set(r.get("ptdf_loss_mode", "") for r in runs)),
            "warmup_epochs":  sorted(set(r.get("warmup_epochs", 0) for r in runs)),
            "seeds":          sorted(set(r.get("seed") for r in runs)),
            # [STSI100526] New hparam summaries in manifest
            "head_modes":     sorted(set(r.get("head_mode", "standard") for r in runs)),
            "use_residuals":  sorted(set(str(r.get("use_residual", False)) for r in runs)),  # [STSI 120526]
            "norm_types":     sorted(set(str(r.get("norm_type", None)) for r in runs)),
            "activations":    sorted(set(r.get("activation", "leaky_relu") for r in runs)),
            "use_dropouts":   sorted(set(str(r.get("use_dropout", False)) for r in runs)),
            "drop_rates":     sorted(set(r.get("drop_rate", 0.1) for r in runs)),
        }
    }
    manifest_path = os.path.join(save_dir, f"{base_name}_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    print(f"  [OK] Manifest       -> {manifest_path}")
    print(f"\n  Saved {len(runs)} runs to '{save_dir}/{base_name}*'")




# Training runs

## Load Networks

In [ ]:
networks_mixed_1500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500_wide.json"))

## Sweep Configs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SWEEP CONFIGURATION — Round 2 (2026-05-29)
# ══════════════════════════════════════════════════════════════════════
# Fixed: edge_delta, batch=64, lr=1e-4, hidden=64, layers=3,
#        head_mode=with_encoder, no residual/dropout/norm, 100 epochs.
# Variables: f_physics, f_ptdf, warmup timing, ptdf_mode, vmag_mode.

SWEEP_BASE = dict(
    batch_sizes      = (64,),
    learning_rates   = (1e-4,),
    num_epochs       = 100,
    hidden_dims      = (64,),
    num_layers_list  = (3,),
    conv_types       = ("gatv2",),
    head_modes       = ("with_encoder",),
    seeds            = (42,),
    angle_modes      = ("edge_delta",),
    verbose          = False,
)

SWEEP_SAVE_DIR = TRAINING_RESULTS_DIR

# ── Physics-only configs (no PTDF) ───────────────────────────────────
pc_none    = PhysicsConfig()
pc_phys_02 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.02, max_w_phys=100.0)
pc_phys_05 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0)
pc_phys_10 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.10, max_w_phys=100.0)
pc_phys_15 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.15, max_w_phys=100.0)

# ── PTDF+Physics configs — flows mode ────────────────────────────────
pc_fp05_ft05_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.05,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")
pc_fp05_ft10_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.10,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")
pc_fp05_ft15_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.15,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")

# ── PTDF+Physics configs — stepwise mode (changepoint=5) ─────────────
pc_fp05_ft05_step5 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.05,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True,
                                   ptdf_loss_mode="stepwise", ptdf_changepoint=5)
pc_fp05_ft10_step5 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.10,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True,
                                   ptdf_loss_mode="stepwise", ptdf_changepoint=5)
pc_fp05_ft15_step5 = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, fraction_ptdf=0.15,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True,
                                   ptdf_loss_mode="stepwise", ptdf_changepoint=5)

# ── Interaction grid configs — flows mode ─────────────────────────────
pc_fp02_ft05_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.02, fraction_ptdf=0.05,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")
pc_fp15_ft15_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.15, fraction_ptdf=0.15,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")
pc_fp15_ft05_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.15, fraction_ptdf=0.05,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")
pc_fp02_ft15_flows = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.02, fraction_ptdf=0.15,
                                   max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")

# ── Best config placeholder ──────────────────────────────────────────
pc_best = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.10, fraction_ptdf=0.10,
                        max_w_phys=100.0, max_w_ptdf=100.0, use_ptdf_loss=True, ptdf_loss_mode="flows")

print("Sweep config loaded (Round 2).")
print(f"  Epochs: {SWEEP_BASE['num_epochs']}, batch: {SWEEP_BASE['batch_sizes'][0]}, lr: {SWEEP_BASE['learning_rates'][0]}")
print(f"  Architecture: layers=3, hidden=64, head=with_encoder, no res/drop/norm")
print(f"  Mode: edge_delta")
print(f"  Save dir: {SWEEP_SAVE_DIR}")


In [ ]:
# ── F1: Baselines — edge_delta vs both, residual vs absolute (4 runs) ────────
# Establishes: (a) edge_delta baseline, (b) both-mode baseline, (c) absolute comparison
# angle_modes × vmag_modes gives 2×2 = 4 runs (full grid).
# All use physics but NO flow loss. 100 epochs.

pc_base = PhysicsConfig(loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0)

f1_results = run_hparam_sweep(
    networks=networks_mixed_1500_wide,
    physics_configs=[pc_base],
    angle_modes=("edge_delta", "both"),
    vmag_modes=("residual", "absolute"),
    warmup_epochs_phys_list=(30,), warmup_epochs_ptdf_list=(20,),
    tag="F1_baselines",
    save_path=os.path.join(SWEEP_SAVE_DIR, "sweep_F1_baselines.json"),
    **{k: v for k, v in SWEEP_BASE.items() if k not in ("angle_modes", "vmag_modes")})

print(f"F1 complete: {len(f1_results)} runs")


In [ ]:
# ── F2: DC-local flow loss — early vs late activation (4 runs) ───────────────
# Feature: use_flow_loss + flow_loss_mode="dc" + flow_angle_mode="local"
# Tests warmup_epochs_flow = 20 (early) vs 50 (late), fraction_flow = 0.05 vs 0.10
# All: angle_mode="both", vmag=residual, phys_act=30, ptdf_act=20

pc_flow_dc_local_f05 = PhysicsConfig(
    loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0,
    use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local",
    w_flow=1.0, fraction_flow=0.05, max_w_flow=100.0,
)
pc_flow_dc_local_f10 = PhysicsConfig(
    loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0,
    use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local",
    w_flow=1.0, fraction_flow=0.10, max_w_flow=100.0,
)

f2_results = run_hparam_sweep(
    networks=networks_mixed_1500_wide,
    physics_configs=[pc_flow_dc_local_f05, pc_flow_dc_local_f10],
    angle_modes=("both",), vmag_modes=("residual",),
    warmup_epochs_phys_list=(30,), warmup_epochs_ptdf_list=(20,),
    warmup_epochs_flow_list=((20, 0, 0), (50, 0, 0)),
    tag="F2_dc_local",
    save_path=os.path.join(SWEEP_SAVE_DIR, "sweep_F2_dc_local.json"),
    **{k: v for k, v in SWEEP_BASE.items() if k not in ("angle_modes", "vmag_modes")})

print(f"F2 complete: {len(f2_results)} runs")


In [ ]:
# ── F3: DC-local + DC-global — global activation timing (4 runs) ─────────────
# Feature: flow_angle_mode="both" (local Δθ/x + global (θ_i-θ_j)/x)
# Local activates at warmup_flow, global activates later at warmup_flow_global.
# Tests: local@20 + global@{40, 60}, local@30 + global@{50, 70}
# All: angle_mode="both", vmag=residual, flow_loss_mode="dc"

pc_flow_dc_both = PhysicsConfig(
    loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0,
    use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="both",
    w_flow=1.0, w_flow_local=0.5, w_flow_global=0.5,
    fraction_flow=0.10, max_w_flow=100.0,
)

f3_results = run_hparam_sweep(
    networks=networks_mixed_1500_wide,
    physics_configs=[pc_flow_dc_both],
    angle_modes=("both",), vmag_modes=("residual",),
    warmup_epochs_phys_list=(30,), warmup_epochs_ptdf_list=(20,),
    warmup_epochs_flow_list=((20, 40, 0), (20, 60, 0), (30, 50, 0), (30, 70, 0)),
    tag="F3_dc_local_global",
    save_path=os.path.join(SWEEP_SAVE_DIR, "sweep_F3_dc_local_global.json"),
    **{k: v for k, v in SWEEP_BASE.items() if k not in ("angle_modes", "vmag_modes")})

print(f"F3 complete: {len(f3_results)} runs")


In [ ]:
# ── F4: Full DC+AC curriculum — AC activation timing (3 runs) ────────────────
# Feature: flow_loss_mode="both" (DC + AC), flow_angle_mode="both" (local + global)
# DC-local @20, DC-global @40, AC @{50, 60, 75} — tests how late AC should start.
# All: angle_mode="both", vmag=residual

pc_flow_full = PhysicsConfig(
    loss_weight_mode="adaptive", fraction_physics=0.05, max_w_phys=100.0,
    use_flow_loss=True, flow_loss_mode="both", flow_angle_mode="both",
    w_flow=1.0, w_flow_local=0.5, w_flow_global=0.5,
    w_flow_dc=0.5, w_flow_ac=0.5,
    fraction_flow=0.10, max_w_flow=100.0,
)

f4_results = run_hparam_sweep(
    networks=networks_mixed_1500_wide,
    physics_configs=[pc_flow_full],
    angle_modes=("both",), vmag_modes=("residual",),
    warmup_epochs_phys_list=(30,), warmup_epochs_ptdf_list=(20,),
    warmup_epochs_flow_list=((20, 40, 50), (20, 40, 60), (20, 40, 75)),
    tag="F4_full_curriculum",
    save_path=os.path.join(SWEEP_SAVE_DIR, "sweep_F4_full_curriculum.json"),
    **{k: v for k, v in SWEEP_BASE.items() if k not in ("angle_modes", "vmag_modes")})

print(f"F4 complete: {len(f4_results)} runs")


In [ ]:
# -- Summary: collect all sweep results ------------------------------------
import pandas as pd

all_sweep = s1_results + s2_results + s3_results + s4_results + s5_results + s6_results + s7_results
rows = []
for r in all_sweep:
    rows.append({
        "tag": r.get("tag", ""),
        "physics_label": r.get("physics_label", ""),
        "ptdf_loss_mode": r.get("ptdf_loss_mode", ""),
        "ptdf_changepoint": r.get("ptdf_changepoint"),
        "vmag_mode": r.get("vmag_mode", ""),
        "warmup_ptdf": r.get("warmup_epochs_ptdf"),
        "warmup_phys": r.get("warmup_epochs_phys"),
        "test_mse": r.get("test_metrics", {}).get("test_mse"),
        "test_physics": r.get("test_metrics", {}).get("test_physics"),
        "test_ptdf": r.get("test_metrics", {}).get("test_ptdf"),
        "final_val_loss": r.get("final_val_loss"),
    })
df_sweep = pd.DataFrame(rows).sort_values("test_mse")
sep = "=" * 70
print(f"\n{sep}")
print(f"SWEEP SUMMARY -- {len(all_sweep)} runs ranked by test_mse")
print(sep)
print(df_sweep.to_string(index=False))
total_time = sum(r.get("training_time", 0) for r in all_sweep) / 60
print(f"\nTotal run time: {total_time:.1f} min")


## Tests

In [ ]:
networks_mixed_1500_w = load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500_w.json"))

#### test phase 1


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST PHASE 1: angle_mode="both" + pipeline regression
# ═══════════════════════════════════════════════════════════════════════════════

import time as _time
_t0 = _time.perf_counter()
_errors = []

print("=" * 70)
print("PHASE 1: angle_mode='both' — Architecture + Pipeline Regression")
print("=" * 70)

_networks = networks_mixed_1500_w[:20]

# ══════════════════════════════════════════════════════════════════════════
# REGRESSION: edge_delta pipeline via sweep (end-to-end)
# ══════════════════════════════════════════════════════════════════════════
print("\n── REGRESSION: edge_delta full pipeline (sweep) ──")
_runs_reg = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(w_phys=1.0, loss_weight_mode="fixed")],
    num_epochs=5, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("edge_delta",),
    warmup_epochs_phys_list=(2,),
    verbose=False,
)
assert len(_runs_reg) == 1, f"FAIL: expected 1 run, got {len(_runs_reg)}"
_tl = _runs_reg[0]["history"]["train_total"]
assert len(_tl) == 5, f"FAIL: expected 5 epochs, got {len(_tl)}"
assert all(np.isfinite(v) for v in _tl), "FAIL: NaN/Inf in train loss"
assert _tl[-1] < _tl[0], f"FAIL: loss not decreasing ({_tl[0]:.4f} → {_tl[-1]:.4f})"
print(f"  5 epochs OK, loss: {_tl[0]:.4f} → {_tl[-1]:.4f} ✓")
del _runs_reg, _tl

# ══════════════════════════════════════════════════════════════════════════
# UNIT: angle_mode="both" architecture tests
# ══════════════════════════════════════════════════════════════════════════

# ── 1a: Forward pass produces both outputs ──
print("\n[1a] Forward pass shape check")
_m = PowerFlowGNN(node_features=7, edge_features=6, hidden_dim=32, num_layers=3, angle_mode="both").to("cpu")
_data = Data(
    x=torch.randn(14, 7),
    edge_index=torch.randint(0, 14, (2, 40)),
    edge_attr=torch.randn(40, 6),
    batch=torch.zeros(14, dtype=torch.long),
)
_node_pred, _ptdf_pred, _delta_theta_pred = _m(_data)
assert _node_pred.shape == (14, 4), f"FAIL: node_pred shape {_node_pred.shape}, expected (14, 4)"
assert _delta_theta_pred is not None, "FAIL: delta_theta_pred is None"
assert _delta_theta_pred.shape == (20,), f"FAIL: delta_theta shape {_delta_theta_pred.shape}, expected (20,)"
print(f"  node_pred={_node_pred.shape}, delta_theta_pred={_delta_theta_pred.shape} ✓")

# ── 1b: Gradients reach both heads ──
print("\n[1b] Gradient flow to both heads")
_m.zero_grad()
(_node_pred.sum() + _delta_theta_pred.sum()).backward()
assert _m.vang_pred.weight.grad is not None, "FAIL: vang_pred no grad"
assert _m.edge_angle_pred.weight.grad is not None, "FAIL: edge_angle_pred no grad"
print(f"  vang_pred grad norm: {_m.vang_pred.weight.grad.norm():.4f} ✓")
print(f"  edge_angle_pred grad norm: {_m.edge_angle_pred.weight.grad.norm():.4f} ✓")
del _m, _data, _node_pred, _ptdf_pred, _delta_theta_pred

# ── 1c: Dataset stores y_delta_theta for "both" ──
print("\n[1c] Dataset with angle_mode='both'")
_ds = PowerFlowDataset(_networks[:5], angle_mode="both", use_pnom_share=False)
_g = _ds[0]
assert hasattr(_g, 'y_delta_theta'), "FAIL: y_delta_theta missing"
assert _g.y.shape[1] == 4, f"FAIL: y shape {_g.y.shape}"
print(f"  y={_g.y.shape}, y_delta_theta={_g.y_delta_theta.shape} ✓")
del _ds, _g

# ══════════════════════════════════════════════════════════════════════════
# END-TO-END: angle_mode="both" sweep + intent validation
# ══════════════════════════════════════════════════════════════════════════
print("\n[1d] End-to-end sweep with angle_mode='both' (5 epochs)")
_runs_both = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(w_phys=1.0, loss_weight_mode="fixed")],
    num_epochs=5, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(2,),
    verbose=False,
)
assert len(_runs_both) == 1, f"FAIL: expected 1 run, got {len(_runs_both)}"
_hist = _runs_both[0]["history"]
_tl2 = _hist["train_total"]
assert len(_tl2) == 5 and all(np.isfinite(v) for v in _tl2), "FAIL: training issue"
print(f"  Train loss: {_tl2[0]:.4f} → {_tl2[-1]:.4f} ✓")

# ── 1e: Δθ MSE actually computed and non-trivial ──
print("\n[1e] Δθ MSE active in training")
_dt_mse_hist = _hist["train_delta_theta_mse"]
assert any(v > 0 for v in _dt_mse_hist), "FAIL: Δθ MSE never active"
print(f"  Δθ MSE: {_dt_mse_hist[0]:.6f} → {_dt_mse_hist[-1]:.6f} ✓")

# ── 1f: Physics loss active after warmup ──
print("\n[1f] Physics loss active after warmup")
_phys_hist = _hist["train_phys"]
assert _phys_hist[-1] > 0, f"FAIL: physics loss still zero after warmup ({_phys_hist})"
print(f"  Physics loss epoch 5: {_phys_hist[-1]:.6f} ✓")

# ── 1g: Dual-head angle consistency (Δθ ≈ θ_i − θ_j) ──
print("\n[1g] Dual-head angle consistency")
_model = _runs_both[0]["model"]
_model.eval()
_ds_check = PowerFlowDataset(_networks[:3], angle_mode="both", use_pnom_share=False)
_g_check = _ds_check[0]
with torch.no_grad():
    _np_c, _, _dt_c = _model(_g_check)
_ei_fwd = _g_check.edge_index[:, ::2]  # forward edges
_theta_diff = _np_c[_ei_fwd[0], 1] - _np_c[_ei_fwd[1], 1]  # θ_i − θ_j from node head
_corr = torch.corrcoef(torch.stack([_dt_c, _theta_diff]))[0, 1].item()
print(f"  Δθ vs (θ_i−θ_j) correlation: {_corr:.3f}", end="")
if _corr > 0.3:
    print(" ✓")
else:
    _errors.append(f"Low Δθ/(θ_i-θ_j) correlation: {_corr:.3f} (expected >0.3 after 5 epochs)")
    print(f" ⚠ (low — may improve with more epochs)")

# ── Summary ──
_elapsed = _time.perf_counter() - _t0
print("\n" + "=" * 70)
if not _errors:
    print(f"✓ PHASE 1 ALL PASSED ({_elapsed:.1f}s)")
else:
    print(f"⚠ {len(_errors)} SOFT WARNINGS:"); [print(f"  • {e}") for e in _errors]
print("=" * 70)
del _networks, _runs_both, _hist, _tl2, _dt_mse_hist, _phys_hist
del _model, _ds_check, _g_check, _np_c, _dt_c, _ei_fwd, _theta_diff, _corr
del _errors, _t0, _elapsed

#### Test phase 2

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST PHASE 2: DC/AC Per-Edge Flow Loss Functions (Task 044)
# ═══════════════════════════════════════════════════════════════════════════════

import time as _time
_t0 = _time.perf_counter()
_errors = []

print("=" * 70)
print("PHASE 2: DC/AC Per-Edge Flow Loss — Physics Correctness + Gradients")
print("=" * 70)

_networks = networks_mixed_1500_w[:10]

# ══════════════════════════════════════════════════════════════════════════
# 2a: DC formula correctness — ground-truth angles should approximate flows
# ══════════════════════════════════════════════════════════════════════════
print("\n[2a] DC flow formula: Δθ_true / x ≈ y_line_p")
_ds = PowerFlowDataset(_networks[:5], angle_mode="both", use_pnom_share=False)
_g = _ds[0]

# Extract ground-truth Δθ and y_line_p
_dt_true = _g.y_delta_theta  # ground-truth Δθ from BFS of AC solution
_y_line_p = _g.y_line_p
_edge_attr_fwd = _g.edge_attr[::2]  # forward edges

# Compute DC predicted flow from TRUE angles
_x_react = _edge_attr_fwd[:, 1]
_f_dc_pred = _dt_true / _x_react.clamp(min=1e-8)
_f_dc_pred_sup = _f_dc_pred[:len(_y_line_p)]

# DC approximation should be reasonable (MAE < 0.1 pu for typical grids)
_dc_mae = (_f_dc_pred_sup - _y_line_p).abs().mean().item()
_dc_max_err = (_f_dc_pred_sup - _y_line_p).abs().max().item()
assert _dc_mae < 0.5, f"FAIL: DC MAE with true angles = {_dc_mae:.4f} (too large)"
print(f"  DC MAE (true Δθ): {_dc_mae:.6f} pu, max err: {_dc_max_err:.6f} pu ✓")
print(f"  (non-zero error expected: DC assumes r=0, |V|=1)")

# ══════════════════════════════════════════════════════════════════════════
# 2b: AC formula correctness — should be much closer to true flows than DC
# ══════════════════════════════════════════════════════════════════════════
print("\n[2b] AC flow formula: full π-model with true V and Δθ")
_vmag_true = _g.y[:, 0]  # true Vmag [N]
_edge_index_fwd = _g.edge_index[:, ::2]
_src = _edge_index_fwd[0]
_dst = _edge_index_fwd[1]
_vi = _vmag_true[_src]
_vj = _vmag_true[_dst]
_gs = _edge_attr_fwd[:, 4]
_bs = _edge_attr_fwd[:, 5]
_cos_dt = torch.cos(_dt_true)
_sin_dt = torch.sin(_dt_true)

_f_ac_pred = _vi**2 * _gs - _vi * _vj * (_gs * _cos_dt + _bs * _sin_dt)
_f_ac_pred_sup = _f_ac_pred[:len(_y_line_p)]
_ac_mae = (_f_ac_pred_sup - _y_line_p).abs().mean().item()
_ac_max_err = (_f_ac_pred_sup - _y_line_p).abs().max().item()

# AC should be much better than DC (since it uses full model with true values)
assert _ac_mae < _dc_mae, f"FAIL: AC MAE ({_ac_mae:.6f}) >= DC MAE ({_dc_mae:.6f})"
print(f"  AC MAE (true V,Δθ): {_ac_mae:.6f} pu, max err: {_ac_max_err:.6f} pu")
print(f"  AC/DC ratio: {_ac_mae/_dc_mae:.3f} (lower = AC more accurate) ✓")

# ══════════════════════════════════════════════════════════════════════════
# 2c: Gradient routing — DC gives grad to angle only; AC Q to angle + Vmag
# ══════════════════════════════════════════════════════════════════════════
print("\n[2c] Gradient routing: DC → angle only; AC Q → angle + Vmag")

# DC local: gradient should reach delta_theta but not vmag
_dt_param = _dt_true.clone().requires_grad_(True)
_vmag_param = _vmag_true.clone().requires_grad_(True)
_loss_dc, _ = compute_flow_loss_dc_local(_dt_param, _edge_attr_fwd, _y_line_p)
_loss_dc.backward()
assert _dt_param.grad is not None and _dt_param.grad.abs().sum() > 0, "FAIL: DC no grad to Δθ"
assert _vmag_param.grad is None, "FAIL: DC should not touch Vmag"
print(f"  DC local: Δθ grad norm={_dt_param.grad.norm():.4f}, Vmag grad=None ✓")

# AC local with flow_target="q": gradient should reach BOTH angle and vmag
_dt_param2 = _dt_true.clone().requires_grad_(True)
_vmag_param2 = _vmag_true.clone().requires_grad_(True)
_y_line_q = _g.y_line_q
_loss_ac_q, _ = compute_flow_loss_ac_local(
    _dt_param2, _vmag_param2, _edge_index_fwd, _edge_attr_fwd,
    _y_line_p, _y_line_q, flow_target="pq")
_loss_ac_q.backward()
assert _dt_param2.grad is not None and _dt_param2.grad.abs().sum() > 0, "FAIL: AC no grad to Δθ"
assert _vmag_param2.grad is not None and _vmag_param2.grad.abs().sum() > 0, "FAIL: AC Q no grad to Vmag"
print(f"  AC pq: Δθ grad norm={_dt_param2.grad.norm():.4f}, Vmag grad norm={_vmag_param2.grad.norm():.4f} ✓")

del _dt_param, _vmag_param, _dt_param2, _vmag_param2, _loss_dc, _loss_ac_q

# ══════════════════════════════════════════════════════════════════════════
# 2d: Dispatcher warmup gates
# ══════════════════════════════════════════════════════════════════════════
print("\n[2d] Dispatcher warmup gate control")
_cfg_flow = PhysicsConfig(use_flow_loss=True, flow_loss_mode="both", flow_angle_mode="both")

# All active
_total_all, _diag_all = compute_flow_loss(
    _cfg_flow, _dt_true, _g.y[:, 1], _vmag_true,
    _g.edge_index, _g.edge_attr, _y_line_p, _y_line_q,
    active_local=True, active_global=True, active_ac=True)
assert _total_all.item() > 0, "FAIL: all-active flow loss is zero"
assert "flow_loss_dc_local" in _diag_all, "FAIL: dc_local missing from diag"
assert "flow_loss_ac_global" in _diag_all, "FAIL: ac_global missing from diag"
print(f"  All active: total={_total_all.item():.6f}, components={len(_diag_all)} ✓")

# Local disabled
_total_no_local, _diag_no_local = compute_flow_loss(
    _cfg_flow, _dt_true, _g.y[:, 1], _vmag_true,
    _g.edge_index, _g.edge_attr, _y_line_p, _y_line_q,
    active_local=False, active_global=True, active_ac=True)
assert "flow_loss_dc_local" not in _diag_no_local, "FAIL: dc_local should be gated"
assert "flow_loss_ac_local" not in _diag_no_local, "FAIL: ac_local should be gated"
assert "flow_loss_dc_global" in _diag_no_local, "FAIL: dc_global should still be active"
print(f"  Local gated: total={_total_no_local.item():.6f}, components={len(_diag_no_local)} ✓")

# AC disabled
_total_no_ac, _diag_no_ac = compute_flow_loss(
    _cfg_flow, _dt_true, _g.y[:, 1], _vmag_true,
    _g.edge_index, _g.edge_attr, _y_line_p, _y_line_q,
    active_local=True, active_global=True, active_ac=False)
assert "flow_loss_ac_local" not in _diag_no_ac, "FAIL: ac_local should be gated"
assert "flow_loss_dc_local" in _diag_no_ac, "FAIL: dc_local should still be active"
print(f"  AC gated: total={_total_no_ac.item():.6f}, components={len(_diag_no_ac)} ✓")

del _cfg_flow, _total_all, _diag_all, _total_no_local, _diag_no_local, _total_no_ac, _diag_no_ac

# ══════════════════════════════════════════════════════════════════════════
# 2e: DC global vs DC local consistency with true angles
# ══════════════════════════════════════════════════════════════════════════
print("\n[2e] DC global ≡ DC local when using consistent θ and Δθ")
_theta_true = _g.y[:, 1]  # node angles from ground truth
_loss_dc_l, _mae_dc_l = compute_flow_loss_dc_local(_dt_true, _edge_attr_fwd, _y_line_p)
_loss_dc_g, _mae_dc_g = compute_flow_loss_dc_global(_theta_true, _edge_index_fwd, _edge_attr_fwd, _y_line_p)

# With true angles, DC local and global should give very similar results
# (not identical because Δθ comes from BFS which may handle multi-graph differently)
_ratio = abs(_loss_dc_l.item() - _loss_dc_g.item()) / max(_loss_dc_l.item(), 1e-10)
assert _ratio < 0.1, f"FAIL: DC local/global diverge: ratio={_ratio:.4f}"
print(f"  DC local loss={_loss_dc_l.item():.6f}, DC global loss={_loss_dc_g.item():.6f}")
print(f"  Relative diff: {_ratio:.6f} ✓")

# ══════════════════════════════════════════════════════════════════════════
# 2f: End-to-end — model predictions through flow loss (backward works)
# ══════════════════════════════════════════════════════════════════════════
print("\n[2f] End-to-end: model forward → flow loss → backward")
_m = PowerFlowGNN(node_features=7, edge_features=6, hidden_dim=32, num_layers=3, angle_mode="both").to("cpu")
_node_pred, _, _dt_pred = _m(_g)
_cfg_e2e = PhysicsConfig(use_flow_loss=True, flow_loss_mode="both", flow_angle_mode="both")
_flow_total, _flow_diag = compute_flow_loss(
    _cfg_e2e, _dt_pred, _node_pred[:, 1], _node_pred[:, 0],
    _g.edge_index, _g.edge_attr, _y_line_p, _y_line_q)
_flow_total.backward()
# Check gradients reach model parameters
_has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in _m.parameters())
assert _has_grad, "FAIL: flow loss gradients don't reach model"
print(f"  Flow loss={_flow_total.item():.4f}, backward OK, model grads present ✓")

del _m, _node_pred, _dt_pred, _cfg_e2e, _flow_total, _flow_diag

# ── Summary ──
_elapsed = _time.perf_counter() - _t0
print("\n" + "=" * 70)
if not _errors:
    print(f"✓ PHASE 2 ALL PASSED ({_elapsed:.1f}s)")
else:
    print(f"✗ {len(_errors)} ISSUES:"); [print(f"  • {e}") for e in _errors]
print("=" * 70)

del _networks, _ds, _g, _dt_true, _y_line_p, _y_line_q, _edge_attr_fwd
del _edge_index_fwd, _vmag_true, _theta_true, _errors, _t0, _elapsed
del _f_dc_pred, _f_dc_pred_sup, _f_ac_pred, _f_ac_pred_sup
del _src, _dst, _vi, _vj, _gs, _bs, _cos_dt, _sin_dt
del _dc_mae, _dc_max_err, _ac_mae, _ac_max_err
del _loss_dc_l, _mae_dc_l, _loss_dc_g, _mae_dc_g, _ratio, _x_react, _has_grad

#### Test Phase 3

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST PHASE 3: Flow Loss Training Loop Integration + Curriculum (Task 045)
# ═══════════════════════════════════════════════════════════════════════════════

import time as _time
_t0 = _time.perf_counter()
_errors = []

print("=" * 70)
print("PHASE 3: Flow Loss Curriculum — Warmup, Training, Sweep Integration")
print("=" * 70)

_networks = networks_mixed_1500_w[:20]

# ══════════════════════════════════════════════════════════════════════════
# 3a: Flow loss activates ONLY after warmup epoch
# ══════════════════════════════════════════════════════════════════════════
print("\n[3a] Warmup gating: flow loss = 0 before warmup, > 0 after")
_runs_warmup = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local", w_flow=1.0,
    )],
    num_epochs=6, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(0,),
    warmup_epochs_flow_list=((3, 5, 6),),
    verbose=False,
)
_hist = _runs_warmup[0]["history"]
_fl = _hist["train_flow_loss"]

# Flow loss should be 0 for epochs 0-2, then > 0 for epochs 3-5
assert all(_fl[i] == 0.0 for i in range(3)), f"FAIL: flow loss non-zero before warmup: {_fl[:3]}"
assert all(_fl[i] > 0.0 for i in range(3, 6)), f"FAIL: flow loss zero after warmup: {_fl[3:]}"
print(f"  Epochs 0-2 (gated): {_fl[:3]}")
print(f"  Epochs 3-5 (active): [{_fl[3]:.4f}, {_fl[4]:.4f}, {_fl[5]:.4f}] ✓")
del _runs_warmup, _hist, _fl

# ══════════════════════════════════════════════════════════════════════════
# 3b: Curriculum phases — DC-local → DC-global → AC
# ══════════════════════════════════════════════════════════════════════════
print("\n[3b] Curriculum: DC-local only (global/AC gated)")
_runs_dc_only = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=0.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="both", flow_angle_mode="both",
        w_flow=1.0, w_flow_dc=0.5, w_flow_ac=0.5, w_flow_local=0.5, w_flow_global=0.5,
    )],
    num_epochs=5, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(99,),  # disable physics
    warmup_epochs_flow_list=((0, 3, 99),),
    verbose=False,
)
_hist_dc = _runs_dc_only[0]["history"]
_fl_dc = _hist_dc["train_flow_loss"]
# Flow loss should increase when global activates at epoch 3
# (more loss terms → typically higher total unless model improved)
assert all(v > 0 for v in _fl_dc), f"FAIL: flow loss has zeros: {_fl_dc}"
# Verify model still learns (total loss decreases)
_tl = _hist_dc["train_total"]
assert _tl[-1] < _tl[0], f"FAIL: total loss not decreasing with flow-only: {_tl[0]:.4f} → {_tl[-1]:.4f}"
print(f"  Flow loss (DC-local only, ep 0-2): {_fl_dc[0]:.4f}, {_fl_dc[1]:.4f}, {_fl_dc[2]:.4f}")
print(f"  Flow loss (DC-local+global, ep 3-4): {_fl_dc[3]:.4f}, {_fl_dc[4]:.4f}")
print(f"  Total loss: {_tl[0]:.4f} → {_tl[-1]:.4f} ✓")
del _runs_dc_only, _hist_dc, _fl_dc, _tl

# ══════════════════════════════════════════════════════════════════════════
# 3c: Flow loss improves angle predictions (intent: better Δθ/x ≈ flow)
# ══════════════════════════════════════════════════════════════════════════
print("\n[3c] Flow loss teaches angle-flow physics (compare with/without)")

# Run WITHOUT flow loss
_runs_no_flow = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(w_phys=1.0, loss_weight_mode="fixed", use_flow_loss=False)],
    num_epochs=10, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(2,),
    verbose=False,
)

# Run WITH flow loss (DC local)
_runs_with_flow = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local", w_flow=1.0,
    )],
    num_epochs=10, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(2,),
    warmup_epochs_flow=2,
    verbose=False,
)

# Compare: model with flow loss should have better flow predictions
# Use the flow loss function itself as the metric (lower = better angle→flow mapping)
_model_no = _runs_no_flow[0]["model"]
_model_yes = _runs_with_flow[0]["model"]
_model_no.eval(); _model_yes.eval()

_ds_eval = PowerFlowDataset(_networks[14:20], angle_mode="both", use_pnom_share=False)
_g_eval = _ds_eval[0]

with torch.no_grad():
    _, _, _dt_no = _model_no(_g_eval)
    _, _, _dt_yes = _model_yes(_g_eval)

_edge_attr_fwd = _g_eval.edge_attr[::2]
_y_lp = _g_eval.y_line_p
_x_react = _edge_attr_fwd[:, 1].clamp(min=1e-8)

_flow_err_no = ((_dt_no / _x_react)[:len(_y_lp)] - _y_lp).abs().mean().item()
_flow_err_yes = ((_dt_yes / _x_react)[:len(_y_lp)] - _y_lp).abs().mean().item()

print(f"  Without flow loss → DC flow MAE: {_flow_err_no:.6f}")
print(f"  With flow loss    → DC flow MAE: {_flow_err_yes:.6f}")
if _flow_err_yes < _flow_err_no:
    print(f"  Improvement: {(_flow_err_no - _flow_err_yes)/_flow_err_no*100:.1f}% ✓")
else:
    _errors.append(f"Flow loss did not improve DC flow MAE ({_flow_err_yes:.4f} >= {_flow_err_no:.4f}) — may need more epochs")
    print(f"  ⚠ No improvement (may need more epochs)")

del _runs_no_flow, _runs_with_flow, _model_no, _model_yes
del _ds_eval, _g_eval, _dt_no, _dt_yes, _edge_attr_fwd, _y_lp, _x_react

# ══════════════════════════════════════════════════════════════════════════
# 3d: Adaptive mode — flow weight scales with MSE
# ══════════════════════════════════════════════════════════════════════════
print("\n[3d] Adaptive mode: flow weight tracks MSE ratio")
_runs_adapt = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="adaptive",
        fraction_physics=0.1, fraction_flow=0.1,
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local",
    )],
    num_epochs=5, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(0,),
    warmup_epochs_flow=0,
    verbose=False,
)
_hist_ad = _runs_adapt[0]["history"]
_w_flow_eff = _hist_ad["train_eff_w_flow"]
# Adaptive weight should be non-zero and finite
assert all(np.isfinite(w) for w in _w_flow_eff), f"FAIL: NaN/Inf in eff_w_flow"
assert any(w > 0 for w in _w_flow_eff), "FAIL: eff_w_flow always zero"
# Should not hit the cap (100) on this tiny dataset
assert all(w <= 100.0 for w in _w_flow_eff), f"FAIL: eff_w_flow exceeded cap"
print(f"  eff_w_flow: {[f'{w:.3f}' for w in _w_flow_eff]}")
print(f"  Range: [{min(_w_flow_eff):.4f}, {max(_w_flow_eff):.4f}] ✓")
del _runs_adapt, _hist_ad, _w_flow_eff

# ══════════════════════════════════════════════════════════════════════════
# 3e: Validation flow loss tracked (not just training)
# ══════════════════════════════════════════════════════════════════════════
print("\n[3e] Validation flow loss in history")
_runs_val = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local", w_flow=1.0,
    )],
    num_epochs=5, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both",),
    warmup_epochs_phys_list=(0,),
    warmup_epochs_flow=0,
    verbose=False,
)
_hist_v = _runs_val[0]["history"]
assert "val_flow_loss" in _hist_v, "FAIL: val_flow_loss not in history"
_vfl = _hist_v["val_flow_loss"]
assert len(_vfl) == 5, f"FAIL: val_flow_loss has {len(_vfl)} entries"
assert all(np.isfinite(v) for v in _vfl), "FAIL: NaN/Inf in val_flow_loss"
assert any(v > 0 for v in _vfl), "FAIL: val_flow_loss all zeros"
print(f"  val_flow_loss: {_vfl[0]:.4f} → {_vfl[-1]:.4f} ✓")

# Also check run_info stores flow config
_ri = _runs_val[0]
assert _ri.get("use_flow_loss") == True, "FAIL: use_flow_loss not in run_info"
assert _ri.get("warmup_epochs_flow") == 0, "FAIL: warmup_epochs_flow not stored"
print(f"  run_info stores flow config ✓")
del _runs_val, _hist_v, _vfl, _ri

# ── Summary ──
_elapsed = _time.perf_counter() - _t0
print("\n" + "=" * 70)
if not _errors:
    print(f"✓ PHASE 3 ALL PASSED ({_elapsed:.1f}s)")
else:
    print(f"⚠ {len(_errors)} SOFT WARNINGS:"); [print(f"  • {e}") for e in _errors]
print("=" * 70)
del _networks, _errors, _t0, _elapsed, _flow_err_no, _flow_err_yes

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST: Product-loop threading — angle_modes, vmag_modes, warmup_epochs_flow_list
# Verifies all tuple-list params produce correct products, distinct keys, and
# reach training correctly.
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("TEST: Product-loop parameter threading")
print("=" * 70)

_networks = networks_mixed_1500_w[:10]
_passed = 0
_failed = 0

def _check(name, cond):
    global _passed, _failed
    if cond:
        _passed += 1
        print(f"  ✓ {name}")
    else:
        _failed += 1
        print(f"  ✗ FAIL: {name}")

# ── 1. angle_modes expansion: 2 modes → 2 runs ──────────────────────────────
print("\n─── 1. angle_modes: 2 values → 2 runs ───")
_runs_am = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(w_phys=1.0, loss_weight_mode="fixed")],
    num_epochs=2, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("edge_delta", "both"),
    vmag_modes=("residual",),
    warmup_epochs_phys_list=(0,),
    verbose=False,
)
_check(f"Got 2 runs (actual: {len(_runs_am)})", len(_runs_am) == 2)
_check("run[0] angle_mode=edge_delta", _runs_am[0]["angle_mode"] == "edge_delta")
_check("run[1] angle_mode=both", _runs_am[1]["angle_mode"] == "both")
_keys_am = [r["run_key"] for r in _runs_am]
_check("Distinct run_keys", len(set(_keys_am)) == 2)
_check("Key[0] has _amedge_delta", "_amedge_delta" in _keys_am[0])
_check("Key[1] has _amboth", "_amboth" in _keys_am[1])
del _runs_am, _keys_am

# ── 2. vmag_modes expansion: 2 modes → 2 runs ───────────────────────────────
print("\n─── 2. vmag_modes: 2 values → 2 runs ───")
_runs_vm = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(w_phys=1.0, loss_weight_mode="fixed")],
    num_epochs=2, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("edge_delta",),
    vmag_modes=("absolute", "residual"),
    warmup_epochs_phys_list=(0,),
    verbose=False,
)
_check(f"Got 2 runs (actual: {len(_runs_vm)})", len(_runs_vm) == 2)
_check("run[0] vmag_mode=absolute", _runs_vm[0]["vmag_mode"] == "absolute")
_check("run[1] vmag_mode=residual", _runs_vm[1]["vmag_mode"] == "residual")
_keys_vm = [r["run_key"] for r in _runs_vm]
_check("Distinct run_keys", len(set(_keys_vm)) == 2)
_check("Key[1] has _vmresidual", "_vmresidual" in _keys_vm[1])
del _runs_vm, _keys_vm

# ── 3. Full cross-product: 2 angle × 2 vmag × 2 flow_triplets = 8 runs ──────
print("\n─── 3. Full product: 2 angle × 2 vmag × 2 triplets = 8 runs ───")
_runs_full = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local", w_flow=1.0,
    )],
    num_epochs=2, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("edge_delta", "both"),
    vmag_modes=("absolute", "residual"),
    warmup_epochs_phys_list=(0,),
    warmup_epochs_flow_list=((0, 0, 0), (1, 0, 0)),
    verbose=False,
)
_check(f"Got 8 runs (actual: {len(_runs_full)})", len(_runs_full) == 8)
_keys_full = [r["run_key"] for r in _runs_full]
_check("All 8 run_keys unique", len(set(_keys_full)) == 8)

# Verify specific combos exist
_combos_found = [(r["angle_mode"], r["vmag_mode"], r["warmup_epochs_flow"]) for r in _runs_full]
_check("(edge_delta, absolute, 0) in results", ("edge_delta", "absolute", 0) in _combos_found)
_check("(both, residual, 1) in results", ("both", "residual", 1) in _combos_found)
_check("(edge_delta, residual, 0) in results", ("edge_delta", "residual", 0) in _combos_found)
_check("(both, absolute, 1) in results", ("both", "absolute", 1) in _combos_found)
del _runs_full, _keys_full, _combos_found

# ── 4. Warmup gating still works (regression) ───────────────────────────────
print("\n─── 4. Warmup gating (regression check) ───")
_runs_gate = run_hparam_sweep(
    _networks,
    physics_configs=[PhysicsConfig(
        w_phys=1.0, loss_weight_mode="fixed",
        use_flow_loss=True, flow_loss_mode="dc", flow_angle_mode="local", w_flow=1.0,
    )],
    num_epochs=3, hidden_dims=(32,), num_layers_list=(3,),
    conv_types=("gatv2",), seeds=(42,),
    angle_modes=("both","edge_delta"), 
    vmag_modes=("residual","absolute"),
    warmup_epochs_phys_list=(0,),
    warmup_epochs_flow_list=((0, 0, 0), (2, 0, 0)),
    verbose=False,
)
_fl_immediate = _runs_gate[0]["history"]["train_flow_loss"]
_fl_delayed = _runs_gate[1]["history"]["train_flow_loss"]
_check("(0,0,0): flow active at epoch 0", _fl_immediate[0] > 0)
_check("(2,0,0): flow=0 at epoch 0", _fl_delayed[0] == 0.0)
_check("(2,0,0): flow=0 at epoch 1", _fl_delayed[1] == 0.0)
_check("(2,0,0): flow>0 at epoch 2", _fl_delayed[2] > 0.0)
del _runs_gate, _fl_immediate, _fl_delayed

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"RESULT: {_passed} passed, {_failed} failed")
if _failed == 0:
    print("All product-loop threading tests PASSED ✓")
else:
    print("SOME TESTS FAILED — check output above")
print("=" * 70)

del _networks, _passed, _failed, _check


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST: use_global_pool — functional + end-to-end sweep
# ═══════════════════════════════════════════════════════════════════════════════

import time

print("=" * 70)
print("TEST SUITE: use_global_pool feature")
print("=" * 70)

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        passed += 1
        print(f"  ✓ {name}")
    else:
        failed += 1
        print(f"  ✗ FAIL: {name}")

# ── 1. Functional Tests ─────────────────────────────────────────────────────

print("\n─── 1. Instantiation ───")
model_gp = PowerFlowGNN(use_global_pool=True, angle_mode="edge_delta")
model_no_gp = PowerFlowGNN(use_global_pool=False, angle_mode="edge_delta")
check("Instantiates with use_global_pool=True", model_gp is not None)
check("Instantiates with use_global_pool=False", model_no_gp is not None)

print("\n─── 2. Model repr (global_proj presence) ───")
check("global_proj IN model when pool=True", "global_proj" in str(model_gp))
check("global_proj NOT in model when pool=False", "global_proj" not in str(model_no_gp))

print("\n─── 3. Parameter count difference ───")
n_params_gp = sum(p.numel() for p in model_gp.parameters())
n_params_no = sum(p.numel() for p in model_no_gp.parameters())
extra_params = n_params_gp - n_params_no
expected_extra = 2 * 64 * 64 + 64  # Linear(128→64) = 128*64 weights + 64 bias
check(f"Extra params = {extra_params} (expected {expected_extra})", extra_params == expected_extra)

print("\n─── 4. Forward pass — single graph (no batch attr) ───")
dummy = Data(
    x=torch.randn(9, 7),
    edge_index=torch.randint(0, 9, (2, 14)),
    edge_attr=torch.randn(14, 4),
)
with torch.no_grad():
    out_gp = model_gp(dummy)
    out_no = model_no_gp(dummy)
check(f"pool=True  shape: {out_gp[0].shape}", out_gp[0].shape == torch.Size([9, 3]))
check(f"pool=False shape: {out_no[0].shape}", out_no[0].shape == torch.Size([9, 3]))
check("Shapes identical", out_gp[0].shape == out_no[0].shape)

print("\n─── 5. Forward pass — batched (PyG Batch) ───")
d1 = Data(x=torch.randn(9, 7), edge_index=torch.randint(0, 9, (2, 12)), edge_attr=torch.randn(12, 4))
d2 = Data(x=torch.randn(5, 7), edge_index=torch.randint(0, 5, (2, 8)), edge_attr=torch.randn(8, 4))
batch = Batch.from_data_list([d1, d2])
with torch.no_grad():
    out_batch = model_gp(batch)
check(f"Batched output shape: {out_batch[0].shape} (expect [14,3])", out_batch[0].shape == torch.Size([14, 3]))

print("\n─── 6. Compatibility: angle_mode='node' ───")
m_node = PowerFlowGNN(use_global_pool=True, angle_mode="node")
with torch.no_grad():
    out_node = m_node(dummy)
check(f"angle_mode=node output: {out_node[0].shape} (expect [9,4])", out_node[0].shape == torch.Size([9, 4]))

print("\n─── 7. Compatibility: vmag_mode='residual' ───")
m_res = PowerFlowGNN(use_global_pool=True, angle_mode="edge_delta", vmag_mode="residual")
with torch.no_grad():
    out_res = m_res(dummy)
check(f"vmag_mode=residual output: {out_res[0].shape}", out_res[0].shape == torch.Size([9, 3]))
# Residual mode should have Vmag near 1.0 at init
vmag_mean = out_res[0][:, 0].mean().item()
check(f"Initial Vmag ≈ 1.0 (got {vmag_mean:.4f})", abs(vmag_mean - 1.0) < 0.1)

print("\n─── 8. Compatibility: head_mode='with_encoder' ───")
m_enc = PowerFlowGNN(use_global_pool=True, angle_mode="edge_delta", head_mode="with_encoder")
dummy_enc = Data(
    x=torch.randn(9, 7),
    edge_index=torch.randint(0, 9, (2, 14)),
    edge_attr=torch.randn(14, 4),
    pq_mask=torch.tensor([0,0,0,1,1,1,1,1,1], dtype=torch.float),
    pv_mask=torch.tensor([0,1,1,0,0,0,0,0,0], dtype=torch.float),
    slack_mask=torch.tensor([1,0,0,0,0,0,0,0,0], dtype=torch.float),
)
with torch.no_grad():
    out_enc = m_enc(dummy_enc)
check(f"with_encoder output: {out_enc[0].shape}", out_enc[0].shape == torch.Size([9, 3]))

# ── 2. End-to-End Test with run_hparam_sweep ────────────────────────────────

print("\n─── 9. End-to-end: run_hparam_sweep with use_global_pool ───")
# Use a tiny subset for speed (3 networks, 3 epochs)
test_networks = networks_mixed_1500_wide[:10]

t0 = time.time()
results_gp_test = run_hparam_sweep(
    networks=test_networks,
    physics_configs=[PhysicsConfig()],  # no physics, fastest
    num_epochs=3,
    batch_sizes=(32,),
    learning_rates=(1e-3,),
    conv_types=("gatv2","transformer"),
    use_global_pool=True,
    angle_mode="edge_delta",
    vmag_mode="absolute",
    tag="test_global_pool",
    verbose=False,
)
dt_gp = time.time() - t0
print(f"  Sweep completed in {dt_gp:.1f}s")

check("run_hparam_sweep returned results", len(results_gp_test) > 0)
r = results_gp_test[0]
check("run_info has 'use_global_pool' key", "use_global_pool" in r)
check("run_info['use_global_pool'] is True", r.get("use_global_pool") is True)
check("run_key contains '_gp'", "_gp" in r.get("run_key", ""))
check("test_metrics present", "test_metrics" in r and r["test_metrics"] is not None)

# Verify model in results actually has global_proj
model_from_sweep = r.get("model")
if model_from_sweep is not None:
    check("Sweep model has global_proj", hasattr(model_from_sweep, "global_proj"))
    check("Sweep model.use_global_pool=True", model_from_sweep.use_global_pool is True)
else:
    check("Sweep returned model", False)

# Compare with pool=False run
t0 = time.time()
results_no_gp_test = run_hparam_sweep(
    networks=test_networks,
    physics_configs=[PhysicsConfig()],
    num_epochs=3,
    batch_sizes=(32,),
    learning_rates=(1e-3,),
    conv_types=("gatv2","transformer"),
    use_global_pool=False,
    angle_mode="edge_delta",
    vmag_mode="absolute",
    tag="test_no_global_pool",
    verbose=False,
)
dt_no = time.time() - t0
r2 = results_no_gp_test[0]
check("No-pool run_key differs from pool run_key", r["run_key"] != r2["run_key"])
check("No-pool model lacks global_proj", not hasattr(r2["model"], "global_proj"))

print(f"\n{'=' * 70}")
print(f"RESULTS: {passed} passed, {failed} failed")
print(f"{'=' * 70}")
assert failed == 0, f"{failed} test(s) failed!"